# import data

In [1]:
import pandas as pd
# Set option to display all columns
pd.set_option('display.max_columns', None)


In [2]:
import duckdb
from pathlib import Path

con = duckdb.connect()

# Low-memory settings
con.execute("PRAGMA threads=1;")
con.execute("PRAGMA preserve_insertion_order=false;")
con.execute("PRAGMA enable_object_cache=false;")
con.execute("PRAGMA memory_limit='2GB';")           # try 1GB if still unstable
con.execute("PRAGMA temp_directory='data/tmp_duckdb';")

# 2) Build paths robustly from the notebook folder
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BASE = ROOT / "data" / "by_server"

# IMPORTANT: your files are hive-partitioned like:
all_backends = (BASE / "*" / "*.parquet").as_posix()

con.execute(f"""
CREATE OR REPLACE VIEW all_backends AS
SELECT * FROM read_parquet('{all_backends}', hive_partitioning=true, union_by_name=true);
""")

# A unified "all_rows" view
con.execute("""
CREATE OR REPLACE VIEW all_rows AS
SELECT * FROM all_backends
""")

print(con.execute("SHOW TABLES").fetchall())


[('all_backends',), ('all_rows',)]


In [3]:
con.execute(f"""
CREATE OR REPLACE VIEW server_thin AS
SELECT
  CAST(record_id AS VARCHAR)           AS record_id,
  CAST(server_name AS VARCHAR)         AS server_name,
  CAST(backend AS VARCHAR)             AS backend,

  CAST(doi AS VARCHAR)                 AS doi,
  CAST(doi_url AS VARCHAR)             AS doi_url,
  CAST(landing_page_url AS VARCHAR)    AS landing_page_url,
  CAST(type_backend_raw AS VARCHAR)    AS type_backend_raw,
  CAST(subtype_backend_raw AS VARCHAR)    AS subtype_backend_raw,
 	
  CAST(title AS VARCHAR) AS title,
  -- CAST(abstract_text AS VARCHAR)      AS abstract_text,
  CAST(authors_flat AS VARCHAR)      AS authors_flat,
  CAST(institutions_flat AS VARCHAR)      AS institutions_flat,
  CAST(countries_flat AS VARCHAR)      AS countries_flat,
  
  -- Dates (helpful for temporal patterns)
  -- CAST(publication_year AS VARCHAR)    AS publication_year,
  -- CAST(date_created AS VARCHAR)        AS date_created,
  -- CAST(date_posted AS VARCHAR)         AS date_posted,
  -- CAST(date_deposited AS VARCHAR)      AS date_deposited,
  -- CAST(date_published AS VARCHAR)      AS date_published,
  -- CAST(date_published_online AS VARCHAR)      AS date_published_online,
  -- CAST(date_issued AS VARCHAR)         AS date_issued,
  -- CAST(date_indexed AS VARCHAR)        AS date_indexed,
  -- CAST(date_updated AS VARCHAR)        AS date_updated,
  -- CAST(date_registered AS VARCHAR)     AS date_registered,

  -- Relationships (keep these for true version links)
  CAST(relations_json AS VARCHAR)       AS relations_json,
  CAST(version_label AS VARCHAR)       AS version_label,
  CAST(is_version_of AS VARCHAR)       AS is_version_of,      -- keep as text; we’ll interpret later
  CAST(is_preprint_of AS VARCHAR)      AS is_preprint_of,
  CAST(has_preprint AS VARCHAR)      AS has_preprint,
  CAST(has_review AS VARCHAR)      AS has_review,
  CAST(has_published_version AS VARCHAR)      AS has_published_version,
  CAST(published_version_ids_json AS VARCHAR) AS published_version_ids_json,
  CAST(version_of_ids_json AS VARCHAR) AS version_of_ids_json,
  CAST(update_to_json AS VARCHAR)      AS update_to_json,
  CAST(raw_relationships_json AS VARCHAR)       AS raw_relationships_json,
FROM all_backends
""")

con.execute("SELECT COUNT(*) AS n FROM server_thin").df()


,n
0,9797749


In [4]:
data = con.execute("SELECT * FROM server_thin").df()
# data.drop_duplicates(subset=['record_id'], keep='first', inplace=False)

data = data.drop_duplicates()
data

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json
0,crossref::10.21467/preprints.48,AIJR Preprints,crossref,10.21467/preprints.48,https://doi.org/10.21467/preprints.48,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,"Bird’s Eye View on the Diagnosis, Treatment, &...","Panchalingala, Sai Bhargavi",None,None,None,None,,,,,false,None,None,None,None
1,crossref::10.21467/preprints.43,AIJR Preprints,crossref,10.21467/preprints.43,https://doi.org/10.21467/preprints.43,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Doxycycline and Minocycline Drugs as a Treatme...,"Mostafa, Mohamed",None,None,None,None,,,,,false,None,None,None,None
2,crossref::10.21467/preprints.39,AIJR Preprints,crossref,10.21467/preprints.39,https://doi.org/10.21467/preprints.39,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,A Genetic Perspective of 2019-nCoV in Relation...,"Dasgupta, Rimjhim",None,None,None,None,,,,,false,None,None,None,None
3,crossref::10.21467/preprints.38,AIJR Preprints,crossref,10.21467/preprints.38,https://doi.org/10.21467/preprints.38,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Marine Algae as a Natural Source for Antiviral...,"Musale, Amar S; G., Raja Krishna Kumar; Sapre,...",None,None,None,None,,,,,false,None,None,None,None
4,crossref::10.21467/preprints.36,AIJR Preprints,crossref,10.21467/preprints.36,https://doi.org/10.21467/preprints.36,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Possible Prevention of COVID 19 by Using Linol...,"Subhash, Venkata; G, Raja Krishna Kumar; Sapre...",None,None,None,None,,,,,false,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9797744,openalex::W999325625,viXra,openalex,None,None,https://vixra.org/pdf/1409.0090v1.pdf,preprint,None,Three Objections to Modern Physics,Lubomir Vlcek,None,None,None,None,None,None,None,None,None,None,None,None,None
9797745,openalex::W999460032,viXra,openalex,None,None,https://vixra.org/abs/1112.0094,preprint,None,Particle Mass Ratios,DT Froedge,None,None,None,None,None,None,None,None,None,None,None,None,None
9797746,openalex::W99967155,viXra,openalex,None,None,https://vixra.org/pdf/1406.0019v1.pdf,preprint,None,Quantum FFF Theory Proposals for Some Unsolved...,Leo Vuyk,None,None,None,None,None,None,None,None,None,None,None,None,None
9797747,openalex::W999790414,viXra,openalex,None,None,https://vixra.org/pdf/1306.0105v3.pdf,preprint,None,Investigation of the Formalism of Particle Dyn...,Chi-Yi Chen,None,None,None,None,None,None,None,None,None,None,None,None,None


# clean data

## keep only records WITHOUT a DOI for specific servers

In [5]:
# List of servers where you want to keep only records WITHOUT a DOI
servers_no_doi = [
    "IACR Cryptology ePrint Archive",
    "Munich Personal RePEc Archive",
    "Organic Eprints",
    "PhilSci-Archive",
    "Digital Access to Scholarship at Harvard (DASH) (Harvard University)",
    "DSpace@MIT",
    "E-LIS Repository",
    "EconStor Preprints",
    "National Bureau of Economic Research",
    "viXra",
    "CogPrints"
]

# Rows from those servers that have no DOI
mask_no_doi = data['server_name'].isin(servers_no_doi) & data['doi'].isna()

# Rows from all other servers (kept as-is, regardless of DOI)
mask_other = ~data['server_name'].isin(servers_no_doi)

df_filtered = data[mask_no_doi | mask_other]

In [6]:
# df_filtered

In [7]:
# Quick sanity check
print("Original shape:", data.shape)
print("Filtered shape:", df_filtered.shape)

# Confirm none of the targeted servers have DOIs remaining
check = df_filtered[df_filtered['server_name'].isin(servers_no_doi)]['doi'].isna().all()
print("All targeted server rows have no DOI:", check)

Original shape: (7984655, 23)
Filtered shape: (7971020, 23)
All targeted server rows have no DOI: True


In [8]:
df_filtered['server_name'].value_counts()

server_name
arXiv                                  2920797
SSRN                                   1259539
HAL                                    1056424
Research Square                         450818
RePEc: Research Papers in Economics     389398
                                        ...   
MNI Open Research                           20
EmeRI                                        8
Prepublicaciones OpenCiencia                 8
NewAddictionsX                               7
Therapoid                                    7
Name: count, Length: 109, dtype: int64

In [9]:
data['server_name'].value_counts()

server_name
arXiv                                  2920797
SSRN                                   1259539
HAL                                    1056424
Research Square                         450818
RePEc: Research Papers in Economics     389398
                                        ...   
MNI Open Research                           20
EmeRI                                        8
Prepublicaciones OpenCiencia                 8
NewAddictionsX                               7
Therapoid                                    7
Name: count, Length: 109, dtype: int64

In [10]:
data[data['server_name']=='IACR Cryptology ePrint Archive']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json
1900241,openalex::W2494078997,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2015/1092.pdf,article,None,Post-quantum key exchange: a new hope,Erdem Alkım; Léo Ducas; Thomas Pöppelmann; Pet...,Ege University; Centrum Wiskunde & Informatica...,TR; NL; DE,None,None,None,None,None,None,None,None,None,None,None
1900242,openalex::W2531800036,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2003/183.pdf,article,None,Certificate-Based Encryption and the Certifica...,Craig Gentry,None,None,None,None,None,None,None,None,None,None,None,None,None
1900243,openalex::W1692463454,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2010/305.pdf,article,None,On the impossibility of cryptography alone for...,Marten van Dijk; Ari Juels,None,None,None,None,None,None,None,None,None,None,None,None,None
1900244,openalex::W2151237818,IACR Cryptology ePrint Archive,openalex,10.4230/dagsemproc.08491.4,https://doi.org/10.4230/dagsemproc.08491.4,https://eprint.iacr.org/2008/481.pdf,article,None,Public-Key Cryptosystems from the Worst-Case S...,Chris Peikert,Massachusetts Institute of Technology,US,None,None,None,None,None,None,None,None,None,None,None
1900245,openalex::W2909978789,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2009/608.pdf,article,None,Non-Malleable Codes.,Stefan Dziembowski; Krzysztof Pietrzak; Daniel...,University of Warsaw; New York University,PL; US,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1912140,openalex::W71556202,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2005/055.pdf,preprint,None,Untraceability of Two Group Signature Schemes.,Zhengjun Cao,None,None,None,None,None,None,None,None,None,None,None,None,None
1912141,openalex::W72512128,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2004/269.pdf,preprint,None,Cryptanalysis of Threshold-Multisignature Sche...,Lifeng Guo,Chinese Academy of Sciences,CN,None,None,None,None,None,None,None,None,None,None,None
1912142,openalex::W750410284,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2009/269.pdf,preprint,None,Side-channel attacks based on linear approxima...,Thomas Roche; Cédric Tavernier,None,None,None,None,None,None,None,None,None,None,None,None,None
1912143,openalex::W81317141,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2005/445.pdf,preprint,None,An Anonymous Authentication Scheme for Trusted...,He Ge,University of North Texas,US,None,None,None,None,None,None,None,None,None,None,None


In [11]:
data[data['server_name']=='IACR Cryptology ePrint Archive']['doi'].value_counts()

doi
10.4230/dagsemproc.08491.4     1
10.13140/rg.2.2.20839.85920    1
10.7916/d8mk6mr2               1
10.13140/rg.2.1.2367.9767      1
10.5281/zenodo.3610301         1
                              ..
10.48550/arxiv.2111.02700      1
10.60882/cispa.24612741        1
10.60882/cispa.24613899        1
10.48550/arxiv.2106.13339      1
10.1111/his.14060              1
Name: count, Length: 208, dtype: int64

In [12]:
df_filtered[df_filtered['server_name']=='IACR Cryptology ePrint Archive']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json
1900241,openalex::W2494078997,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2015/1092.pdf,article,None,Post-quantum key exchange: a new hope,Erdem Alkım; Léo Ducas; Thomas Pöppelmann; Pet...,Ege University; Centrum Wiskunde & Informatica...,TR; NL; DE,None,None,None,None,None,None,None,None,None,None,None
1900242,openalex::W2531800036,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2003/183.pdf,article,None,Certificate-Based Encryption and the Certifica...,Craig Gentry,None,None,None,None,None,None,None,None,None,None,None,None,None
1900243,openalex::W1692463454,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2010/305.pdf,article,None,On the impossibility of cryptography alone for...,Marten van Dijk; Ari Juels,None,None,None,None,None,None,None,None,None,None,None,None,None
1900245,openalex::W2909978789,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2009/608.pdf,article,None,Non-Malleable Codes.,Stefan Dziembowski; Krzysztof Pietrzak; Daniel...,University of Warsaw; New York University,PL; US,None,None,None,None,None,None,None,None,None,None,None
1900246,openalex::W2116428199,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2014/1004.pdf,article,None,CONIKS: bringing key transparency to end users,Marcela S. Melara; Aaron Blankstein; Joseph Bo...,Princeton University; Stanford University,US,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1912140,openalex::W71556202,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2005/055.pdf,preprint,None,Untraceability of Two Group Signature Schemes.,Zhengjun Cao,None,None,None,None,None,None,None,None,None,None,None,None,None
1912141,openalex::W72512128,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2004/269.pdf,preprint,None,Cryptanalysis of Threshold-Multisignature Sche...,Lifeng Guo,Chinese Academy of Sciences,CN,None,None,None,None,None,None,None,None,None,None,None
1912142,openalex::W750410284,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2009/269.pdf,preprint,None,Side-channel attacks based on linear approxima...,Thomas Roche; Cédric Tavernier,None,None,None,None,None,None,None,None,None,None,None,None,None
1912143,openalex::W81317141,IACR Cryptology ePrint Archive,openalex,None,None,https://eprint.iacr.org/2005/445.pdf,preprint,None,An Anonymous Authentication Scheme for Trusted...,He Ge,University of North Texas,US,None,None,None,None,None,None,None,None,None,None,None


In [13]:
df_filtered[df_filtered['server_name']=='IACR Cryptology ePrint Archive']['doi'].value_counts()

Series([], Name: count, dtype: int64)

## manage type

### ScienceOpen Preprints

In [14]:
# Mask for ScienceOpen Preprints rows with subtype 'other'
mask_scienceopen_remove = ~(
    (data['server_name'] == 'ScienceOpen Preprints') & 
    (data['subtype_backend_raw'] == 'other')
)

df_filtered = df_filtered[mask_scienceopen_remove]

/tmp/ipykernel_46769/744606357.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_filtered = df_filtered[mask_scienceopen_remove]


In [15]:
check = df_filtered[
    (df_filtered['server_name'] == 'ScienceOpen Preprints') & 
    (df_filtered['subtype_backend_raw'] == 'other')
]
print("Remaining ScienceOpen 'other' rows:", len(check))  # Should be 0

Remaining ScienceOpen 'other' rows: 0


### OSF

In [16]:
# Mapping of URL pattern -> new server_name
osf_url_to_server = {
    'psyarxiv.com':       'PsyArXiv',
    'eartharxiv.org':     'EarthArXiv',
    'thesiscommons.org':  'Thesis Commons',
    'marxiv.org':         'MarXiv',
    'engrxiv.org':        'engrXiv',
    'arabixiv.org':       'Arabixiv',
    'mindrxiv.org':       'MindRxiv',
    'agrixiv.org':        'AgriRxiv',
    'paleorxiv.org':      'PaleorXiv',
    'ecsarxiv.org':       'ECSarXiv',

    'osf.io/preprints/inarxiv':       'INA-Rxiv',
    'osf.io/preprints/socarxiv':       'SocArXiv',
    'osf.io/preprints/psyarxiv':       'PsyArXiv',
    'osf.io/preprints/sportrxiv':       'SportRxiv',
    'osf.io/preprints/lawarxiv':       'Law Archive',
    'osf.io/preprints/paleorxiv':       'PaleorXiv',
    'osf.io/preprints/lissa':       'LIS Scholarship Archive',

    'osf.io/preprints/bitss':       'MetaArXiv',
    'osf.io/preprints/eartharxiv':       'EarthArXiv',
    'osf.io/preprints/nutrixiv':       'NutriXiv',
    
}

# Only apply to rows currently assigned to Open Science Framework
osf_mask = df_filtered['server_name'] == 'Open Science Framework'

for url_pattern, new_name in osf_url_to_server.items():
    pattern_mask = osf_mask & df_filtered['landing_page_url'].str.contains(url_pattern, na=False)
    df_filtered.loc[pattern_mask, 'server_name'] = new_name

In [17]:
# See the distribution of OSF-related servers after renaming
osf_related = ['Open Science Framework'] + list(osf_url_to_server.values())

print(df_filtered[df_filtered['server_name'].isin(osf_related)]['server_name'].value_counts())

server_name
Open Science Framework     112580
PsyArXiv                    58219
SocArXiv                    23944
INA-Rxiv                    19579
EarthArXiv                   6803
engrXiv                      5066
Thesis Commons               4175
Law Archive                  2106
SportRxiv                     898
MetaArXiv                     894
AgriRxiv                      858
MarXiv                        666
Arabixiv                      584
LIS Scholarship Archive       465
MindRxiv                      381
ECSarXiv                      326
PaleorXiv                     322
NutriXiv                      105
Name: count, dtype: int64


In [18]:
# See the distribution of OSF-related servers after renaming
osf_related = ['Open Science Framework'] + list(osf_url_to_server.values())

print(data[data['server_name'].isin(osf_related)]['server_name'].value_counts())

server_name
Open Science Framework     119481
PsyArXiv                    56866
SocArXiv                    21541
INA-Rxiv                    17837
EarthArXiv                   6537
engrXiv                      4929
Thesis Commons               3959
Law Archive                  1808
MetaArXiv                     880
SportRxiv                     878
AgriRxiv                      818
MarXiv                        508
Arabixiv                      502
LIS Scholarship Archive       397
MindRxiv                      335
ECSarXiv                      314
PaleorXiv                     287
NutriXiv                       94
Name: count, dtype: int64


# match data

In [19]:
import pandas as pd

# ================================
# 1) Load saved artifacts
# ================================
records_hierarchy_df = pd.read_pickle("outputs_new/records_hierarchy_df.pkl")
date_first_seen_df   = pd.read_pickle("outputs_new/date_first_seen.pkl")

In [20]:
# ------------------------------------------------------------------
# Replace server_name in date_first_seen_df
# using values from records_hierarchy_df
# matched on record_id
# ------------------------------------------------------------------
# Create mapping dictionary
server_mapping = (
    records_hierarchy_df
    .set_index("record_id")["server_name"]
)
# Replace/update server_name
date_first_seen_df["server_name"] = (
    date_first_seen_df["record_id"]
    .map(server_mapping)
)


# ================================
# 2) Normalize keys
# ================================
def norm_key(s):
    return s.astype(str).str.strip()

for df_ in (records_hierarchy_df, date_first_seen_df):
    df_["record_id"] = norm_key(df_["record_id"])
    df_["server_name"] = norm_key(df_["server_name"])

df_filtered["record_id"] = norm_key(df_filtered["record_id"])
df_filtered["server_name"] = norm_key(df_filtered["server_name"])

# ================================
# 3) Ensure each RHS table is unique on (record_id, server_name)
# ================================
records_hierarchy_df = records_hierarchy_df.drop_duplicates(["record_id", "server_name"])
date_first_seen_df   = date_first_seen_df.drop_duplicates(["record_id", "server_name"])

# ================================
# 4) Build MASTER pairs from records_hierarchy_df
#    (this is what you said you want to keep)
# ================================
master_pairs = set(zip(records_hierarchy_df["record_id"], records_hierarchy_df["server_name"]))

# Filter helpers
def filter_to_master_pairs(df: pd.DataFrame) -> pd.DataFrame:
    pairs = list(zip(df["record_id"], df["server_name"]))
    return df.loc[pd.Series(pairs, index=df.index).isin(master_pairs)].copy()

# ================================
# 5) Filter data and date_first_seen to MASTER pairs
# ================================
data_master = filter_to_master_pairs(df_filtered) # data
date_first_seen_master = filter_to_master_pairs(date_first_seen_df)

# Optional safety: also dedupe data on the same key (should not change if clean)
data_master = data_master.drop_duplicates(["record_id", "server_name"], keep="first")

# ================================
# 6) Merge (left join from MASTER DATA)
# ================================
join_keys = ["record_id", "server_name"]

data_clean_hierarchy = (
    data_master
      .merge(records_hierarchy_df, on=join_keys, how="left", validate="one_to_one")
      .merge(date_first_seen_master, on=join_keys, how="left", validate="one_to_one")
)

# ================================
# 7) Sanity checks (correct ones)
# ================================
print("Master pairs (records_hierarchy_df rows):", len(records_hierarchy_df))
print("Master unique record_id:", records_hierarchy_df["record_id"].nunique())

print("Rows in raw data:", len(data))
print("Rows in raw df_filtered:", len(df_filtered))
print("Rows in data after filtering to master pairs:", len(data_master))

print("Rows in date_first_seen after filtering to master pairs:", len(date_first_seen_master))
print("Final rows in data_clean_hierarchy:", len(data_clean_hierarchy))

print("\nMissing records_hierarchy:", data_clean_hierarchy["records_hierarchy"].isna().sum())
print("Missing date_first_seen:", data_clean_hierarchy["date_first_seen"].isna().sum())
print("Missing publication_year_first_seen:", data_clean_hierarchy["publication_year_first_seen"].isna().sum())

print("\nDuplicates on (record_id, server_name) in final:",
      data_clean_hierarchy.duplicated(["record_id","server_name"]).sum())

print("\nHierarchy counts:")
print(data_clean_hierarchy["records_hierarchy"].value_counts(dropna=False).head(30))


Master pairs (records_hierarchy_df rows): 7966281
Master unique record_id: 7966281
Rows in raw data: 7984655
Rows in raw df_filtered: 7970011
Rows in data after filtering to master pairs: 7961690
Rows in date_first_seen after filtering to master pairs: 7966281
Final rows in data_clean_hierarchy: 7961690

Missing records_hierarchy: 2593
Missing date_first_seen: 1824
Missing publication_year_first_seen: 0

Duplicates on (record_id, server_name) in final: 0

Hierarchy counts:
records_hierarchy
parent                              7821123
version                              102318
publish_version                       12888
mirror (AgEcon Search)                 6607
mirror (arXiv)                         6419
part_of                                5921
NaN                                    2593
child                                  2061
mirror (ResearchGate)                   827
correction                              354
comment                                 242
mirror (Zenodo)     

# limit data to those between 1990 to 2025 include

In [21]:
date_first_seen_df[~date_first_seen_df['raw_dates'].isna()]

,record_id,server_name,date_first_seen,publication_year,publication_year_first_seen,raw_dates
10240,datacite::10.11588/artdok.00004120,ART-Dok,2016-06-14,2016.0,2016,"[{""date"": ""2016"", ""dateType"": ""Issued""}]"
10241,datacite::10.11588/artdok.00004128,ART-Dok,2016-06-14,2016.0,2016,"[{""date"": ""2016"", ""dateType"": ""Issued""}]"
10242,datacite::10.11588/artdok.00004129,ART-Dok,2016-06-14,2016.0,2016,"[{""date"": ""2016"", ""dateType"": ""Issued""}]"
10243,datacite::10.11588/artdok.00004121,ART-Dok,2016-06-14,2016.0,2016,"[{""date"": ""2016"", ""dateType"": ""Issued""}]"
10244,datacite::10.11588/artdok.00004133,ART-Dok,2016-06-14,2016.0,2016,"[{""date"": ""2016"", ""dateType"": ""Issued""}]"
...,...,...,...,...,...,...
8031087,datacite::10.48550/arxiv.2501.12237,arXiv,2025-01-21,2025.0,2025,"[{""date"": ""2025-01-21T15:59:03Z"", ""dateInforma..."
8031088,datacite::10.48550/arxiv.2501.12238,arXiv,2025-01-21,2025.0,2025,"[{""date"": ""2025-01-21T15:59:19Z"", ""dateInforma..."
8031089,datacite::10.48550/arxiv.2501.12248,arXiv,2025-01-21,2025.0,2025,"[{""date"": ""2025-01-21T16:09:02Z"", ""dateInforma..."
8031090,datacite::10.48550/arxiv.2501.12281,arXiv,2025-01-21,2025.0,2025,"[{""date"": ""2025-01-21T16:52:42Z"", ""dateInforma..."


In [22]:
# ------------------------------------------------------------------
# ASSUMPTIONS
# ------------------------------------------------------------------
# Main dataframe: df_filtered
# Server column: server_name
# Year column  : publication_year_first_seen
#
# Replace "publication_year" below if your year column has another name.
# ------------------------------------------------------------------

YEAR_COL = "publication_year_first_seen"

# Make sure year column is numeric
data_clean_hierarchy[YEAR_COL] = pd.to_numeric(
    data_clean_hierarchy[YEAR_COL],
    errors="coerce"
)

# ------------------------------------------------------------------
# Create summary dataframe
# ------------------------------------------------------------------
server_year_counts = (
    data_clean_hierarchy
    .groupby("server_name")
    .agg(
        count_1990_2025=(YEAR_COL, lambda x: ((x >= 1990) & (x <= 2025)).sum()),
        count_before_1990=(YEAR_COL, lambda x: (x < 1990).sum()),
        count_after_2025=(YEAR_COL, lambda x: (x > 2025).sum()),
        count_missing_year=(YEAR_COL, lambda x: x.isna().sum()),
        total_records=(YEAR_COL, "size")
    )
    .reset_index()
    .sort_values(
        by="count_1990_2025",
        ascending=False
    )
)

# ------------------------------------------------------------------
# Display
# ------------------------------------------------------------------
# Optional: save
server_year_counts.to_csv("outputs_new/tracker_data/server_year_counts_before_dedupe.csv", index=False)

server_year_counts.head(60)

,server_name,count_1990_2025,count_before_1990,count_after_2025,count_missing_year,total_records
101,arXiv,2920789,8,0,0,2920797
86,SSRN,1258585,671,0,0,1259256
42,HAL,1056415,0,0,0,1056415
82,Research Square,450818,0,0,0,450818
81,RePEc: Research Papers in Economics,389398,0,0,0,389398
102,bioRxiv,306948,0,0,0,306948
83,ResearchGate,180896,335,0,0,181231
100,Zenodo,166765,21,0,0,166786
7,AgEcon Search,140430,47743,0,0,188173
76,Preprints.org,115815,0,0,0,115815


In [23]:
server_year_counts.tail(60)

,server_name,count_1990_2025,count_before_1990,count_after_2025,count_missing_year,total_records
6,AfricArXiv,2184,6,0,0,2190
74,PhilSci-Archive,2167,0,0,0,2167
88,ScienceOpen Preprints,1961,0,0,0,1961
67,Open Research Europe,1877,0,0,0,1877
62,National Bureau of Economic Research,1828,0,0,0,1828
55,Law Archive,1808,0,0,0,1808
84,ResearchHub,1636,0,0,0,1636
18,CogPrints,1498,0,0,0,1498
2,APSA Preprints,1470,0,0,0,1470
71,PREPRINTS.RU,1415,0,0,0,1415


In [24]:
server_year_counts['count_missing_year'].value_counts()

count_missing_year
0    109
Name: count, dtype: int64

In [25]:
data_clean_hierarchy[data_clean_hierarchy['server_name']=='Arabixiv']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,date_first_seen,publication_year,publication_year_first_seen,raw_dates
207840,crossref::10.31221/osf.io/4mqu5,Arabixiv,crossref,10.31221/osf.io/4mqu5,https://doi.org/10.31221/osf.io/4mqu5,https://osf.io/4mqu5,posted-content,preprint,اضطراب الذاتوية: بين الصعوبات التشخيصية والآفا...,"مجلة العلوم النفسية والتربوية, مجلة العلوم الن...",None,None,None,None,,,,,false,None,None,None,None,parent,2018-02-16,2018.0,2018,None
207841,crossref::10.31221/osf.io/ys2xq,Arabixiv,crossref,10.31221/osf.io/ys2xq,https://doi.org/10.31221/osf.io/ys2xq,https://osf.io/ys2xq,posted-content,preprint,فعالية أسلوبي التعزيز والنمذجة في خفض مستوى ال...,"مجلة العلوم النفسية والتربوية, مجلة العلوم الن...",None,None,None,None,,,,,false,None,None,None,None,parent,2018-02-16,2018.0,2018,None
207842,crossref::10.31221/osf.io/yejkf,Arabixiv,crossref,10.31221/osf.io/yejkf,https://doi.org/10.31221/osf.io/yejkf,https://osf.io/yejkf,posted-content,preprint,مستوى النرجسية لدى المراهق الجزائري المتمدرس د...,"مجلة العلوم النفسية والتربوية, مجلة العلوم الن...",None,None,None,None,,,,,false,None,None,None,None,parent,2018-02-16,2018.0,2018,None
207843,crossref::10.31221/osf.io/wyrmc,Arabixiv,crossref,10.31221/osf.io/wyrmc,https://doi.org/10.31221/osf.io/wyrmc,https://osf.io/wyrmc,posted-content,preprint,التفكير الابتكاري لدى تلامذة المرحلة التحضيري...,"مجلة العلوم النفسية والتربوية, مجلة العلوم الن...",None,None,None,None,,,,,false,None,None,None,None,parent,2018-02-16,2018.0,2018,None
207844,crossref::10.31221/osf.io/rczk8,Arabixiv,crossref,10.31221/osf.io/rczk8,https://doi.org/10.31221/osf.io/rczk8,https://osf.io/rczk8,posted-content,preprint,خبرة الطلبة الموقوفين عن الدراسة مؤقتا بسبب ال...,"مجلة العلوم النفسية والتربوية, مجلة العلوم الن...",None,None,None,None,,,,,false,None,None,None,None,parent,2018-01-26,2018.0,2018,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1845412,datacite::10.17605/osf.io/t734r,Arabixiv,datacite,10.17605/osf.io/t734r,https://doi.org/10.17605/osf.io/t734r,https://arabixiv.org/t734r/,Text,Preprint,ما هي المساحة التي يحتاجها جميع البشر ليوم الحشر؟,"Moustafa, Khaled",None,None,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""cos.osf"", ""type"": ...",parent,2018-06-25,2018.0,2018,"[{""date"": ""2018-06-25T06:08:49.597186+00:00"", ..."
1845441,datacite::10.17605/osf.io/r6xd9,Arabixiv,datacite,10.17605/osf.io/r6xd9,https://doi.org/10.17605/osf.io/r6xd9,https://arabixiv.org/r6xd9/,Text,Preprint,السلوك السوي والمضطرب عند الأطفال: تحديده، وأس...,"Rudwan, Samer",None,None,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""cos.osf"", ""type"": ...",parent,2018-06-26,2018.0,2018,"[{""date"": ""2018-06-26T01:12:50.342394+00:00"", ..."
1845493,datacite::10.17605/osf.io/vc3qm,Arabixiv,datacite,10.17605/osf.io/vc3qm,https://doi.org/10.17605/osf.io/vc3qm,https://arabixiv.org/vc3qm/,Text,Preprint,معجم مصطلحات كرة السلة,"Alshaikhi, Yahya",None,None,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""cos.osf"", ""type"": ...",parent,2018-06-28,2018.0,2018,"[{""date"": ""2018-06-28T07:34:00.740036+00:00"", ..."
1845495,datacite::10.17605/osf.io/9t8x6,Arabixiv,datacite,10.17605/osf.io/9t8x6,https://doi.org/10.17605/osf.io/9t8x6,https://arabixiv.org/9t8x6/,Text,Preprint,العِزّة: تطبيق لوحة مفاتيح عربية ذكية للأجهزة ...,"Bouhadjera, Abdelmalek",None,None,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""cos.osf"", ""type"": ...",parent,2018-06-28,2018.0,2018,"[{""date"": ""2018-06-28T10:00:42.442091+00:00"", ..."


In [26]:
# ------------------------------------------------------------------
# ASSUMPTIONS
# ------------------------------------------------------------------
# Main dataframe: df_filtered
# Server column: server_name
# Year column  : publication_year_first_seen
#
# Replace "publication_year" below if your year column has another name.
# ------------------------------------------------------------------

YEAR_COL = "publication_year"

# Make sure year column is numeric
data_clean_hierarchy[YEAR_COL] = pd.to_numeric(
    data_clean_hierarchy[YEAR_COL],
    errors="coerce"
)

# ------------------------------------------------------------------
# Create summary dataframe
# ------------------------------------------------------------------
server_year_counts = (
    data_clean_hierarchy
    .groupby("server_name")
    .agg(
        count_1990_2025=(YEAR_COL, lambda x: ((x >= 1990) & (x <= 2025)).sum()),
        count_before_1990=(YEAR_COL, lambda x: (x < 1990).sum()),
        count_after_2025=(YEAR_COL, lambda x: (x > 2025).sum()),
        count_missing_year=(YEAR_COL, lambda x: x.isna().sum()),
        total_records=(YEAR_COL, "size")
    )
    .reset_index()
    .sort_values(
        by="count_1990_2025",
        ascending=False
    )
)

# ------------------------------------------------------------------
# Display
# ------------------------------------------------------------------
# Optional: save
server_year_counts.to_csv("outputs_new/tracker_data/server_year_counts_before_dedupe2.csv", index=False)

server_year_counts.head(60)

,server_name,count_1990_2025,count_before_1990,count_after_2025,count_missing_year,total_records
101,arXiv,2920797,0,0,0,2920797
86,SSRN,1258578,671,7,0,1259256
42,HAL,1056415,0,0,0,1056415
82,Research Square,450818,0,0,0,450818
81,RePEc: Research Papers in Economics,389398,0,0,0,389398
102,bioRxiv,306948,0,0,0,306948
83,ResearchGate,180860,335,36,0,181231
100,Zenodo,166112,21,653,0,166786
7,AgEcon Search,140392,47743,38,0,188173
76,Preprints.org,115815,0,0,0,115815


In [27]:
server_year_counts.tail(60)

,server_name,count_1990_2025,count_before_1990,count_after_2025,count_missing_year,total_records
6,AfricArXiv,2184,6,0,0,2190
74,PhilSci-Archive,2167,0,0,0,2167
88,ScienceOpen Preprints,1961,0,0,0,1961
67,Open Research Europe,1877,0,0,0,1877
62,National Bureau of Economic Research,1828,0,0,0,1828
55,Law Archive,1808,0,0,0,1808
84,ResearchHub,1636,0,0,0,1636
18,CogPrints,1498,0,0,0,1498
2,APSA Preprints,1470,0,0,0,1470
71,PREPRINTS.RU,1415,0,0,0,1415


In [28]:
# ── Correct way ───────────────────────────────────────────────────────────────

# Option 1 — isin with explicit range (your original intent)
data_clean_hierarchy_1990_2025 = data_clean_hierarchy[
    data_clean_hierarchy["publication_year_first_seen"].isin(range(1990, 2026))
]

# # Option 2 — between (cleaner, faster for large DataFrames)
# data_clean_hierarchy_1990_2025 = data_clean_hierarchy_1990_2025[
#     data_clean_hierarchy["publication_year"].isin(range(1990, 2026))
# ]


print(data_clean_hierarchy_1990_2025.shape)
print(data_clean_hierarchy_1990_2025["publication_year_first_seen"].value_counts().sort_index())

(7912655, 28)
publication_year_first_seen
1990     12143
1991     13414
1992     17728
1993     21510
1994     26980
1995     31242
1996     35247
1997     40908
1998     48009
1999     53483
2000     61169
2001     66518
2002     74695
2003     87834
2004     96860
2005    112803
2006    125761
2007    134501
2008    146606
2009    159119
2010    173729
2011    187460
2012    202937
2013    216086
2014    227606
2015    240633
2016    258193
2017    289075
2018    338396
2019    391139
2020    514635
2021    541539
2022    589638
2023    659772
2024    742950
2025    972337
Name: count, dtype: int64


In [29]:
data_clean_hierarchy_1990_2025 = data_clean_hierarchy_1990_2025[
    data_clean_hierarchy_1990_2025["publication_year"].isin(range(1990, 2026))
]

In [30]:
data_clean_hierarchy = data_clean_hierarchy_1990_2025.copy()

In [31]:
data_clean_hierarchy

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,date_first_seen,publication_year,publication_year_first_seen,raw_dates
0,crossref::10.21467/preprints.48,AIJR Preprints,crossref,10.21467/preprints.48,https://doi.org/10.21467/preprints.48,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,"Bird’s Eye View on the Diagnosis, Treatment, &...","Panchalingala, Sai Bhargavi",None,None,None,None,,,,,false,None,None,None,None,parent,2020-05-03,2020.0,2020,None
1,crossref::10.21467/preprints.43,AIJR Preprints,crossref,10.21467/preprints.43,https://doi.org/10.21467/preprints.43,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Doxycycline and Minocycline Drugs as a Treatme...,"Mostafa, Mohamed",None,None,None,None,,,,,false,None,None,None,None,parent,2020-04-25,2020.0,2020,None
2,crossref::10.21467/preprints.39,AIJR Preprints,crossref,10.21467/preprints.39,https://doi.org/10.21467/preprints.39,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,A Genetic Perspective of 2019-nCoV in Relation...,"Dasgupta, Rimjhim",None,None,None,None,,,,,false,None,None,None,None,parent,2020-04-16,2020.0,2020,None
3,crossref::10.21467/preprints.38,AIJR Preprints,crossref,10.21467/preprints.38,https://doi.org/10.21467/preprints.38,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Marine Algae as a Natural Source for Antiviral...,"Musale, Amar S; G., Raja Krishna Kumar; Sapre,...",None,None,None,None,,,,,false,None,None,None,None,parent,2020-04-15,2020.0,2020,None
4,crossref::10.21467/preprints.36,AIJR Preprints,crossref,10.21467/preprints.36,https://doi.org/10.21467/preprints.36,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Possible Prevention of COVID 19 by Using Linol...,"Subhash, Venkata; G, Raja Krishna Kumar; Sapre...",None,None,None,None,,,,,false,None,None,None,None,parent,2020-04-15,2020.0,2020,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7961685,openalex::W999325625,viXra,openalex,None,None,https://vixra.org/pdf/1409.0090v1.pdf,preprint,None,Three Objections to Modern Physics,Lubomir Vlcek,None,None,None,None,None,None,None,None,None,None,None,None,None,parent,2014-09-01,2014.0,2014,None
7961686,openalex::W999460032,viXra,openalex,None,None,https://vixra.org/abs/1112.0094,preprint,None,Particle Mass Ratios,DT Froedge,None,None,None,None,None,None,None,None,None,None,None,None,None,parent,2011-12-01,2011.0,2011,None
7961687,openalex::W99967155,viXra,openalex,None,None,https://vixra.org/pdf/1406.0019v1.pdf,preprint,None,Quantum FFF Theory Proposals for Some Unsolved...,Leo Vuyk,None,None,None,None,None,None,None,None,None,None,None,None,None,parent,2014-06-01,2014.0,2014,None
7961688,openalex::W999790414,viXra,openalex,None,None,https://vixra.org/pdf/1306.0105v3.pdf,preprint,None,Investigation of the Formalism of Particle Dyn...,Chi-Yi Chen,None,None,None,None,None,None,None,None,None,None,None,None,None,parent,2013-06-01,2013.0,2013,None


## explore

In [32]:
# 1. Create the backup copy in a new column
data_clean_hierarchy["records_hierarchy_backup"] = data_clean_hierarchy["records_hierarchy"].copy()

# 2. Overwrite all cells in the original column with the string 'parent'
data_clean_hierarchy["records_hierarchy"] = 'parent'


In [33]:
data_clean_hierarchy["records_hierarchy"].value_counts(dropna=False).head(60)

records_hierarchy
parent    7911901
Name: count, dtype: int64

In [34]:
data_clean_hierarchy["records_hierarchy_backup"].value_counts(dropna=False).head(60)

records_hierarchy_backup
parent                              7771465
version                              102316
publish_version                       12888
mirror (AgEcon Search)                 6607
mirror (arXiv)                         6419
part_of                                5921
NaN                                    2464
child                                  2061
mirror (ResearchGate)                   827
correction                              354
comment                                 242
mirror (Zenodo)                         191
mirror (bioRxiv)                         29
review                                   27
mirror (SSRN)                            23
mirror (Open Science Framework)          21
mirror (Humanities Commons CORE)         16
others                                   12
parent_duplicate                          4
mirror (AfricArXiv)                       2
mirror (Research Square)                  2
mirror (EarthArXiv)                       2
mirror 

In [35]:
data_clean_hierarchy.count()

record_id                      7911901
server_name                    7911901
backend                        7911901
doi                            6286069
doi_url                        6286069
landing_page_url               7832455
type_backend_raw               7910607
subtype_backend_raw            5406608
title                          7911892
authors_flat                   7891285
institutions_flat              1824170
countries_flat                  865124
relations_json                 4040199
version_label                  2974228
is_version_of                  6220565
is_preprint_of                 6220565
has_preprint                   6220565
has_review                     6220565
has_published_version          6220565
published_version_ids_json           0
version_of_ids_json                  0
update_to_json                    8915
raw_relationships_json         3468481
records_hierarchy              7911901
date_first_seen                7911901
publication_year         

In [36]:
len(data_clean_hierarchy["server_name"].value_counts())

109

In [37]:
# data_clean_hierarchy = data_clean_hierarchy.sort_values(by='record_id')

# dedupe on title+authors (+ optional year)

## function

In [38]:
"""
Reproducible 2-pass dedupe pipeline (Exact pass -> Fuzzy pass) with:
- Strong-but-cheap title normalization (cached)
- 3 author signatures: tokenbag | last_initial | last
- Stage A strict (title + authors_fp) exact
- Stage B relaxed (shared authors overlap) within exact-title groups (optional per stage)
- Optional fuzzy title fallback (token containment) BLOCKED by authors_fp (+ optional year)
- Prefilter modes:
    * title_dup  : keep rows where cleaned title repeats (fast exact stages)
    * author_dup : keep rows where authors_fp repeats (enables fuzzy stages when titles differ)
    * none       : keep all eligible (debug)

Includes:
- Metrics counters per stage
- Summary printing + early stop
- Deterministic labeling
- Designed for speed + low false positives (especially with last_initial)

USAGE:
1) Define STAGES_EXACT and STAGES_FUZZY
2) Run:
   df_out, metrics = run_dedupe_pipeline_two_passes(
       df,
       stages_exact=STAGES_EXACT,
       stages_fuzzy=STAGES_FUZZY,
       early_stop_if_new_labels_lt=500,
       print_summary=True,
       return_all_metrics=True,
       servers=None,
       across_servers=True,
       use_year=False,
       choose_parent="oldest",
       prefilter=True,
       date_candidates=('date_first_seen',),
       hierarchy_col="records_hierarchy",
       parent_id_col="parent_record_id",
       group_id_col="dup_group_id",
       add_authors_fingerprint_col=True,
       add_title_clean_col=True,
   )
"""

import pandas as pd
import numpy as np
import re
import time
import unicodedata
from typing import Iterable, Optional, Dict, Any, Tuple, List

# ============================================================
# 0) Regex + NA helpers
# ============================================================
_WS = re.compile(r"\s+")
_PUNCT_ALL = re.compile(r"[^\w\s]", re.UNICODE)  # remove everything except word chars + spaces
NA_LIKE = {"", "none", "null", "nan", "n/a", "[]", "{}", "na"}

# ============================================================
# 0b) Generic / risky title filter
# ============================================================
# All values already lowercase; _clean_title_series_v2 lowercases before comparison.
GENERIC_TITLES: frozenset[str] = frozenset({
    "front matter", "back matter", "summary", "summaries",
    "reviews in brief", "publications received",
    "agricultural letter", "administrasi kurikulum",
    "administrasi peserta didik", "experiment ended",
    "editorial", "introduction", "preface", "contents",
    "table of contents", "index", "book review", "letter",
    "news", "announcement", "abstract", "poster", "supplement",
    "no title", "résumés", "rãésumãés", "retracted",      # keep both accent forms
    "bibliographie", "bremsstrahlung",
})

MIN_TITLE_TOKENS: int = 3      # centralised so every stage uses the same threshold
MIN_TITLE_CHARS: int = 25


def is_risky_title(title_clean: str) -> bool:
    """
    Returns True when a cleaned title should be EXCLUDED from deduplication.

    Designed to operate on the OUTPUT of _clean_title_series_v2 (already
    lowercase, accent-stripped, punctuation-removed), so GENERIC_TITLES
    matching is automatically case-insensitive.
    """
    if not title_clean:
        return True

    # Exact match against the generic-title blocklist
    if title_clean in GENERIC_TITLES:
        return True

    tokens = title_clean.split()

    # Too few tokens
    if len(tokens) < MIN_TITLE_TOKENS:
        return True

    # Too short in characters
    if len(title_clean) < MIN_TITLE_CHARS:
        return True

    return False


def _blank_risky_titles(title_clean_series: pd.Series) -> pd.Series:
    """
    Vectorised wrapper: returns a copy of the series with risky titles
    replaced by "" so they are excluded downstream.
    """
    return title_clean_series.where(
        ~title_clean_series.apply(is_risky_title),
        other="",
    )
    
# ============================================================
# 1) Utility: pick a date column + record_id numeric fallback
# ============================================================
def _pick_first_existing(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _record_id_key(s: pd.Series) -> pd.Series:
    """Fast numeric key from record_id (extract first digits)."""
    digits = s.astype("string").str.extract(r"(\d+)")[0]
    return pd.to_numeric(digits, errors="coerce")


# ============================================================
# 2) Title normalization (cheap, high ROI) + token containment
# ============================================================
def _strip_accents_text(x: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c)
    )


def _clean_title_series_v2(s: pd.Series) -> pd.Series:
    """
    Strong-but-cheap title normalization:
      - lowercase
      - strip accents
      - remove punctuation -> spaces
      - collapse whitespace
    """
    s = s.astype("string").fillna("").str.strip().str.lower()
    s = s.where(~s.isin(list(NA_LIKE)), "")
    s = s.apply(_strip_accents_text)
    s = s.str.replace(_PUNCT_ALL, " ", regex=True)
    s = s.str.replace(_WS, " ", regex=True).str.strip()
    return s


def _title_tokens_from_clean(title_clean: str) -> List[str]:
    """Tokenize already-clean title into tokens; drop very short tokens (len < 2)."""
    if not title_clean:
        return []
    return [t for t in title_clean.split(" ") if len(t) >= 2]


def _containment_score(a_tokens: List[str], b_tokens: List[str]) -> float:
    """
    Containment score:
        |A ∩ B| / min(|A|, |B|)
    Good for small title differences when tokens still mostly match.
    """
    if not a_tokens or not b_tokens:
        return 0.0
    A, B = set(a_tokens), set(b_tokens)
    denom = min(len(A), len(B))
    if denom <= 0:
        return 0.0
    return len(A & B) / denom


# ============================================================
# 3) Author canonicalization (3 modes)
# ============================================================
def _strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )


def _normalize_one_author_tokenbag(author: str) -> str:
    """
    Token-bag per author:
    - remove punctuation
    - split tokens
    - sort tokens within author
    - join with "_"
    """
    if not author:
        return ""
    a = _strip_accents(str(author)).lower().strip()
    if not a or a in NA_LIKE:
        return ""
    a = _PUNCT_ALL.sub(" ", a)
    a = _WS.sub(" ", a).strip()
    if not a:
        return ""
    toks = [t for t in a.split(" ") if t]
    if not toks:
        return ""
    toks = sorted(toks)
    return "_".join(toks)


def _normalize_one_author_last_initial(author: str) -> str:
    """
    Middle-ground signature: "last|first_initial"
    Rules:
      - If comma: "Last, First ..." -> last = first token before comma;
                                     initial = first token after comma (first-name token only)
      - If no comma: "First ... Last" -> last = last token; initial = first token
      - If we can't find an initial, return "" (reduces false positives)
    """
    if not author:
        return ""
    a = _strip_accents(str(author)).lower().strip()
    if not a or a in NA_LIKE:
        return ""

    if "," in a:
        left, right = a.split(",", 1)
        left = _PUNCT_ALL.sub(" ", left)
        right = _PUNCT_ALL.sub(" ", right)
        left = _WS.sub(" ", left).strip()
        right = _WS.sub(" ", right).strip()
        if not left:
            return ""
        last_toks = [t for t in left.split(" ") if t]
        if not last_toks:
            return ""
        last = last_toks[0]  # keep your "first token if multi-token surname" philosophy

        first_toks = [t for t in right.split(" ") if t]
        if not first_toks:
            return ""  # avoid false positives
        ini = first_toks[0][:1]
        return f"{last}|{ini}" if ini else ""
    else:
        a = _PUNCT_ALL.sub(" ", a)
        a = _WS.sub(" ", a).strip()
        toks = [t for t in a.split(" ") if t]
        if len(toks) < 2:
            return ""
        ini = toks[0][:1]
        last = toks[-1]
        return f"{last}|{ini}" if (ini and last) else ""


def _normalize_one_author_last(author: str) -> str:
    """Last-name-only signature (high recall, more false positives)."""
    if not author:
        return ""
    a = _strip_accents(str(author)).lower().strip()
    if not a or a in NA_LIKE:
        return ""

    if "," in a:
        left = a.split(",", 1)[0].strip()
        left = _PUNCT_ALL.sub(" ", left)
        left = _WS.sub(" ", left).strip()
        if not left:
            return ""
        toks = [t for t in left.split(" ") if t]
        if not toks:
            return ""
        return toks[0]
    else:
        a = _PUNCT_ALL.sub(" ", a)
        a = _WS.sub(" ", a).strip()
        toks = [t for t in a.split(" ") if t]
        if not toks:
            return ""
        return toks[-1]


def build_authors_fingerprint_series(authors_flat: pd.Series, mode: str) -> pd.Series:
    """
    Build author fingerprint per row:
      - split authors on ';'
      - normalize each author (depends on mode)
      - drop empties
      - dedupe within row
      - sort
      - join with ';'
    """
    if mode not in {"tokenbag", "last_initial", "last"}:
        raise ValueError("mode must be tokenbag | last_initial | last")

    s = authors_flat.astype("string").fillna("").str.strip()
    s = s.where(~s.str.lower().isin(list(NA_LIKE)), "")

    if mode == "tokenbag":
        norm_fn = _normalize_one_author_tokenbag
    elif mode == "last_initial":
        norm_fn = _normalize_one_author_last_initial
    else:
        norm_fn = _normalize_one_author_last

    def row_to_fp(x: str) -> str:
        if not x:
            return ""
        authors = [a.strip() for a in str(x).split(";") if a.strip()]
        norm = [norm_fn(a) for a in authors]
        norm = [z for z in norm if z]
        norm = sorted(set(norm))
        return ";".join(norm)

    return s.apply(row_to_fp)


def _author_tokens_from_fp(fp: str) -> List[str]:
    if not fp:
        return []
    return [t for t in fp.split(";") if t]


def _overlap_count(a_tokens: List[str], b_tokens: List[str]) -> int:
    if not a_tokens or not b_tokens:
        return 0
    return len(set(a_tokens) & set(b_tokens))


# ============================================================
# 4) Single-stage dedupe:
#    - Prefilter (title_dup/author_dup/none)
#    - Stage A strict (title + authors_fp) exact match
#    - Optional fuzzy title fallback (within same authors_fp)
#    - Optional Stage B relaxed (shared authors overlap) within exact-title groups
# ============================================================
def dedupe_title_authors_stage(
    df: pd.DataFrame,
    *,
    # stage config
    stage_name: str = "stage",
    authors_fp_mode: str = "tokenbag",         # tokenbag | last_initial | last

    # fuzzy config (title containment), executed only if enabled
    title_fuzzy_fallback: bool = False,
    min_title_tokens: int = 6,
    min_title_containment: float = 0.70,
    fuzzy_compare_strategy: str = "parent_only",  # parent_only | all_pairs_small (parent_only is safest/fastest)

    # relaxed (shared authors overlap) config (exact title only)
    relaxed_shared_authors: bool = True,
    min_authors_required: int = 2,
    min_shared_authors: int = 2,

    # prefilter strategy (important!)
    prefilter_mode: str = "title_dup",         # title_dup | author_dup | none
    prefilter: bool = True,                    # if False, skip ">=2" group filter (slower)

    # global options
    servers=None,
    across_servers: bool = True,
    use_year: bool = False,
    choose_parent: str = "oldest",             # oldest | most_recent
    overwrite_mode: str = "parent_only",       # any | parent_only | unlabeled_only

    # columns
    server_col: str = "server_name",
    record_id_col: str = "record_id",
    title_col: str = "title",
    authors_col: str = "authors_flat",
    year_col: str = "publication_year_first_seen",
    date_candidates: Tuple[str, ...] = ("date_first_seen",),

    hierarchy_col: str = "records_hierarchy",
    parent_id_col: str = "parent_record_id",
    group_id_col: str = "dup_group_id",

    # caching/debug columns
    add_authors_fingerprint_col: bool = True,
    authors_fingerprint_col: str = "authors_fp",
    add_title_clean_col: bool = True,
    title_clean_col: str = "title_clean_v2",

    return_metrics: bool = False,
) -> pd.DataFrame | Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    One dedupe stage. Designed to be composed into a multi-stage pipeline.
    """

    t0 = time.perf_counter()

    metrics: Dict[str, Any] = {
        "stage_name": stage_name,
        "n_rows_df": int(len(df)),
        "n_candidates_initial": 0,
        "prefilter_mode": prefilter_mode,
        "prefilter_rows": 0,
        "prefilter_groups": 0,
        "work_rows_after_keys": 0,

        "stageA_groups": 0,
        "stageA_children_labeled": 0,

        "fuzzy_enabled": bool(title_fuzzy_fallback),
        "fuzzy_groups": 0,
        "fuzzy_pairs_checked": 0,
        "fuzzy_children_labeled": 0,

        "stageB_enabled": bool(relaxed_shared_authors),
        "stageB_title_groups": 0,
        "stageB_clusters": 0,
        "stageB_children_labeled": 0,

        "time_s": 0.0,
    }

    # ------------------------------------------------------
    # Ensure output cols exist
    # ------------------------------------------------------
    for c in (hierarchy_col, parent_id_col, group_id_col):
        if c not in df.columns:
            df[c] = pd.NA

    if add_authors_fingerprint_col and authors_fingerprint_col not in df.columns:
        df[authors_fingerprint_col] = pd.NA
    if add_title_clean_col and title_clean_col not in df.columns:
        df[title_clean_col] = pd.NA

    # ------------------------------------------------------
    # Eligibility
    # ------------------------------------------------------
    h = df[hierarchy_col]
    if overwrite_mode == "any":
        eligible = pd.Series(True, index=df.index)
    elif overwrite_mode == "parent_only":
        eligible = h.astype("string").str.lower().str.strip().eq("parent")
    elif overwrite_mode == "unlabeled_only":
        eligible = h.isna()
    else:
        raise ValueError("overwrite_mode must be any | parent_only | unlabeled_only")

    # server filter
    if servers is None:
        server_mask = pd.Series(True, index=df.index)
    elif isinstance(servers, str):
        server_mask = df[server_col].eq(servers)
    else:
        server_mask = df[server_col].isin(list(servers))

    m = eligible & server_mask
    metrics["n_candidates_initial"] = int(m.sum())
    if not m.any():
        metrics["time_s"] = time.perf_counter() - t0
        return (df, metrics) if return_metrics else df

    # ------------------------------------------------------
    # Prefilter: decide which indices to consider in this stage
    # ------------------------------------------------------
    if prefilter_mode == "title_dup":
        # title-based prefilter (fast for exact title stages)
        t_clean = df.loc[m, title_clean_col] if (add_title_clean_col and title_clean_col in df.columns and df.loc[m, title_clean_col].notna().any()) else None
        if t_clean is None:
            t_clean = _clean_title_series_v2(df.loc[m, title_col])
        vc = t_clean.value_counts()
        keep_idx = t_clean[t_clean.isin(vc[vc >= 2].index)].index
        metrics["prefilter_groups"] = int((vc >= 2).sum())

    elif prefilter_mode == "author_dup":
        # author-fp based prefilter (crucial for fuzzy pass; titles may differ)
        # compute fp only for m rows
        a_fp = build_authors_fingerprint_series(df.loc[m, authors_col], mode=authors_fp_mode)
        vc = a_fp.value_counts()
        keep_idx = a_fp[a_fp.isin(vc[vc >= 2].index)].index
        metrics["prefilter_groups"] = int((vc >= 2).sum())

    elif prefilter_mode == "none":
        keep_idx = df.index[m]
        metrics["prefilter_groups"] = 0

    else:
        raise ValueError("prefilter_mode must be title_dup | author_dup | none")

    metrics["prefilter_rows"] = int(len(keep_idx))
    if len(keep_idx) == 0:
        metrics["time_s"] = time.perf_counter() - t0
        return (df, metrics) if return_metrics else df

    # ------------------------------------------------------
    # Work subset + compute/attach cached normalization keys
    # ------------------------------------------------------
    cols_needed = [server_col, record_id_col, title_col, authors_col]
    if use_year:
        cols_needed.append(year_col)
    date_col = _pick_first_existing(df, date_candidates)
    if date_col:
        cols_needed.append(date_col)

    work = df.loc[keep_idx, cols_needed].copy()

    # Title clean (cache to df if asked)
    if add_title_clean_col:
        # compute for missing only (cheap)
        t_missing = df.loc[work.index, title_clean_col].isna()
        if t_missing.any():
            df.loc[work.index[t_missing], title_clean_col] = _clean_title_series_v2(df.loc[work.index[t_missing], title_col]).values
        work["_t"] = df.loc[work.index, title_clean_col].astype("string").fillna("")
    else:
        work["_t"] = _clean_title_series_v2(work[title_col])
    # ── NEW: blank out generic / too-short titles before any key is built ──
    work["_t"] = _blank_risky_titles(work["_t"])
    
    # Authors fp (mode-specific; cache into df column if asked)
    work["_a_fp"] = build_authors_fingerprint_series(work[authors_col], mode=authors_fp_mode)
    if add_authors_fingerprint_col:
        df.loc[work.index, authors_fingerprint_col] = work["_a_fp"].values

    # Year (optional)
    if use_year:
        y = pd.to_numeric(work[year_col], errors="coerce")
        y = y.where((y >= 1000) & (y <= 3000)).round().astype("Int64")
        work["_y"] = y
    else:
        work["_y"] = pd.NA

    # require non-empty keys
    if use_year:
        work = work[(work["_t"] != "") & (work["_a_fp"] != "") & work["_y"].notna()].copy()
    else:
        work = work[(work["_t"] != "") & (work["_a_fp"] != "")].copy()
    metrics["risky_titles_suppressed"] = int((work["_t"] == "").sum())
    metrics["work_rows_after_keys"] = int(len(work))
    if work.empty:
        metrics["time_s"] = time.perf_counter() - t0
        return (df, metrics) if return_metrics else df

    # ------------------------------------------------------
    # Stage A STRICT: exact match on (title_clean + authors_fp [+year] [+server scope])
    # ------------------------------------------------------
    if use_year:
        strict_base = work["_t"] + "||" + work["_a_fp"] + "||" + work["_y"].astype("string")
    else:
        strict_base = work["_t"] + "||" + work["_a_fp"]

    if across_servers:
        work["_grp_strict"] = strict_base
    else:
        work["_grp_strict"] = work[server_col].astype("string") + "||" + strict_base

    strict = work
    if prefilter:
        vcg = work["_grp_strict"].value_counts()
        dup_keys = vcg[vcg >= 2].index
        strict = work[work["_grp_strict"].isin(dup_keys)].copy()

    metrics["stageA_groups"] = int(strict["_grp_strict"].nunique()) if not strict.empty else 0

    # sort keys for parent choice
    if date_col and date_col in strict.columns:
        strict["_dt"] = pd.to_datetime(strict[date_col], errors="coerce")
    else:
        strict["_dt"] = pd.NaT
    strict["_rid"] = _record_id_key(strict[record_id_col])

    if not strict.empty:
        if choose_parent == "oldest":
            strict = strict.sort_values(
                by=["_grp_strict", "_dt", "_rid"],
                ascending=[True, True, True],
                na_position="last",
            )
        elif choose_parent == "most_recent":
            strict = strict.sort_values(
                by=["_grp_strict", "_dt", "_rid"],
                ascending=[True, False, False],
                na_position="last",
            )
        else:
            raise ValueError("choose_parent must be oldest | most_recent")

        parents = strict.groupby("_grp_strict", sort=False).head(1)
        parent_rid_map = parents.set_index("_grp_strict")[record_id_col]
        parent_srv_map = parents.set_index("_grp_strict")[server_col]

        strict["_parent_rid"] = strict["_grp_strict"].map(parent_rid_map)
        strict["_parent_srv"] = strict["_grp_strict"].map(parent_srv_map)

        is_parent = strict[record_id_col].eq(strict["_parent_rid"])
        parent_idx = strict.index[is_parent]
        child_idx = strict.index[~is_parent]

        metrics["stageA_children_labeled"] = int(len(child_idx))

        df.loc[parent_idx, hierarchy_col] = "parent"
        df.loc[parent_idx, parent_id_col] = pd.NA
        df.loc[child_idx, hierarchy_col] = (
            "parent - duplicate (" + strict.loc[child_idx, "_parent_srv"].astype("string") + ")"
        )
        df.loc[child_idx, parent_id_col] = strict.loc[child_idx, "_parent_rid"].values

        # deterministic group id
        df.loc[strict.index, group_id_col] = (
            pd.util.hash_pandas_object(strict["_grp_strict"], index=False)
            .astype("uint64")
            .astype(str)
            .values
        )

    # ------------------------------------------------------
    # Fuzzy title fallback (BLOCKED by authors_fp [+year], only remaining eligible)
    # Important: this can find near-duplicate titles because we do NOT rely on title_dup.
    # ------------------------------------------------------
    if title_fuzzy_fallback:
        # remaining eligible after Stage A
        h2 = df[hierarchy_col]
        if overwrite_mode == "parent_only":
            eligible2 = h2.astype("string").str.lower().str.strip().eq("parent")
        elif overwrite_mode == "unlabeled_only":
            eligible2 = h2.isna()
        else:
            eligible2 = pd.Series(True, index=df.index)

        remain_idx = work.index.intersection(df.index[eligible2])
        wF = work.loc[remain_idx].copy()

        if not wF.empty:
            # block by authors_fp (+year) because authors are "more trustworthy"
            if use_year:
                wF["_grp_auth"] = wF["_a_fp"] + "||" + wF["_y"].astype("string")
            else:
                wF["_grp_auth"] = wF["_a_fp"]

            # keep only blocks with >=2 rows
            vc_auth = wF["_grp_auth"].value_counts()
            keep_auth = vc_auth[vc_auth >= 2].index
            wF = wF[wF["_grp_auth"].isin(keep_auth)].copy()

            metrics["fuzzy_groups"] = int(wF["_grp_auth"].nunique()) if not wF.empty else 0

            if not wF.empty:
                # date/rid for parent selection
                if date_col and date_col in wF.columns:
                    wF["_dt"] = pd.to_datetime(wF[date_col], errors="coerce")
                else:
                    wF["_dt"] = pd.NaT
                wF["_rid"] = _record_id_key(wF[record_id_col])

                # tokens cache per row (within this stage)
                tokens_map = {idx: _title_tokens_from_clean(wF.loc[idx, "_t"]) for idx in wF.index}

                for grp, g in wF.groupby("_grp_auth", sort=False):
                    if len(g) < 2:
                        continue

                    # gate: ignore titles with too few tokens
                    idxs = [idx for idx in g.index if len(tokens_map.get(idx, [])) >= min_title_tokens]
                    if len(idxs) < 2:
                        continue

                    gg = g.loc[idxs].copy()
                    if choose_parent == "oldest":
                        gg = gg.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
                    else:
                        gg = gg.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

                    if fuzzy_compare_strategy == "parent_only":
                        parent_idx = gg.index[0]
                        parent_tokens = tokens_map[parent_idx]
                        parent_rid = gg.loc[parent_idx, record_id_col]
                        parent_srv = gg.loc[parent_idx, server_col]

                        # ensure parent labeled
                        df.loc[parent_idx, hierarchy_col] = "parent"
                        df.loc[parent_idx, parent_id_col] = pd.NA

                        for idx in gg.index[1:]:
                            metrics["fuzzy_pairs_checked"] += 1
                            sc = _containment_score(parent_tokens, tokens_map[idx])
                            if sc >= min_title_containment:
                                df.loc[idx, hierarchy_col] = f"parent - duplicate ({parent_srv})"
                                df.loc[idx, parent_id_col] = parent_rid
                                df.loc[idx, group_id_col] = f"fuzzy::{authors_fp_mode}::{grp}"
                                metrics["fuzzy_children_labeled"] += 1

                    elif fuzzy_compare_strategy == "all_pairs_small":
                        # safer than global all-pairs; still can be heavy if blocks are large.
                        # We'll cluster by greedy expansion (bounded within block).
                        idxs2 = gg.index.tolist()
                        used = set()
                        for i in idxs2:
                            if i in used:
                                continue
                            used.add(i)
                            cluster = [i]
                            for j in idxs2:
                                if j in used:
                                    continue
                                metrics["fuzzy_pairs_checked"] += 1
                                sc = _containment_score(tokens_map[i], tokens_map[j])
                                if sc >= min_title_containment:
                                    used.add(j)
                                    cluster.append(j)

                            if len(cluster) >= 2:
                                # choose parent (oldest/most recent) within cluster
                                cldf = gg.loc[cluster].copy()
                                if choose_parent == "oldest":
                                    cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
                                else:
                                    cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

                                p_idx = cldf.index[0]
                                p_rid = cldf.loc[p_idx, record_id_col]
                                p_srv = cldf.loc[p_idx, server_col]
                                df.loc[p_idx, hierarchy_col] = "parent"
                                df.loc[p_idx, parent_id_col] = pd.NA
                                for cidx in cldf.index[1:]:
                                    df.loc[cidx, hierarchy_col] = f"parent - duplicate ({p_srv})"
                                    df.loc[cidx, parent_id_col] = p_rid
                                    df.loc[cidx, group_id_col] = f"fuzzy::{authors_fp_mode}::{grp}"
                                    metrics["fuzzy_children_labeled"] += 1
                    else:
                        raise ValueError("fuzzy_compare_strategy must be parent_only | all_pairs_small")

    # ------------------------------------------------------
    # Stage B RELAXED (shared authors overlap) within exact title groups
    # ------------------------------------------------------
    if relaxed_shared_authors:
        h3 = df[hierarchy_col]
        if overwrite_mode == "parent_only":
            eligible3 = h3.astype("string").str.lower().str.strip().eq("parent")
        elif overwrite_mode == "unlabeled_only":
            eligible3 = h3.isna()
        else:
            eligible3 = pd.Series(True, index=df.index)

        remain_idx = work.index.intersection(df.index[eligible3])
        w2 = work.loc[remain_idx].copy()
        if not w2.empty:
            if use_year:
                relaxed_base = w2["_t"] + "||" + w2["_y"].astype("string")
            else:
                relaxed_base = w2["_t"]

            if across_servers:
                w2["_grp_title"] = relaxed_base
            else:
                w2["_grp_title"] = w2[server_col].astype("string") + "||" + relaxed_base

            # keep only repeated titles
            vc2 = w2["_grp_title"].value_counts()
            keep_groups = vc2[vc2 >= 2].index
            w2 = w2[w2["_grp_title"].isin(keep_groups)].copy()

            metrics["stageB_title_groups"] = int(w2["_grp_title"].nunique()) if not w2.empty else 0

            if not w2.empty:
                w2["_a_tokens"] = w2["_a_fp"].apply(_author_tokens_from_fp)
                w2["_a_n"] = w2["_a_tokens"].apply(len)

                if date_col and date_col in w2.columns:
                    w2["_dt"] = pd.to_datetime(w2[date_col], errors="coerce")
                else:
                    w2["_dt"] = pd.NaT
                w2["_rid"] = _record_id_key(w2[record_id_col])

                group_counter = 0
                children_total = 0

                for grp, g in w2.groupby("_grp_title", sort=False):
                    if len(g) < 2:
                        continue

                    g = g[g["_a_n"] >= min_authors_required].copy()
                    if len(g) < 2:
                        continue

                    idxs = g.index.tolist()
                    used = set()
                    clusters = []

                    # simple greedy clustering based on author overlap
                    for i in idxs:
                        if i in used:
                            continue
                        used.add(i)
                        cl = [i]
                        for j in idxs:
                            if j in used:
                                continue
                            if _overlap_count(g.loc[i, "_a_tokens"], g.loc[j, "_a_tokens"]) >= min_shared_authors:
                                used.add(j)
                                cl.append(j)
                        if len(cl) >= 2:
                            clusters.append(cl)

                    for cl in clusters:
                        group_counter += 1
                        cldf = g.loc[cl].copy()
                        if choose_parent == "oldest":
                            cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
                        else:
                            cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

                        parent_idx = cldf.index[0]
                        parent_rid = cldf.loc[parent_idx, record_id_col]
                        parent_srv = cldf.loc[parent_idx, server_col]

                        df.loc[parent_idx, hierarchy_col] = "parent"
                        df.loc[parent_idx, parent_id_col] = pd.NA
                        df.loc[parent_idx, group_id_col] = f"relaxed::{stage_name}::{group_counter}"

                        child_idxs = [x for x in cldf.index if x != parent_idx]
                        children_total += len(child_idxs)

                        df.loc[child_idxs, hierarchy_col] = f"parent - duplicate ({parent_srv})"
                        df.loc[child_idxs, parent_id_col] = parent_rid
                        df.loc[child_idxs, group_id_col] = f"relaxed::{stage_name}::{group_counter}"

                metrics["stageB_clusters"] = int(group_counter)
                metrics["stageB_children_labeled"] = int(children_total)

    metrics["time_s"] = time.perf_counter() - t0
    return (df, metrics) if return_metrics else df


# ============================================================
# 5) Stage runner with summary + early stop
# ============================================================
def _count_children_labels(series: pd.Series) -> int:
    s = series.astype("string").fillna("")
    return int(s.str.startswith("parent - duplicate").sum())


def run_dedupe_stages(
    df: pd.DataFrame,
    *,
    stages: List[Dict[str, Any]],
    early_stop_if_new_labels_lt: int = 100,
    print_summary: bool = True,
    return_all_metrics: bool = True,
    # common kwargs passed to every stage
    **common_kwargs,
) -> Tuple[pd.DataFrame, List[Dict[str, Any]]] | pd.DataFrame:
    """
    Runs a list of stages sequentially with:
      - delta duplicates added per stage
      - early stop
    """
    df_out = df
    metrics_all: List[Dict[str, Any]] = []

    prev_children = _count_children_labels(df_out[common_kwargs.get("hierarchy_col", "records_hierarchy")])

    for stage in stages:
        name = stage.get("name", stage.get("stage_name", "stage"))
        t0 = time.perf_counter()

        df_out, m = dedupe_title_authors_stage(
            df_out,
            return_metrics=True,
            stage_name=name,
            **common_kwargs,
            **{k: v for k, v in stage.items() if k not in {"name", "stage_name"}},
        )

        now_children = _count_children_labels(df_out[common_kwargs.get("hierarchy_col", "records_hierarchy")])
        delta = now_children - prev_children
        prev_children = now_children

        m["stage_runtime_s"] = time.perf_counter() - t0
        m["new_children_added"] = int(delta)
        metrics_all.append(m)

        if print_summary:
            print(
                f"[{name}] new_children={delta} | "
                f"cand={m['n_candidates_initial']} | "
                f"prefilter_rows={m['prefilter_rows']} | "
                f"A_children={m['stageA_children_labeled']} | "
                f"fuzzy_children={m['fuzzy_children_labeled']} | "
                f"B_children={m['stageB_children_labeled']} | "
                f"time={m['stage_runtime_s']:.2f}s"
            )

        if delta < early_stop_if_new_labels_lt:
            if print_summary:
                print(f"Early stop after {name}: delta {delta} < {early_stop_if_new_labels_lt}")
            break

    return (df_out, metrics_all) if return_all_metrics else df_out


# ============================================================
# 6) Two-pass pipeline: Exact pass -> Fuzzy pass on remaining parents
# ============================================================
def run_dedupe_pipeline_two_passes(
    df: pd.DataFrame,
    *,
    stages_exact: List[Dict[str, Any]],
    stages_fuzzy: List[Dict[str, Any]],
    early_stop_if_new_labels_lt: int = 100,
    print_summary: bool = True,
    return_all_metrics: bool = True,
    **common_kwargs,
) -> Tuple[pd.DataFrame, List[Dict[str, Any]]] | pd.DataFrame:
    """
    Pass A: run stages_exact (typically no fuzzy, prefilter_mode=title_dup).
    Pass B: run stages_fuzzy (fuzzy enabled, prefilter_mode=author_dup), on remaining parents only.

    IMPORTANT:
      - For Pass A, it is normal to use overwrite_mode="any" for stage1, then "parent_only" for stage2-3.
      - For Pass B, use overwrite_mode="parent_only" so we only touch unresolved parents.
    """
    all_metrics: List[Dict[str, Any]] = []
    df_out = df

    if print_summary:
        print("\n=== PASS A: EXACT ===")

    df_out, mA = run_dedupe_stages(
        df_out,
        stages=stages_exact,
        early_stop_if_new_labels_lt=early_stop_if_new_labels_lt,
        print_summary=print_summary,
        return_all_metrics=True,
        **common_kwargs,
    )
    all_metrics.extend(mA)

    if print_summary:
        print("\n=== PASS B: FUZZY (remaining parents) ===")

    df_out, mB = run_dedupe_stages(
        df_out,
        stages=stages_fuzzy,
        early_stop_if_new_labels_lt=early_stop_if_new_labels_lt,
        print_summary=print_summary,
        return_all_metrics=True,
        **common_kwargs,
    )
    all_metrics.extend(mB)

    return (df_out, all_metrics) if return_all_metrics else df_out


# ============================================================
# 7) Default stage configs (recommended)
# ============================================================

# PASS A (EXACT) — fast + high precision
STAGES_EXACT = [
    dict(
        name="A1_tokenbag_exact",
        authors_fp_mode="tokenbag",
        prefilter_mode="title_dup",
        title_fuzzy_fallback=False,
        relaxed_shared_authors=True,
        min_authors_required=1,
        min_shared_authors=1,
        overwrite_mode="parent_only",
        authors_fingerprint_col="authors_fp_tokenbag",
    ),
    dict(
        name="A2_last_initial_exact",
        authors_fp_mode="last_initial",
        prefilter_mode="title_dup",
        title_fuzzy_fallback=False,
        relaxed_shared_authors=True,
        min_authors_required=1,
        min_shared_authors=1,
        overwrite_mode="parent_only",
        authors_fingerprint_col="authors_fp_last_initial",
    ),
    dict(
        name="A3_last_exact_strict",
        authors_fp_mode="last",
        prefilter_mode="title_dup",
        title_fuzzy_fallback=False,
        relaxed_shared_authors=False,  # last-only already high recall; keep strict
        overwrite_mode="parent_only",
        authors_fingerprint_col="authors_fp_last",
    ),
]

# PASS B (FUZZY) — only remaining parents; block by authors_fp repetition
# Note: relaxed_shared_authors is usually OFF here to keep false positives down.
STAGES_FUZZY = [
    dict(
        name="B1_tokenbag_fuzzy",
        authors_fp_mode="tokenbag",
        prefilter_mode="author_dup",
        title_fuzzy_fallback=True,
        min_title_tokens=6,
        min_title_containment=0.80,  # start conservative; lower = more recall, more risk
        fuzzy_compare_strategy="parent_only",
        relaxed_shared_authors=False,
        # min_authors_required=1,
        # min_shared_authors=1,
        overwrite_mode="parent_only",
        authors_fingerprint_col="authors_fp_tokenbag",
    ),
    dict(
        name="B2_last_initial_fuzzy",
        authors_fp_mode="last_initial",
        prefilter_mode="author_dup",
        title_fuzzy_fallback=True,
        min_title_tokens=6,
        min_title_containment=0.90,
        fuzzy_compare_strategy="parent_only",
        relaxed_shared_authors=False,
        # min_authors_required=1,
        # min_shared_authors=1,
        overwrite_mode="parent_only",
        authors_fingerprint_col="authors_fp_last_initial",
    ),
]

# ============================================================
# 8) Example usage
# ============================================================
# df_out, metrics = run_dedupe_pipeline_two_passes(
#     df,
#     stages_exact=STAGES_EXACT,
#     stages_fuzzy=STAGES_FUZZY,
#     early_stop_if_new_labels_lt=500,
#     print_summary=True,
#     return_all_metrics=True,
#     servers=None,
#     across_servers=True,
#     use_year=False,
#     choose_parent="oldest",
#     prefilter=True,
#     date_candidates=('date_first_seen',),
#     hierarchy_col="records_hierarchy",
#     parent_id_col="parent_record_id",
#     group_id_col="dup_group_id",
#     add_authors_fingerprint_col=True,
#     add_title_clean_col=True,
#     title_clean_col="title_clean_v2",
# )
#
# print(metrics[-1])
# print(df_out["records_hierarchy"].value_counts(dropna=False).head(60))

In [39]:
data_out, metrics = run_dedupe_pipeline_two_passes(
    data_clean_hierarchy,
    stages_exact=STAGES_EXACT,
    stages_fuzzy=STAGES_FUZZY,
    early_stop_if_new_labels_lt=1,
    print_summary=True,
    return_all_metrics=True,
    servers=None,
    across_servers=True,
    use_year=False,
    choose_parent="oldest",
    prefilter=True,
    date_candidates=('date_first_seen',),
    hierarchy_col="records_hierarchy",
    parent_id_col="parent_record_id",
    group_id_col="dup_group_id",
    add_authors_fingerprint_col=True,
    add_title_clean_col=True,
    title_clean_col="title_clean_v2",
)

print(metrics[-1])
print(data_out["records_hierarchy"].value_counts(dropna=False).head(60))



=== PASS A: EXACT ===
[A1_tokenbag_exact] new_children=437758 | cand=7911901 | prefilter_rows=893355 | A_children=367853 | fuzzy_children=0 | B_children=69905 | time=434.90s
[A2_last_initial_exact] new_children=13155 | cand=7474143 | prefilter_rows=106603 | A_children=10681 | fuzzy_children=0 | B_children=2474 | time=40.12s
[A3_last_exact_strict] new_children=876 | cand=7460988 | prefilter_rows=80510 | A_children=876 | fuzzy_children=0 | B_children=0 | time=16.90s

=== PASS B: FUZZY (remaining parents) ===
[B1_tokenbag_fuzzy] new_children=100289 | cand=7460112 | prefilter_rows=3035578 | A_children=0 | fuzzy_children=100289 | B_children=0 | time=1804.92s
[B2_last_initial_fuzzy] new_children=9648 | cand=7359823 | prefilter_rows=3217294 | A_children=0 | fuzzy_children=9648 | B_children=0 | time=1615.86s
{'stage_name': 'B2_last_initial_fuzzy', 'n_rows_df': 7911901, 'n_candidates_initial': 7359823, 'prefilter_mode': 'author_dup', 'prefilter_rows': 3217294, 'prefilter_groups': 706542, 'work

In [41]:
# """
# Reproducible 2-pass dedupe pipeline (Exact pass -> Fuzzy pass) with:
# - Strong-but-cheap title normalization (cached)
# - 3 author signatures: tokenbag | last_initial | last
# - Stage A strict (title + authors_fp) exact
# - Stage B relaxed (shared authors overlap) within exact-title groups (optional per stage)
# - Optional fuzzy title fallback (token containment) BLOCKED by authors_fp (+ optional year)
# - Prefilter modes:
#     * title_dup  : keep rows where cleaned title repeats (fast exact stages)
#     * author_dup : keep rows where authors_fp repeats (enables fuzzy stages when titles differ)
#     * none       : keep all eligible (debug)

# Includes:
# - Metrics counters per stage
# - Summary printing + early stop
# - Deterministic labeling
# - Designed for speed + low false positives (especially with last_initial)

# USAGE:
# 1) Define STAGES_EXACT and STAGES_FUZZY
# 2) Run:
#    df_out, metrics = run_dedupe_pipeline_two_passes(
#        df,
#        stages_exact=STAGES_EXACT,
#        stages_fuzzy=STAGES_FUZZY,
#        early_stop_if_new_labels_lt=500,
#        print_summary=True,
#        return_all_metrics=True,
#        servers=None,
#        across_servers=True,
#        use_year=False,
#        choose_parent="oldest",
#        prefilter=True,
#        date_candidates=('date_first_seen',),
#        hierarchy_col="records_hierarchy",
#        parent_id_col="parent_record_id",
#        group_id_col="dup_group_id",
#        add_authors_fingerprint_col=True,
#        add_title_clean_col=True,
#    )
# """

# import pandas as pd
# import numpy as np
# import re
# import time
# import unicodedata
# from typing import Iterable, Optional, Dict, Any, Tuple, List

# # ============================================================
# # 0) Regex + NA helpers
# # ============================================================
# _WS = re.compile(r"\s+")
# _PUNCT_ALL = re.compile(r"[^\w\s]", re.UNICODE)  # remove everything except word chars + spaces
# NA_LIKE = {"", "none", "null", "nan", "n/a", "[]", "{}", "na"}

# # ============================================================
# # 0b) Generic / risky title filter
# # ============================================================
# # All values already lowercase; _clean_title_series_v2 lowercases before comparison.
# GENERIC_TITLES: frozenset[str] = frozenset({
#     "front matter", "back matter", "summary", "summaries",
#     "reviews in brief", "publications received",
#     "agricultural letter", "administrasi kurikulum",
#     "administrasi peserta didik", "experiment ended",
#     "editorial", "introduction", "preface", "contents",
#     "table of contents", "index", "book review", "letter",
#     "news", "announcement", "abstract", "poster", "supplement",
#     "no title", "résumés", "rãésumãés", "retracted",      # keep both accent forms
#     "bibliographie", "bremsstrahlung",
# })

# MIN_TITLE_TOKENS: int = 3      # centralised so every stage uses the same threshold
# MIN_TITLE_CHARS: int = 25


# def is_risky_title(title_clean: str) -> bool:
#     """
#     Returns True when a cleaned title should be EXCLUDED from deduplication.

#     Designed to operate on the OUTPUT of _clean_title_series_v2 (already
#     lowercase, accent-stripped, punctuation-removed), so GENERIC_TITLES
#     matching is automatically case-insensitive.
#     """
#     if not title_clean:
#         return True

#     # Exact match against the generic-title blocklist
#     if title_clean in GENERIC_TITLES:
#         return True

#     tokens = title_clean.split()

#     # Too few tokens
#     if len(tokens) < MIN_TITLE_TOKENS:
#         return True

#     # Too short in characters
#     if len(title_clean) < MIN_TITLE_CHARS:
#         return True

#     return False


# def _blank_risky_titles(title_clean_series: pd.Series) -> pd.Series:
#     """
#     Vectorised wrapper: returns a copy of the series with risky titles
#     replaced by "" so they are excluded downstream.
#     """
#     return title_clean_series.where(
#         ~title_clean_series.apply(is_risky_title),
#         other="",
#     )
    
# # ============================================================
# # 1) Utility: pick a date column + record_id numeric fallback
# # ============================================================
# def _pick_first_existing(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
#     for c in candidates:
#         if c in df.columns:
#             return c
#     return None


# def _record_id_key(s: pd.Series) -> pd.Series:
#     """Fast numeric key from record_id (extract first digits)."""
#     digits = s.astype("string").str.extract(r"(\d+)")[0]
#     return pd.to_numeric(digits, errors="coerce")


# # ============================================================
# # 2) Title normalization (cheap, high ROI) + token containment
# # ============================================================
# def _strip_accents_text(x: str) -> str:
#     return "".join(
#         c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c)
#     )


# def _clean_title_series_v2(s: pd.Series) -> pd.Series:
#     """
#     Strong-but-cheap title normalization:
#       - lowercase
#       - strip accents
#       - remove punctuation -> spaces
#       - collapse whitespace
#     """
#     s = s.astype("string").fillna("").str.strip().str.lower()
#     s = s.where(~s.isin(list(NA_LIKE)), "")
#     s = s.apply(_strip_accents_text)
#     s = s.str.replace(_PUNCT_ALL, " ", regex=True)
#     s = s.str.replace(_WS, " ", regex=True).str.strip()
#     return s


# def _title_tokens_from_clean(title_clean: str) -> List[str]:
#     """Tokenize already-clean title into tokens; drop very short tokens (len < 2)."""
#     if not title_clean:
#         return []
#     return [t for t in title_clean.split(" ") if len(t) >= 2]


# def _containment_score(a_tokens: List[str], b_tokens: List[str]) -> float:
#     """
#     Containment score:
#         |A ∩ B| / min(|A|, |B|)
#     Good for small title differences when tokens still mostly match.
#     """
#     if not a_tokens or not b_tokens:
#         return 0.0
#     A, B = set(a_tokens), set(b_tokens)
#     denom = min(len(A), len(B))
#     if denom <= 0:
#         return 0.0
#     return len(A & B) / denom


# # ============================================================
# # 3) Author canonicalization (3 modes)
# # ============================================================
# def _strip_accents(s: str) -> str:
#     return "".join(
#         c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
#     )


# def _normalize_one_author_tokenbag(author: str) -> str:
#     """
#     Token-bag per author:
#     - remove punctuation
#     - split tokens
#     - sort tokens within author
#     - join with "_"
#     """
#     if not author:
#         return ""
#     a = _strip_accents(str(author)).lower().strip()
#     if not a or a in NA_LIKE:
#         return ""
#     a = _PUNCT_ALL.sub(" ", a)
#     a = _WS.sub(" ", a).strip()
#     if not a:
#         return ""
#     toks = [t for t in a.split(" ") if t]
#     if not toks:
#         return ""
#     toks = sorted(toks)
#     return "_".join(toks)


# def _normalize_one_author_last_initial(author: str) -> str:
#     """
#     Middle-ground signature: "last|first_initial"
#     Rules:
#       - If comma: "Last, First ..." -> last = first token before comma;
#                                      initial = first token after comma (first-name token only)
#       - If no comma: "First ... Last" -> last = last token; initial = first token
#       - If we can't find an initial, return "" (reduces false positives)
#     """
#     if not author:
#         return ""
#     a = _strip_accents(str(author)).lower().strip()
#     if not a or a in NA_LIKE:
#         return ""

#     if "," in a:
#         left, right = a.split(",", 1)
#         left = _PUNCT_ALL.sub(" ", left)
#         right = _PUNCT_ALL.sub(" ", right)
#         left = _WS.sub(" ", left).strip()
#         right = _WS.sub(" ", right).strip()
#         if not left:
#             return ""
#         last_toks = [t for t in left.split(" ") if t]
#         if not last_toks:
#             return ""
#         last = last_toks[0]  # keep your "first token if multi-token surname" philosophy

#         first_toks = [t for t in right.split(" ") if t]
#         if not first_toks:
#             return ""  # avoid false positives
#         ini = first_toks[0][:1]
#         return f"{last}|{ini}" if ini else ""
#     else:
#         a = _PUNCT_ALL.sub(" ", a)
#         a = _WS.sub(" ", a).strip()
#         toks = [t for t in a.split(" ") if t]
#         if len(toks) < 2:
#             return ""
#         ini = toks[0][:1]
#         last = toks[-1]
#         return f"{last}|{ini}" if (ini and last) else ""


# def _normalize_one_author_last(author: str) -> str:
#     """Last-name-only signature (high recall, more false positives)."""
#     if not author:
#         return ""
#     a = _strip_accents(str(author)).lower().strip()
#     if not a or a in NA_LIKE:
#         return ""

#     if "," in a:
#         left = a.split(",", 1)[0].strip()
#         left = _PUNCT_ALL.sub(" ", left)
#         left = _WS.sub(" ", left).strip()
#         if not left:
#             return ""
#         toks = [t for t in left.split(" ") if t]
#         if not toks:
#             return ""
#         return toks[0]
#     else:
#         a = _PUNCT_ALL.sub(" ", a)
#         a = _WS.sub(" ", a).strip()
#         toks = [t for t in a.split(" ") if t]
#         if not toks:
#             return ""
#         return toks[-1]


# def build_authors_fingerprint_series(authors_flat: pd.Series, mode: str) -> pd.Series:
#     """
#     Build author fingerprint per row:
#       - split authors on ';'
#       - normalize each author (depends on mode)
#       - drop empties
#       - dedupe within row
#       - sort
#       - join with ';'
#     """
#     if mode not in {"tokenbag", "last_initial", "last"}:
#         raise ValueError("mode must be tokenbag | last_initial | last")

#     s = authors_flat.astype("string").fillna("").str.strip()
#     s = s.where(~s.str.lower().isin(list(NA_LIKE)), "")

#     if mode == "tokenbag":
#         norm_fn = _normalize_one_author_tokenbag
#     elif mode == "last_initial":
#         norm_fn = _normalize_one_author_last_initial
#     else:
#         norm_fn = _normalize_one_author_last

#     def row_to_fp(x: str) -> str:
#         if not x:
#             return ""
#         authors = [a.strip() for a in str(x).split(";") if a.strip()]
#         norm = [norm_fn(a) for a in authors]
#         norm = [z for z in norm if z]
#         norm = sorted(set(norm))
#         return ";".join(norm)

#     return s.apply(row_to_fp)


# def _author_tokens_from_fp(fp: str) -> List[str]:
#     if not fp:
#         return []
#     return [t for t in fp.split(";") if t]


# def _overlap_count(a_tokens: List[str], b_tokens: List[str]) -> int:
#     if not a_tokens or not b_tokens:
#         return 0
#     return len(set(a_tokens) & set(b_tokens))


# # ============================================================
# # 4) Single-stage dedupe:
# #    - Prefilter (title_dup/author_dup/none)
# #    - Stage A strict (title + authors_fp) exact match
# #    - Optional fuzzy title fallback (within same authors_fp)
# #    - Optional Stage B relaxed (shared authors overlap) within exact-title groups
# # ============================================================
# def dedupe_title_authors_stage(
#     df: pd.DataFrame,
#     *,
#     # stage config
#     stage_name: str = "stage",
#     authors_fp_mode: str = "tokenbag",         # tokenbag | last_initial | last

#     # fuzzy config (title containment), executed only if enabled
#     title_fuzzy_fallback: bool = False,
#     min_title_tokens: int = 6,
#     min_title_containment: float = 0.70,
#     fuzzy_compare_strategy: str = "parent_only",  # parent_only | all_pairs_small (parent_only is safest/fastest)

#     # relaxed (shared authors overlap) config (exact title only)
#     relaxed_shared_authors: bool = True,
#     min_authors_required: int = 2,
#     min_shared_authors: int = 2,

#     # prefilter strategy (important!)
#     prefilter_mode: str = "title_dup",         # title_dup | author_dup | none
#     prefilter: bool = True,                    # if False, skip ">=2" group filter (slower)

#     # global options
#     servers=None,
#     across_servers: bool = True,
#     use_year: bool = False,
#     choose_parent: str = "oldest",             # oldest | most_recent
#     overwrite_mode: str = "parent_only",       # any | parent_only | unlabeled_only

#     # columns
#     server_col: str = "server_name",
#     record_id_col: str = "record_id",
#     title_col: str = "title",
#     authors_col: str = "authors_flat",
#     year_col: str = "publication_year_first_seen",
#     date_candidates: Tuple[str, ...] = ("date_first_seen",),

#     hierarchy_col: str = "records_hierarchy",
#     parent_id_col: str = "parent_record_id",
#     group_id_col: str = "dup_group_id",

#     # caching/debug columns
#     add_authors_fingerprint_col: bool = True,
#     authors_fingerprint_col: str = "authors_fp",
#     add_title_clean_col: bool = True,
#     title_clean_col: str = "title_clean_v2",

#     return_metrics: bool = False,
# ) -> pd.DataFrame | Tuple[pd.DataFrame, Dict[str, Any]]:
#     """
#     One dedupe stage. Designed to be composed into a multi-stage pipeline.
#     """

#     t0 = time.perf_counter()

#     metrics: Dict[str, Any] = {
#         "stage_name": stage_name,
#         "n_rows_df": int(len(df)),
#         "n_candidates_initial": 0,
#         "prefilter_mode": prefilter_mode,
#         "prefilter_rows": 0,
#         "prefilter_groups": 0,
#         "work_rows_after_keys": 0,

#         "stageA_groups": 0,
#         "stageA_children_labeled": 0,

#         "fuzzy_enabled": bool(title_fuzzy_fallback),
#         "fuzzy_groups": 0,
#         "fuzzy_pairs_checked": 0,
#         "fuzzy_children_labeled": 0,

#         "stageB_enabled": bool(relaxed_shared_authors),
#         "stageB_title_groups": 0,
#         "stageB_clusters": 0,
#         "stageB_children_labeled": 0,

#         "time_s": 0.0,
#     }

#     # ------------------------------------------------------
#     # Ensure output cols exist
#     # ------------------------------------------------------
#     for c in (hierarchy_col, parent_id_col, group_id_col):
#         if c not in df.columns:
#             df[c] = pd.NA

#     if add_authors_fingerprint_col and authors_fingerprint_col not in df.columns:
#         df[authors_fingerprint_col] = pd.NA
#     if add_title_clean_col and title_clean_col not in df.columns:
#         df[title_clean_col] = pd.NA

#     # ------------------------------------------------------
#     # Eligibility
#     # ------------------------------------------------------
#     h = df[hierarchy_col]
#     if overwrite_mode == "any":
#         eligible = pd.Series(True, index=df.index)
#     elif overwrite_mode == "parent_only":
#         eligible = h.astype("string").str.lower().str.strip().eq("parent")
#     elif overwrite_mode == "unlabeled_only":
#         eligible = h.isna()
#     else:
#         raise ValueError("overwrite_mode must be any | parent_only | unlabeled_only")

#     # server filter
#     if servers is None:
#         server_mask = pd.Series(True, index=df.index)
#     elif isinstance(servers, str):
#         server_mask = df[server_col].eq(servers)
#     else:
#         server_mask = df[server_col].isin(list(servers))

#     m = eligible & server_mask
#     metrics["n_candidates_initial"] = int(m.sum())
#     if not m.any():
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     # ------------------------------------------------------
#     # Prefilter: decide which indices to consider in this stage
#     # ------------------------------------------------------
#     if prefilter_mode == "title_dup":
#         # title-based prefilter (fast for exact title stages)
#         t_clean = df.loc[m, title_clean_col] if (add_title_clean_col and title_clean_col in df.columns and df.loc[m, title_clean_col].notna().any()) else None
#         if t_clean is None:
#             t_clean = _clean_title_series_v2(df.loc[m, title_col])
#         vc = t_clean.value_counts()
#         keep_idx = t_clean[t_clean.isin(vc[vc >= 2].index)].index
#         metrics["prefilter_groups"] = int((vc >= 2).sum())

#     elif prefilter_mode == "author_dup":
#         # author-fp based prefilter (crucial for fuzzy pass; titles may differ)
#         # compute fp only for m rows
#         a_fp = build_authors_fingerprint_series(df.loc[m, authors_col], mode=authors_fp_mode)
#         vc = a_fp.value_counts()
#         keep_idx = a_fp[a_fp.isin(vc[vc >= 2].index)].index
#         metrics["prefilter_groups"] = int((vc >= 2).sum())

#     elif prefilter_mode == "none":
#         keep_idx = df.index[m]
#         metrics["prefilter_groups"] = 0

#     else:
#         raise ValueError("prefilter_mode must be title_dup | author_dup | none")

#     metrics["prefilter_rows"] = int(len(keep_idx))
#     if len(keep_idx) == 0:
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     # ------------------------------------------------------
#     # Work subset + compute/attach cached normalization keys
#     # ------------------------------------------------------
#     cols_needed = [server_col, record_id_col, title_col, authors_col]
#     if use_year:
#         cols_needed.append(year_col)
#     date_col = _pick_first_existing(df, date_candidates)
#     if date_col:
#         cols_needed.append(date_col)

#     work = df.loc[keep_idx, cols_needed].copy()

#     # Title clean (cache to df if asked)
#     if add_title_clean_col:
#         # compute for missing only (cheap)
#         t_missing = df.loc[work.index, title_clean_col].isna()
#         if t_missing.any():
#             df.loc[work.index[t_missing], title_clean_col] = _clean_title_series_v2(df.loc[work.index[t_missing], title_col]).values
#         work["_t"] = df.loc[work.index, title_clean_col].astype("string").fillna("")
#     else:
#         work["_t"] = _clean_title_series_v2(work[title_col])
        
#     # ── NEW: blank out generic / too-short titles before any key is built ──
#     work["_t"] = _blank_risky_titles(work["_t"])

#     # Authors fp (mode-specific; cache into df column if asked)
#     work["_a_fp"] = build_authors_fingerprint_series(work[authors_col], mode=authors_fp_mode)
#     if add_authors_fingerprint_col:
#         df.loc[work.index, authors_fingerprint_col] = work["_a_fp"].values

#     # Year (optional)
#     if use_year:
#         y = pd.to_numeric(work[year_col], errors="coerce")
#         y = y.where((y >= 1000) & (y <= 3000)).round().astype("Int64")
#         work["_y"] = y
#     else:
#         work["_y"] = pd.NA

#     # require non-empty keys
#     if use_year:
#         work = work[(work["_t"] != "") & (work["_a_fp"] != "") & work["_y"].notna()].copy()
#     else:
#         work = work[(work["_t"] != "") & (work["_a_fp"] != "")].copy()

#     metrics["work_rows_after_keys"] = int(len(work))
#     if work.empty:
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     # ------------------------------------------------------
#     # Stage A STRICT: exact match on (title_clean + authors_fp [+year] [+server scope])
#     # ------------------------------------------------------
#     if use_year:
#         strict_base = work["_t"] + "||" + work["_a_fp"] + "||" + work["_y"].astype("string")
#     else:
#         strict_base = work["_t"] + "||" + work["_a_fp"]

#     if across_servers:
#         work["_grp_strict"] = strict_base
#     else:
#         work["_grp_strict"] = work[server_col].astype("string") + "||" + strict_base

#     strict = work
#     if prefilter:
#         vcg = work["_grp_strict"].value_counts()
#         dup_keys = vcg[vcg >= 2].index
#         strict = work[work["_grp_strict"].isin(dup_keys)].copy()

#     metrics["stageA_groups"] = int(strict["_grp_strict"].nunique()) if not strict.empty else 0

#     # sort keys for parent choice
#     if date_col and date_col in strict.columns:
#         strict["_dt"] = pd.to_datetime(strict[date_col], errors="coerce")
#     else:
#         strict["_dt"] = pd.NaT
#     strict["_rid"] = _record_id_key(strict[record_id_col])

#     if not strict.empty:
#         if choose_parent == "oldest":
#             strict = strict.sort_values(
#                 by=["_grp_strict", "_dt", "_rid"],
#                 ascending=[True, True, True],
#                 na_position="last",
#             )
#         elif choose_parent == "most_recent":
#             strict = strict.sort_values(
#                 by=["_grp_strict", "_dt", "_rid"],
#                 ascending=[True, False, False],
#                 na_position="last",
#             )
#         else:
#             raise ValueError("choose_parent must be oldest | most_recent")

#         parents = strict.groupby("_grp_strict", sort=False).head(1)
#         parent_rid_map = parents.set_index("_grp_strict")[record_id_col]
#         parent_srv_map = parents.set_index("_grp_strict")[server_col]

#         strict["_parent_rid"] = strict["_grp_strict"].map(parent_rid_map)
#         strict["_parent_srv"] = strict["_grp_strict"].map(parent_srv_map)

#         is_parent = strict[record_id_col].eq(strict["_parent_rid"])
#         parent_idx = strict.index[is_parent]
#         child_idx = strict.index[~is_parent]

#         metrics["stageA_children_labeled"] = int(len(child_idx))

#         df.loc[parent_idx, hierarchy_col] = "parent"
#         df.loc[parent_idx, parent_id_col] = pd.NA
#         df.loc[child_idx, hierarchy_col] = (
#             "parent - duplicate (" + strict.loc[child_idx, "_parent_srv"].astype("string") + ")"
#         )
#         df.loc[child_idx, parent_id_col] = strict.loc[child_idx, "_parent_rid"].values

#         # deterministic group id
#         df.loc[strict.index, group_id_col] = (
#             pd.util.hash_pandas_object(strict["_grp_strict"], index=False)
#             .astype("uint64")
#             .astype(str)
#             .values
#         )

#     # ------------------------------------------------------
#     # Fuzzy title fallback (BLOCKED by authors_fp [+year], only remaining eligible)
#     # Important: this can find near-duplicate titles because we do NOT rely on title_dup.
#     # ------------------------------------------------------
#     if title_fuzzy_fallback:
#         # remaining eligible after Stage A
#         h2 = df[hierarchy_col]
#         if overwrite_mode == "parent_only":
#             eligible2 = h2.astype("string").str.lower().str.strip().eq("parent")
#         elif overwrite_mode == "unlabeled_only":
#             eligible2 = h2.isna()
#         else:
#             eligible2 = pd.Series(True, index=df.index)

#         remain_idx = work.index.intersection(df.index[eligible2])
#         wF = work.loc[remain_idx].copy()

#         if not wF.empty:
#             # block by authors_fp (+year) because authors are "more trustworthy"
#             if use_year:
#                 wF["_grp_auth"] = wF["_a_fp"] + "||" + wF["_y"].astype("string")
#             else:
#                 wF["_grp_auth"] = wF["_a_fp"]

#             # keep only blocks with >=2 rows
#             vc_auth = wF["_grp_auth"].value_counts()
#             keep_auth = vc_auth[vc_auth >= 2].index
#             wF = wF[wF["_grp_auth"].isin(keep_auth)].copy()

#             metrics["fuzzy_groups"] = int(wF["_grp_auth"].nunique()) if not wF.empty else 0

#             if not wF.empty:
#                 # date/rid for parent selection
#                 if date_col and date_col in wF.columns:
#                     wF["_dt"] = pd.to_datetime(wF[date_col], errors="coerce")
#                 else:
#                     wF["_dt"] = pd.NaT
#                 wF["_rid"] = _record_id_key(wF[record_id_col])

#                 # tokens cache per row (within this stage)
#                 tokens_map = {idx: _title_tokens_from_clean(wF.loc[idx, "_t"]) for idx in wF.index}

#                 for grp, g in wF.groupby("_grp_auth", sort=False):
#                     if len(g) < 2:
#                         continue

#                     # gate: ignore titles with too few tokens
#                     idxs = [idx for idx in g.index if len(tokens_map.get(idx, [])) >= min_title_tokens]
#                     if len(idxs) < 2:
#                         continue

#                     gg = g.loc[idxs].copy()
#                     if choose_parent == "oldest":
#                         gg = gg.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
#                     else:
#                         gg = gg.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

#                     if fuzzy_compare_strategy == "parent_only":
#                         parent_idx = gg.index[0]
#                         parent_tokens = tokens_map[parent_idx]
#                         parent_rid = gg.loc[parent_idx, record_id_col]
#                         parent_srv = gg.loc[parent_idx, server_col]

#                         # ensure parent labeled
#                         df.loc[parent_idx, hierarchy_col] = "parent"
#                         df.loc[parent_idx, parent_id_col] = pd.NA

#                         for idx in gg.index[1:]:
#                             metrics["fuzzy_pairs_checked"] += 1
#                             sc = _containment_score(parent_tokens, tokens_map[idx])
#                             if sc >= min_title_containment:
#                                 df.loc[idx, hierarchy_col] = f"parent - duplicate ({parent_srv})"
#                                 df.loc[idx, parent_id_col] = parent_rid
#                                 df.loc[idx, group_id_col] = f"fuzzy::{authors_fp_mode}::{grp}"
#                                 metrics["fuzzy_children_labeled"] += 1

#                     elif fuzzy_compare_strategy == "all_pairs_small":
#                         # safer than global all-pairs; still can be heavy if blocks are large.
#                         # We'll cluster by greedy expansion (bounded within block).
#                         idxs2 = gg.index.tolist()
#                         used = set()
#                         for i in idxs2:
#                             if i in used:
#                                 continue
#                             used.add(i)
#                             cluster = [i]
#                             for j in idxs2:
#                                 if j in used:
#                                     continue
#                                 metrics["fuzzy_pairs_checked"] += 1
#                                 sc = _containment_score(tokens_map[i], tokens_map[j])
#                                 if sc >= min_title_containment:
#                                     used.add(j)
#                                     cluster.append(j)

#                             if len(cluster) >= 2:
#                                 # choose parent (oldest/most recent) within cluster
#                                 cldf = gg.loc[cluster].copy()
#                                 if choose_parent == "oldest":
#                                     cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
#                                 else:
#                                     cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

#                                 p_idx = cldf.index[0]
#                                 p_rid = cldf.loc[p_idx, record_id_col]
#                                 p_srv = cldf.loc[p_idx, server_col]
#                                 df.loc[p_idx, hierarchy_col] = "parent"
#                                 df.loc[p_idx, parent_id_col] = pd.NA
#                                 for cidx in cldf.index[1:]:
#                                     df.loc[cidx, hierarchy_col] = f"parent - duplicate ({p_srv})"
#                                     df.loc[cidx, parent_id_col] = p_rid
#                                     df.loc[cidx, group_id_col] = f"fuzzy::{authors_fp_mode}::{grp}"
#                                     metrics["fuzzy_children_labeled"] += 1
#                     else:
#                         raise ValueError("fuzzy_compare_strategy must be parent_only | all_pairs_small")

#     # ------------------------------------------------------
#     # Stage B RELAXED (shared authors overlap) within exact title groups
#     # ------------------------------------------------------
#     if relaxed_shared_authors:
#         h3 = df[hierarchy_col]
#         if overwrite_mode == "parent_only":
#             eligible3 = h3.astype("string").str.lower().str.strip().eq("parent")
#         elif overwrite_mode == "unlabeled_only":
#             eligible3 = h3.isna()
#         else:
#             eligible3 = pd.Series(True, index=df.index)

#         remain_idx = work.index.intersection(df.index[eligible3])
#         w2 = work.loc[remain_idx].copy()
#         if not w2.empty:
#             if use_year:
#                 relaxed_base = w2["_t"] + "||" + w2["_y"].astype("string")
#             else:
#                 relaxed_base = w2["_t"]

#             if across_servers:
#                 w2["_grp_title"] = relaxed_base
#             else:
#                 w2["_grp_title"] = w2[server_col].astype("string") + "||" + relaxed_base

#             # keep only repeated titles
#             vc2 = w2["_grp_title"].value_counts()
#             keep_groups = vc2[vc2 >= 2].index
#             w2 = w2[w2["_grp_title"].isin(keep_groups)].copy()

#             metrics["stageB_title_groups"] = int(w2["_grp_title"].nunique()) if not w2.empty else 0

#             if not w2.empty:
#                 w2["_a_tokens"] = w2["_a_fp"].apply(_author_tokens_from_fp)
#                 w2["_a_n"] = w2["_a_tokens"].apply(len)

#                 if date_col and date_col in w2.columns:
#                     w2["_dt"] = pd.to_datetime(w2[date_col], errors="coerce")
#                 else:
#                     w2["_dt"] = pd.NaT
#                 w2["_rid"] = _record_id_key(w2[record_id_col])

#                 group_counter = 0
#                 children_total = 0

#                 for grp, g in w2.groupby("_grp_title", sort=False):
#                     if len(g) < 2:
#                         continue

#                     g = g[g["_a_n"] >= min_authors_required].copy()
#                     if len(g) < 2:
#                         continue

#                     idxs = g.index.tolist()
#                     used = set()
#                     clusters = []

#                     # simple greedy clustering based on author overlap
#                     for i in idxs:
#                         if i in used:
#                             continue
#                         used.add(i)
#                         cl = [i]
#                         for j in idxs:
#                             if j in used:
#                                 continue
#                             if _overlap_count(g.loc[i, "_a_tokens"], g.loc[j, "_a_tokens"]) >= min_shared_authors:
#                                 used.add(j)
#                                 cl.append(j)
#                         if len(cl) >= 2:
#                             clusters.append(cl)

#                     for cl in clusters:
#                         group_counter += 1
#                         cldf = g.loc[cl].copy()
#                         if choose_parent == "oldest":
#                             cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
#                         else:
#                             cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

#                         parent_idx = cldf.index[0]
#                         parent_rid = cldf.loc[parent_idx, record_id_col]
#                         parent_srv = cldf.loc[parent_idx, server_col]

#                         df.loc[parent_idx, hierarchy_col] = "parent"
#                         df.loc[parent_idx, parent_id_col] = pd.NA
#                         df.loc[parent_idx, group_id_col] = f"relaxed::{stage_name}::{group_counter}"

#                         child_idxs = [x for x in cldf.index if x != parent_idx]
#                         children_total += len(child_idxs)

#                         df.loc[child_idxs, hierarchy_col] = f"parent - duplicate ({parent_srv})"
#                         df.loc[child_idxs, parent_id_col] = parent_rid
#                         df.loc[child_idxs, group_id_col] = f"relaxed::{stage_name}::{group_counter}"

#                 metrics["stageB_clusters"] = int(group_counter)
#                 metrics["stageB_children_labeled"] = int(children_total)

#     metrics["time_s"] = time.perf_counter() - t0
#     return (df, metrics) if return_metrics else df


# # ============================================================
# # 5) Stage runner with summary + early stop
# # ============================================================
# def _count_children_labels(series: pd.Series) -> int:
#     s = series.astype("string").fillna("")
#     return int(s.str.startswith("parent - duplicate").sum())


# def run_dedupe_stages(
#     df: pd.DataFrame,
#     *,
#     stages: List[Dict[str, Any]],
#     early_stop_if_new_labels_lt: int = 100,
#     print_summary: bool = True,
#     return_all_metrics: bool = True,
#     # common kwargs passed to every stage
#     **common_kwargs,
# ) -> Tuple[pd.DataFrame, List[Dict[str, Any]]] | pd.DataFrame:
#     """
#     Runs a list of stages sequentially with:
#       - delta duplicates added per stage
#       - early stop
#     """
#     df_out = df
#     metrics_all: List[Dict[str, Any]] = []

#     prev_children = _count_children_labels(df_out[common_kwargs.get("hierarchy_col", "records_hierarchy")])

#     for stage in stages:
#         name = stage.get("name", stage.get("stage_name", "stage"))
#         t0 = time.perf_counter()

#         df_out, m = dedupe_title_authors_stage(
#             df_out,
#             return_metrics=True,
#             stage_name=name,
#             **common_kwargs,
#             **{k: v for k, v in stage.items() if k not in {"name", "stage_name"}},
#         )

#         now_children = _count_children_labels(df_out[common_kwargs.get("hierarchy_col", "records_hierarchy")])
#         delta = now_children - prev_children
#         prev_children = now_children

#         m["stage_runtime_s"] = time.perf_counter() - t0
#         m["new_children_added"] = int(delta)
#         metrics_all.append(m)

#         if print_summary:
#             print(
#                 f"[{name}] new_children={delta} | "
#                 f"cand={m['n_candidates_initial']} | "
#                 f"prefilter_rows={m['prefilter_rows']} | "
#                 f"A_children={m['stageA_children_labeled']} | "
#                 f"fuzzy_children={m['fuzzy_children_labeled']} | "
#                 f"B_children={m['stageB_children_labeled']} | "
#                 f"time={m['stage_runtime_s']:.2f}s"
#             )

#         if delta < early_stop_if_new_labels_lt:
#             if print_summary:
#                 print(f"Early stop after {name}: delta {delta} < {early_stop_if_new_labels_lt}")
#             break

#     return (df_out, metrics_all) if return_all_metrics else df_out


# # ============================================================
# # 6) Two-pass pipeline: Exact pass -> Fuzzy pass on remaining parents
# # ============================================================
# def run_dedupe_pipeline_two_passes(
#     df: pd.DataFrame,
#     *,
#     stages_exact: List[Dict[str, Any]],
#     stages_fuzzy: List[Dict[str, Any]],
#     early_stop_if_new_labels_lt: int = 100,
#     print_summary: bool = True,
#     return_all_metrics: bool = True,
#     **common_kwargs,
# ) -> Tuple[pd.DataFrame, List[Dict[str, Any]]] | pd.DataFrame:
#     """
#     Pass A: run stages_exact (typically no fuzzy, prefilter_mode=title_dup).
#     Pass B: run stages_fuzzy (fuzzy enabled, prefilter_mode=author_dup), on remaining parents only.

#     IMPORTANT:
#       - For Pass A, it is normal to use overwrite_mode="any" for stage1, then "parent_only" for stage2-3.
#       - For Pass B, use overwrite_mode="parent_only" so we only touch unresolved parents.
#     """
#     all_metrics: List[Dict[str, Any]] = []
#     df_out = df

#     if print_summary:
#         print("\n=== PASS A: EXACT ===")

#     df_out, mA = run_dedupe_stages(
#         df_out,
#         stages=stages_exact,
#         early_stop_if_new_labels_lt=early_stop_if_new_labels_lt,
#         print_summary=print_summary,
#         return_all_metrics=True,
#         **common_kwargs,
#     )
#     all_metrics.extend(mA)

#     if print_summary:
#         print("\n=== PASS B: FUZZY (remaining parents) ===")

#     df_out, mB = run_dedupe_stages(
#         df_out,
#         stages=stages_fuzzy,
#         early_stop_if_new_labels_lt=early_stop_if_new_labels_lt,
#         print_summary=print_summary,
#         return_all_metrics=True,
#         **common_kwargs,
#     )
#     all_metrics.extend(mB)

#     return (df_out, all_metrics) if return_all_metrics else df_out


# # ============================================================
# # 7) Default stage configs (recommended)
# # ============================================================

# # PASS A (EXACT) — fast + high precision
# STAGES_EXACT = [
#     dict(
#         name="A1_tokenbag_exact",
#         authors_fp_mode="tokenbag",
#         prefilter_mode="title_dup",
#         min_title_tokens=3,
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=True,
#         min_authors_required=1,
#         min_shared_authors=1,
#         overwrite_mode="parent_only",
#         authors_fingerprint_col="authors_fp_tokenbag",
#     ),
#     dict(
#         name="A2_last_initial_exact",
#         authors_fp_mode="last_initial",
#         prefilter_mode="title_dup",
#         min_title_tokens=3,
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=True,
#         min_authors_required=2,
#         min_shared_authors=2,
#         overwrite_mode="parent_only",
#         authors_fingerprint_col="authors_fp_last_initial",
#     ),
#     dict(
#         name="A3_last_exact_strict",
#         authors_fp_mode="last",
#         prefilter_mode="title_dup",
#         min_title_tokens=3,
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=False,  # last-only already high recall; keep strict
#         overwrite_mode="parent_only",
#         authors_fingerprint_col="authors_fp_last",
#     ),
# ]

# # PASS B (FUZZY) — only remaining parents; block by authors_fp repetition
# # Note: relaxed_shared_authors is usually OFF here to keep false positives down.
# STAGES_FUZZY = [
#     dict(
#         name="B1_tokenbag_fuzzy",
#         authors_fp_mode="tokenbag",
#         prefilter_mode="author_dup",
#         title_fuzzy_fallback=True,
#         min_title_tokens=8,
#         min_title_containment=0.75,  # start conservative; lower = more recall, more risk
#         fuzzy_compare_strategy="parent_only",
#         relaxed_shared_authors=False,
#         # min_authors_required=1,
#         # min_shared_authors=1,
#         overwrite_mode="parent_only",
#         authors_fingerprint_col="authors_fp_tokenbag",
#     ),
#     dict(
#         name="B2_last_initial_fuzzy",
#         authors_fp_mode="last_initial",
#         prefilter_mode="author_dup",
#         title_fuzzy_fallback=True,
#         min_title_tokens=8,
#         min_title_containment=0.90,
#         fuzzy_compare_strategy="parent_only",
#         relaxed_shared_authors=False,
#         # min_authors_required=1,
#         # min_shared_authors=1,
#         overwrite_mode="parent_only",
#         authors_fingerprint_col="authors_fp_last_initial",
#     ),
# ]

# # ============================================================
# # 8) Example usage
# # ============================================================
# # df_out, metrics = run_dedupe_pipeline_two_passes(
# #     df,
# #     stages_exact=STAGES_EXACT,
# #     stages_fuzzy=STAGES_FUZZY,
# #     early_stop_if_new_labels_lt=500,
# #     print_summary=True,
# #     return_all_metrics=True,
# #     servers=None,
# #     across_servers=True,
# #     use_year=False,
# #     choose_parent="oldest",
# #     prefilter=True,
# #     date_candidates=('date_first_seen',),
# #     hierarchy_col="records_hierarchy",
# #     parent_id_col="parent_record_id",
# #     group_id_col="dup_group_id",
# #     add_authors_fingerprint_col=True,
# #     add_title_clean_col=True,
# #     title_clean_col="title_clean_v2",
# # )
# #
# # print(metrics[-1])
# # print(df_out["records_hierarchy"].value_counts(dropna=False).head(60))


In [42]:
# """
# Reproducible 2-pass dedupe pipeline (Exact pass -> Fuzzy pass) with:
# - Strong-but-cheap title normalization (cached)
# - 3 author signatures: tokenbag | last_initial | last
# - Stage A strict (title + authors_fp) exact
# - Stage B relaxed (shared authors overlap) within exact-title groups (optional per stage)
# - Optional fuzzy title fallback (token containment) BLOCKED by authors_fp (+ optional year)
# - Prefilter modes:
#     * title_dup  : keep rows where cleaned title repeats (fast exact stages)
#     * author_dup : keep rows where authors_fp repeats (enables fuzzy stages when titles differ)
#     * none       : keep all eligible (debug)

# Includes:
# - Metrics counters per stage
# - Summary printing + early stop
# - Deterministic labeling
# - Designed for speed + low false positives (especially with last_initial)

# USAGE:
# 1) Define STAGES_EXACT and STAGES_FUZZY
# 2) Run:
#    df_out, metrics = run_dedupe_pipeline_two_passes(
#        df,
#        stages_exact=STAGES_EXACT,
#        stages_fuzzy=STAGES_FUZZY,
#        early_stop_if_new_labels_lt=500,
#        print_summary=True,
#        return_all_metrics=True,
#        servers=None,
#        across_servers=True,
#        use_year=False,
#        choose_parent="oldest",
#        prefilter=True,
#        date_candidates=('date_first_seen',),
#        hierarchy_col="records_hierarchy",
#        parent_id_col="parent_record_id",
#        group_id_col="dup_group_id",
#        add_authors_fingerprint_col=True,
#        add_title_clean_col=True,
#    )
# """

# import pandas as pd
# import numpy as np
# import re
# import time
# import unicodedata
# from typing import Iterable, Optional, Dict, Any, Tuple, List

# # ============================================================
# # 0) Regex + NA helpers
# # ============================================================
# _WS = re.compile(r"\s+")
# _PUNCT_ALL = re.compile(r"[^\w\s]", re.UNICODE)  # remove everything except word chars + spaces
# NA_LIKE = {"", "none", "null", "nan", "n/a", "[]", "{}", "na"}

# # ============================================================
# # 0b) Generic / risky title filter
# # ============================================================
# # All values already lowercase; _clean_title_series_v2 lowercases before comparison.
# GENERIC_TITLES: frozenset[str] = frozenset({
#     "front matter", "back matter", "summary", "summaries",
#     "reviews in brief", "publications received",
#     "agricultural letter", "administrasi kurikulum",
#     "administrasi peserta didik", "experiment ended",
#     "editorial", "introduction", "preface", "contents",
#     "table of contents", "index", "book review", "letter",
#     "news", "announcement", "abstract", "poster", "supplement",
#     "no title", "résumés", "rãésumãés", "retracted",      # keep both accent forms
#     "bibliographie", "bremsstrahlung",
# })

# MIN_TITLE_TOKENS: int = 3      # centralised so every stage uses the same threshold
# MIN_TITLE_CHARS: int = 25


# def is_risky_title(title_clean: str) -> bool:
#     """
#     Returns True when a cleaned title should be EXCLUDED from deduplication.

#     Designed to operate on the OUTPUT of _clean_title_series_v2 (already
#     lowercase, accent-stripped, punctuation-removed), so GENERIC_TITLES
#     matching is automatically case-insensitive.
#     """
#     if not title_clean:
#         return True

#     # Exact match against the generic-title blocklist
#     if title_clean in GENERIC_TITLES:
#         return True

#     tokens = title_clean.split()

#     # Too few tokens
#     if len(tokens) < MIN_TITLE_TOKENS:
#         return True

#     # Too short in characters
#     if len(title_clean) < MIN_TITLE_CHARS:
#         return True

#     return False


# def _blank_risky_titles(title_clean_series: pd.Series) -> pd.Series:
#     """
#     Vectorised wrapper: returns a copy of the series with risky titles
#     replaced by "" so they are excluded downstream.
#     """
#     return title_clean_series.where(
#         ~title_clean_series.apply(is_risky_title),
#         other="",
#     )
    
# # ============================================================
# # 1) Utility: pick a date column + record_id numeric fallback
# # ============================================================
# def _pick_first_existing(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
#     for c in candidates:
#         if c in df.columns:
#             return c
#     return None


# def _record_id_key(s: pd.Series) -> pd.Series:
#     """Fast numeric key from record_id (extract first digits)."""
#     digits = s.astype("string").str.extract(r"(\d+)")[0]
#     return pd.to_numeric(digits, errors="coerce")


# # ============================================================
# # 2) Title normalization (cheap, high ROI) + token containment
# # ============================================================
# def _strip_accents_text(x: str) -> str:
#     return "".join(
#         c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c)
#     )


# def _clean_title_series_v2(s: pd.Series) -> pd.Series:
#     """
#     Strong-but-cheap title normalization:
#       - lowercase
#       - strip accents
#       - remove punctuation -> spaces
#       - collapse whitespace
#     """
#     s = s.astype("string").fillna("").str.strip().str.lower()
#     s = s.where(~s.isin(list(NA_LIKE)), "")
#     s = s.apply(_strip_accents_text)
#     s = s.str.replace(_PUNCT_ALL, " ", regex=True)
#     s = s.str.replace(_WS, " ", regex=True).str.strip()
#     return s


# def _title_tokens_from_clean(title_clean: str) -> List[str]:
#     """Tokenize already-clean title into tokens; drop very short tokens (len < 2)."""
#     if not title_clean:
#         return []
#     return [t for t in title_clean.split(" ") if len(t) >= 2]


# def _containment_score(a_tokens: List[str], b_tokens: List[str]) -> float:
#     """
#     Containment score:
#         |A ∩ B| / min(|A|, |B|)
#     Good for small title differences when tokens still mostly match.
#     """
#     if not a_tokens or not b_tokens:
#         return 0.0
#     A, B = set(a_tokens), set(b_tokens)
#     denom = min(len(A), len(B))
#     if denom <= 0:
#         return 0.0
#     return len(A & B) / denom


# # ============================================================
# # 3) Author canonicalization (3 modes)
# # ============================================================
# def _strip_accents(s: str) -> str:
#     return "".join(
#         c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
#     )


# def _normalize_one_author_tokenbag(author: str) -> str:
#     """
#     Token-bag per author:
#     - remove punctuation
#     - split tokens
#     - sort tokens within author
#     - join with "_"
#     """
#     if not author:
#         return ""
#     a = _strip_accents(str(author)).lower().strip()
#     if not a or a in NA_LIKE:
#         return ""
#     a = _PUNCT_ALL.sub(" ", a)
#     a = _WS.sub(" ", a).strip()
#     if not a:
#         return ""
#     toks = [t for t in a.split(" ") if t]
#     if not toks:
#         return ""
#     toks = sorted(toks)
#     return "_".join(toks)


# def _normalize_one_author_last_initial(author: str) -> str:
#     """
#     Middle-ground signature: "last|first_initial"
#     Rules:
#       - If comma: "Last, First ..." -> last = first token before comma;
#                                      initial = first token after comma (first-name token only)
#       - If no comma: "First ... Last" -> last = last token; initial = first token
#       - If we can't find an initial, return "" (reduces false positives)
#     """
#     if not author:
#         return ""
#     a = _strip_accents(str(author)).lower().strip()
#     if not a or a in NA_LIKE:
#         return ""

#     if "," in a:
#         left, right = a.split(",", 1)
#         left = _PUNCT_ALL.sub(" ", left)
#         right = _PUNCT_ALL.sub(" ", right)
#         left = _WS.sub(" ", left).strip()
#         right = _WS.sub(" ", right).strip()
#         if not left:
#             return ""
#         last_toks = [t for t in left.split(" ") if t]
#         if not last_toks:
#             return ""
#         last = last_toks[0]  # keep your "first token if multi-token surname" philosophy

#         first_toks = [t for t in right.split(" ") if t]
#         if not first_toks:
#             return ""  # avoid false positives
#         ini = first_toks[0][:1]
#         return f"{last}|{ini}" if ini else ""
#     else:
#         a = _PUNCT_ALL.sub(" ", a)
#         a = _WS.sub(" ", a).strip()
#         toks = [t for t in a.split(" ") if t]
#         if len(toks) < 2:
#             return ""
#         ini = toks[0][:1]
#         last = toks[-1]
#         return f"{last}|{ini}" if (ini and last) else ""


# def _normalize_one_author_last(author: str) -> str:
#     """Last-name-only signature (high recall, more false positives)."""
#     if not author:
#         return ""
#     a = _strip_accents(str(author)).lower().strip()
#     if not a or a in NA_LIKE:
#         return ""

#     if "," in a:
#         left = a.split(",", 1)[0].strip()
#         left = _PUNCT_ALL.sub(" ", left)
#         left = _WS.sub(" ", left).strip()
#         if not left:
#             return ""
#         toks = [t for t in left.split(" ") if t]
#         if not toks:
#             return ""
#         return toks[0]
#     else:
#         a = _PUNCT_ALL.sub(" ", a)
#         a = _WS.sub(" ", a).strip()
#         toks = [t for t in a.split(" ") if t]
#         if not toks:
#             return ""
#         return toks[-1]


# def build_authors_fingerprint_series(authors_flat: pd.Series, mode: str) -> pd.Series:
#     """
#     Build author fingerprint per row:
#       - split authors on ';'
#       - normalize each author (depends on mode)
#       - drop empties
#       - dedupe within row
#       - sort
#       - join with ';'
#     """
#     if mode not in {"tokenbag", "last_initial", "last"}:
#         raise ValueError("mode must be tokenbag | last_initial | last")

#     s = authors_flat.astype("string").fillna("").str.strip()
#     s = s.where(~s.str.lower().isin(list(NA_LIKE)), "")

#     if mode == "tokenbag":
#         norm_fn = _normalize_one_author_tokenbag
#     elif mode == "last_initial":
#         norm_fn = _normalize_one_author_last_initial
#     else:
#         norm_fn = _normalize_one_author_last

#     def row_to_fp(x: str) -> str:
#         if not x:
#             return ""
#         authors = [a.strip() for a in str(x).split(";") if a.strip()]
#         norm = [norm_fn(a) for a in authors]
#         norm = [z for z in norm if z]
#         norm = sorted(set(norm))
#         return ";".join(norm)

#     return s.apply(row_to_fp)


# def _author_tokens_from_fp(fp: str) -> List[str]:
#     if not fp:
#         return []
#     return [t for t in fp.split(";") if t]


# def _overlap_count(a_tokens: List[str], b_tokens: List[str]) -> int:
#     if not a_tokens or not b_tokens:
#         return 0
#     return len(set(a_tokens) & set(b_tokens))


# # ============================================================
# # 4) Single-stage dedupe:
# #    - Prefilter (title_dup/author_dup/none)
# #    - Stage A strict (title + authors_fp) exact match
# #    - Optional fuzzy title fallback (within same authors_fp)
# #    - Optional Stage B relaxed (shared authors overlap) within exact-title groups
# # ============================================================
# def dedupe_title_authors_stage(
#     df: pd.DataFrame,
#     *,
#     # stage config
#     stage_name: str = "stage",
#     authors_fp_mode: str = "tokenbag",         # tokenbag | last_initial | last

#     # fuzzy config (title containment), executed only if enabled
#     title_fuzzy_fallback: bool = False,
#     min_title_tokens: int = 6,
#     min_title_containment: float = 0.70,
#     fuzzy_compare_strategy: str = "parent_only",  # parent_only | all_pairs_small (parent_only is safest/fastest)

#     # relaxed (shared authors overlap) config (exact title only)
#     relaxed_shared_authors: bool = True,
#     min_authors_required: int = 2,
#     min_shared_authors: int = 2,

#     # prefilter strategy (important!)
#     prefilter_mode: str = "title_dup",         # title_dup | author_dup | none
#     prefilter: bool = True,                    # if False, skip ">=2" group filter (slower)

#     # global options
#     servers=None,
#     across_servers: bool = True,
#     use_year: bool = False,
#     choose_parent: str = "oldest",             # oldest | most_recent
#     overwrite_mode: str = "parent_only",       # any | parent_only | unlabeled_only

#     # columns
#     server_col: str = "server_name",
#     record_id_col: str = "record_id",
#     title_col: str = "title",
#     authors_col: str = "authors_flat",
#     year_col: str = "publication_year_first_seen",
#     date_candidates: Tuple[str, ...] = ("date_first_seen",),

#     hierarchy_col: str = "records_hierarchy",
#     parent_id_col: str = "parent_record_id",
#     group_id_col: str = "dup_group_id",

#     # caching/debug columns
#     add_authors_fingerprint_col: bool = True,
#     authors_fingerprint_col: str = "authors_fp",
#     add_title_clean_col: bool = True,
#     title_clean_col: str = "title_clean_v2",

#     return_metrics: bool = False,
# ) -> pd.DataFrame | Tuple[pd.DataFrame, Dict[str, Any]]:
#     """
#     One dedupe stage. Designed to be composed into a multi-stage pipeline.
#     """

#     t0 = time.perf_counter()

#     metrics: Dict[str, Any] = {
#         "stage_name": stage_name,
#         "n_rows_df": int(len(df)),
#         "n_candidates_initial": 0,
#         "prefilter_mode": prefilter_mode,
#         "prefilter_rows": 0,
#         "prefilter_groups": 0,
#         "work_rows_after_keys": 0,

#         "stageA_groups": 0,
#         "stageA_children_labeled": 0,

#         "fuzzy_enabled": bool(title_fuzzy_fallback),
#         "fuzzy_groups": 0,
#         "fuzzy_pairs_checked": 0,
#         "fuzzy_children_labeled": 0,

#         "stageB_enabled": bool(relaxed_shared_authors),
#         "stageB_title_groups": 0,
#         "stageB_clusters": 0,
#         "stageB_children_labeled": 0,

#         "time_s": 0.0,
#     }

#     # ------------------------------------------------------
#     # Ensure output cols exist
#     # ------------------------------------------------------
#     for c in (hierarchy_col, parent_id_col, group_id_col):
#         if c not in df.columns:
#             df[c] = pd.NA

#     if add_authors_fingerprint_col and authors_fingerprint_col not in df.columns:
#         df[authors_fingerprint_col] = pd.NA
#     if add_title_clean_col and title_clean_col not in df.columns:
#         df[title_clean_col] = pd.NA

#     # ------------------------------------------------------
#     # Eligibility
#     # ------------------------------------------------------
#     h = df[hierarchy_col]
#     if overwrite_mode == "any":
#         eligible = pd.Series(True, index=df.index)
#     elif overwrite_mode == "parent_only":
#         eligible = h.astype("string").str.lower().str.strip().eq("parent")
#     elif overwrite_mode == "unlabeled_only":
#         eligible = h.isna()
#     else:
#         raise ValueError("overwrite_mode must be any | parent_only | unlabeled_only")

#     # server filter
#     if servers is None:
#         server_mask = pd.Series(True, index=df.index)
#     elif isinstance(servers, str):
#         server_mask = df[server_col].eq(servers)
#     else:
#         server_mask = df[server_col].isin(list(servers))

#     m = eligible & server_mask
#     metrics["n_candidates_initial"] = int(m.sum())
#     if not m.any():
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     # ------------------------------------------------------
#     # Prefilter: decide which indices to consider in this stage
#     # ------------------------------------------------------
#     if prefilter_mode == "title_dup":
#         # title-based prefilter (fast for exact title stages)
#         t_clean = df.loc[m, title_clean_col] if (add_title_clean_col and title_clean_col in df.columns and df.loc[m, title_clean_col].notna().any()) else None
#         if t_clean is None:
#             t_clean = _clean_title_series_v2(df.loc[m, title_col])
#         vc = t_clean.value_counts()
#         keep_idx = t_clean[t_clean.isin(vc[vc >= 2].index)].index
#         metrics["prefilter_groups"] = int((vc >= 2).sum())

#     elif prefilter_mode == "author_dup":
#         # author-fp based prefilter (crucial for fuzzy pass; titles may differ)
#         # compute fp only for m rows
#         a_fp = build_authors_fingerprint_series(df.loc[m, authors_col], mode=authors_fp_mode)
#         vc = a_fp.value_counts()
#         keep_idx = a_fp[a_fp.isin(vc[vc >= 2].index)].index
#         metrics["prefilter_groups"] = int((vc >= 2).sum())

#     elif prefilter_mode == "none":
#         keep_idx = df.index[m]
#         metrics["prefilter_groups"] = 0

#     else:
#         raise ValueError("prefilter_mode must be title_dup | author_dup | none")

#     metrics["prefilter_rows"] = int(len(keep_idx))
#     if len(keep_idx) == 0:
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     # ------------------------------------------------------
#     # Work subset + compute/attach cached normalization keys
#     # ------------------------------------------------------
#     cols_needed = [server_col, record_id_col, title_col, authors_col]
#     if use_year:
#         cols_needed.append(year_col)
#     date_col = _pick_first_existing(df, date_candidates)
#     if date_col:
#         cols_needed.append(date_col)

#     work = df.loc[keep_idx, cols_needed].copy()

#     # Title clean (cache to df if asked)
#     if add_title_clean_col:
#         # compute for missing only (cheap)
#         t_missing = df.loc[work.index, title_clean_col].isna()
#         if t_missing.any():
#             df.loc[work.index[t_missing], title_clean_col] = _clean_title_series_v2(df.loc[work.index[t_missing], title_col]).values
#         work["_t"] = df.loc[work.index, title_clean_col].astype("string").fillna("")
#     else:
#         work["_t"] = _clean_title_series_v2(work[title_col])

#     work["_t"] = _blank_risky_titles(work["_t"])

    
#     # Authors fp (mode-specific; cache into df column if asked)
#     work["_a_fp"] = build_authors_fingerprint_series(work[authors_col], mode=authors_fp_mode)
#     if add_authors_fingerprint_col:
#         df.loc[work.index, authors_fingerprint_col] = work["_a_fp"].values

#     # Year (optional)
#     if use_year:
#         y = pd.to_numeric(work[year_col], errors="coerce")
#         y = y.where((y >= 1000) & (y <= 3000)).round().astype("Int64")
#         work["_y"] = y
#     else:
#         work["_y"] = pd.NA

#     # require non-empty keys
#     if use_year:
#         work = work[(work["_t"] != "") & (work["_a_fp"] != "") & work["_y"].notna()].copy()
#     else:
#         work = work[(work["_t"] != "") & (work["_a_fp"] != "")].copy()

#     metrics["work_rows_after_keys"] = int(len(work))
#     if work.empty:
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     # ------------------------------------------------------
#     # Stage A STRICT: exact match on (title_clean + authors_fp [+year] [+server scope])
#     # ------------------------------------------------------
#     if use_year:
#         strict_base = work["_t"] + "||" + work["_a_fp"] + "||" + work["_y"].astype("string")
#     else:
#         strict_base = work["_t"] + "||" + work["_a_fp"]

#     if across_servers:
#         work["_grp_strict"] = strict_base
#     else:
#         work["_grp_strict"] = work[server_col].astype("string") + "||" + strict_base

#     strict = work
#     if prefilter:
#         vcg = work["_grp_strict"].value_counts()
#         dup_keys = vcg[vcg >= 2].index
#         strict = work[work["_grp_strict"].isin(dup_keys)].copy()

#     metrics["stageA_groups"] = int(strict["_grp_strict"].nunique()) if not strict.empty else 0

#     # sort keys for parent choice
#     if date_col and date_col in strict.columns:
#         strict["_dt"] = pd.to_datetime(strict[date_col], errors="coerce")
#     else:
#         strict["_dt"] = pd.NaT
#     strict["_rid"] = _record_id_key(strict[record_id_col])

#     if not strict.empty:
#         if choose_parent == "oldest":
#             strict = strict.sort_values(
#                 by=["_grp_strict", "_dt", "_rid"],
#                 ascending=[True, True, True],
#                 na_position="last",
#             )
#         elif choose_parent == "most_recent":
#             strict = strict.sort_values(
#                 by=["_grp_strict", "_dt", "_rid"],
#                 ascending=[True, False, False],
#                 na_position="last",
#             )
#         else:
#             raise ValueError("choose_parent must be oldest | most_recent")

#         parents = strict.groupby("_grp_strict", sort=False).head(1)
#         parent_rid_map = parents.set_index("_grp_strict")[record_id_col]
#         parent_srv_map = parents.set_index("_grp_strict")[server_col]

#         strict["_parent_rid"] = strict["_grp_strict"].map(parent_rid_map)
#         strict["_parent_srv"] = strict["_grp_strict"].map(parent_srv_map)

#         is_parent = strict[record_id_col].eq(strict["_parent_rid"])
#         parent_idx = strict.index[is_parent]
#         child_idx = strict.index[~is_parent]

#         metrics["stageA_children_labeled"] = int(len(child_idx))

#         df.loc[parent_idx, hierarchy_col] = "parent"
#         df.loc[parent_idx, parent_id_col] = pd.NA
#         df.loc[child_idx, hierarchy_col] = (
#             "parent - duplicate (" + strict.loc[child_idx, "_parent_srv"].astype("string") + ")"
#         )
#         df.loc[child_idx, parent_id_col] = strict.loc[child_idx, "_parent_rid"].values

#         # deterministic group id
#         df.loc[strict.index, group_id_col] = (
#             pd.util.hash_pandas_object(strict["_grp_strict"], index=False)
#             .astype("uint64")
#             .astype(str)
#             .values
#         )

#     # ------------------------------------------------------
#     # Fuzzy title fallback (BLOCKED by authors_fp [+year], only remaining eligible)
#     # Important: this can find near-duplicate titles because we do NOT rely on title_dup.
#     # ------------------------------------------------------
#     if title_fuzzy_fallback:
#         # remaining eligible after Stage A
#         h2 = df[hierarchy_col]
#         if overwrite_mode == "parent_only":
#             eligible2 = h2.astype("string").str.lower().str.strip().eq("parent")
#         elif overwrite_mode == "unlabeled_only":
#             eligible2 = h2.isna()
#         else:
#             eligible2 = pd.Series(True, index=df.index)

#         remain_idx = work.index.intersection(df.index[eligible2])
#         wF = work.loc[remain_idx].copy()

#         if not wF.empty:
#             # block by authors_fp (+year) because authors are "more trustworthy"
#             if use_year:
#                 wF["_grp_auth"] = wF["_a_fp"] + "||" + wF["_y"].astype("string")
#             else:
#                 wF["_grp_auth"] = wF["_a_fp"]

#             # keep only blocks with >=2 rows
#             vc_auth = wF["_grp_auth"].value_counts()
#             keep_auth = vc_auth[vc_auth >= 2].index
#             wF = wF[wF["_grp_auth"].isin(keep_auth)].copy()

#             metrics["fuzzy_groups"] = int(wF["_grp_auth"].nunique()) if not wF.empty else 0

#             if not wF.empty:
#                 # date/rid for parent selection
#                 if date_col and date_col in wF.columns:
#                     wF["_dt"] = pd.to_datetime(wF[date_col], errors="coerce")
#                 else:
#                     wF["_dt"] = pd.NaT
#                 wF["_rid"] = _record_id_key(wF[record_id_col])

#                 # tokens cache per row (within this stage)
#                 tokens_map = {idx: _title_tokens_from_clean(wF.loc[idx, "_t"]) for idx in wF.index}

#                 for grp, g in wF.groupby("_grp_auth", sort=False):
#                     if len(g) < 2:
#                         continue

#                     # gate: ignore titles with too few tokens
#                     idxs = [idx for idx in g.index if len(tokens_map.get(idx, [])) >= min_title_tokens]
#                     if len(idxs) < 2:
#                         continue

#                     gg = g.loc[idxs].copy()
#                     if choose_parent == "oldest":
#                         gg = gg.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
#                     else:
#                         gg = gg.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

#                     if fuzzy_compare_strategy == "parent_only":
#                         parent_idx = gg.index[0]
#                         parent_tokens = tokens_map[parent_idx]
#                         parent_rid = gg.loc[parent_idx, record_id_col]
#                         parent_srv = gg.loc[parent_idx, server_col]

#                         # ensure parent labeled
#                         df.loc[parent_idx, hierarchy_col] = "parent"
#                         df.loc[parent_idx, parent_id_col] = pd.NA

#                         for idx in gg.index[1:]:
#                             metrics["fuzzy_pairs_checked"] += 1
#                             sc = _containment_score(parent_tokens, tokens_map[idx])
#                             if sc >= min_title_containment:
#                                 df.loc[idx, hierarchy_col] = f"parent - duplicate ({parent_srv})"
#                                 df.loc[idx, parent_id_col] = parent_rid
#                                 df.loc[idx, group_id_col] = f"fuzzy::{authors_fp_mode}::{grp}"
#                                 metrics["fuzzy_children_labeled"] += 1

#                     elif fuzzy_compare_strategy == "all_pairs_small":
#                         # safer than global all-pairs; still can be heavy if blocks are large.
#                         # We'll cluster by greedy expansion (bounded within block).
#                         idxs2 = gg.index.tolist()
#                         used = set()
#                         for i in idxs2:
#                             if i in used:
#                                 continue
#                             used.add(i)
#                             cluster = [i]
#                             for j in idxs2:
#                                 if j in used:
#                                     continue
#                                 metrics["fuzzy_pairs_checked"] += 1
#                                 sc = _containment_score(tokens_map[i], tokens_map[j])
#                                 if sc >= min_title_containment:
#                                     used.add(j)
#                                     cluster.append(j)

#                             if len(cluster) >= 2:
#                                 # choose parent (oldest/most recent) within cluster
#                                 cldf = gg.loc[cluster].copy()
#                                 if choose_parent == "oldest":
#                                     cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
#                                 else:
#                                     cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

#                                 p_idx = cldf.index[0]
#                                 p_rid = cldf.loc[p_idx, record_id_col]
#                                 p_srv = cldf.loc[p_idx, server_col]
#                                 df.loc[p_idx, hierarchy_col] = "parent"
#                                 df.loc[p_idx, parent_id_col] = pd.NA
#                                 for cidx in cldf.index[1:]:
#                                     df.loc[cidx, hierarchy_col] = f"parent - duplicate ({p_srv})"
#                                     df.loc[cidx, parent_id_col] = p_rid
#                                     df.loc[cidx, group_id_col] = f"fuzzy::{authors_fp_mode}::{grp}"
#                                     metrics["fuzzy_children_labeled"] += 1
#                     else:
#                         raise ValueError("fuzzy_compare_strategy must be parent_only | all_pairs_small")

#     # ------------------------------------------------------
#     # Stage B RELAXED (shared authors overlap) within exact title groups
#     # ------------------------------------------------------
#     if relaxed_shared_authors:
#         h3 = df[hierarchy_col]
#         if overwrite_mode == "parent_only":
#             eligible3 = h3.astype("string").str.lower().str.strip().eq("parent")
#         elif overwrite_mode == "unlabeled_only":
#             eligible3 = h3.isna()
#         else:
#             eligible3 = pd.Series(True, index=df.index)

#         remain_idx = work.index.intersection(df.index[eligible3])
#         w2 = work.loc[remain_idx].copy()
#         if not w2.empty:
#             if use_year:
#                 relaxed_base = w2["_t"] + "||" + w2["_y"].astype("string")
#             else:
#                 relaxed_base = w2["_t"]

#             if across_servers:
#                 w2["_grp_title"] = relaxed_base
#             else:
#                 w2["_grp_title"] = w2[server_col].astype("string") + "||" + relaxed_base

#             # keep only repeated titles
#             vc2 = w2["_grp_title"].value_counts()
#             keep_groups = vc2[vc2 >= 2].index
#             w2 = w2[w2["_grp_title"].isin(keep_groups)].copy()

#             metrics["stageB_title_groups"] = int(w2["_grp_title"].nunique()) if not w2.empty else 0

#             if not w2.empty:
#                 w2["_a_tokens"] = w2["_a_fp"].apply(_author_tokens_from_fp)
#                 w2["_a_n"] = w2["_a_tokens"].apply(len)

#                 if date_col and date_col in w2.columns:
#                     w2["_dt"] = pd.to_datetime(w2[date_col], errors="coerce")
#                 else:
#                     w2["_dt"] = pd.NaT
#                 w2["_rid"] = _record_id_key(w2[record_id_col])

#                 group_counter = 0
#                 children_total = 0

#                 for grp, g in w2.groupby("_grp_title", sort=False):
#                     if len(g) < 2:
#                         continue

#                     g = g[g["_a_n"] >= min_authors_required].copy()
#                     if len(g) < 2:
#                         continue

#                     idxs = g.index.tolist()
#                     used = set()
#                     clusters = []

#                     # simple greedy clustering based on author overlap
#                     for i in idxs:
#                         if i in used:
#                             continue
#                         used.add(i)
#                         cl = [i]
#                         for j in idxs:
#                             if j in used:
#                                 continue
#                             if _overlap_count(g.loc[i, "_a_tokens"], g.loc[j, "_a_tokens"]) >= min_shared_authors:
#                                 used.add(j)
#                                 cl.append(j)
#                         if len(cl) >= 2:
#                             clusters.append(cl)

#                     for cl in clusters:
#                         group_counter += 1
#                         cldf = g.loc[cl].copy()
#                         if choose_parent == "oldest":
#                             cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
#                         else:
#                             cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

#                         parent_idx = cldf.index[0]
#                         parent_rid = cldf.loc[parent_idx, record_id_col]
#                         parent_srv = cldf.loc[parent_idx, server_col]

#                         df.loc[parent_idx, hierarchy_col] = "parent"
#                         df.loc[parent_idx, parent_id_col] = pd.NA
#                         df.loc[parent_idx, group_id_col] = f"relaxed::{stage_name}::{group_counter}"

#                         child_idxs = [x for x in cldf.index if x != parent_idx]
#                         children_total += len(child_idxs)

#                         df.loc[child_idxs, hierarchy_col] = f"parent - duplicate ({parent_srv})"
#                         df.loc[child_idxs, parent_id_col] = parent_rid
#                         df.loc[child_idxs, group_id_col] = f"relaxed::{stage_name}::{group_counter}"

#                 metrics["stageB_clusters"] = int(group_counter)
#                 metrics["stageB_children_labeled"] = int(children_total)

#     metrics["time_s"] = time.perf_counter() - t0
#     return (df, metrics) if return_metrics else df


# # ============================================================
# # 5) Stage runner with summary + early stop
# # ============================================================
# def _count_children_labels(series: pd.Series) -> int:
#     s = series.astype("string").fillna("")
#     return int(s.str.startswith("parent - duplicate").sum())


# def run_dedupe_stages(
#     df: pd.DataFrame,
#     *,
#     stages: List[Dict[str, Any]],
#     early_stop_if_new_labels_lt: int = 100,
#     print_summary: bool = True,
#     return_all_metrics: bool = True,
#     # common kwargs passed to every stage
#     **common_kwargs,
# ) -> Tuple[pd.DataFrame, List[Dict[str, Any]]] | pd.DataFrame:
#     """
#     Runs a list of stages sequentially with:
#       - delta duplicates added per stage
#       - early stop
#     """
#     df_out = df
#     metrics_all: List[Dict[str, Any]] = []

#     prev_children = _count_children_labels(df_out[common_kwargs.get("hierarchy_col", "records_hierarchy")])

#     for stage in stages:
#         name = stage.get("name", stage.get("stage_name", "stage"))
#         t0 = time.perf_counter()

#         df_out, m = dedupe_title_authors_stage(
#             df_out,
#             return_metrics=True,
#             stage_name=name,
#             **common_kwargs,
#             **{k: v for k, v in stage.items() if k not in {"name", "stage_name"}},
#         )

#         now_children = _count_children_labels(df_out[common_kwargs.get("hierarchy_col", "records_hierarchy")])
#         delta = now_children - prev_children
#         prev_children = now_children

#         m["stage_runtime_s"] = time.perf_counter() - t0
#         m["new_children_added"] = int(delta)
#         metrics_all.append(m)

#         if print_summary:
#             print(
#                 f"[{name}] new_children={delta} | "
#                 f"cand={m['n_candidates_initial']} | "
#                 f"prefilter_rows={m['prefilter_rows']} | "
#                 f"A_children={m['stageA_children_labeled']} | "
#                 f"fuzzy_children={m['fuzzy_children_labeled']} | "
#                 f"B_children={m['stageB_children_labeled']} | "
#                 f"time={m['stage_runtime_s']:.2f}s"
#             )

#         if delta < early_stop_if_new_labels_lt:
#             if print_summary:
#                 print(f"Early stop after {name}: delta {delta} < {early_stop_if_new_labels_lt}")
#             break

#     return (df_out, metrics_all) if return_all_metrics else df_out


# # ============================================================
# # 6) Two-pass pipeline: Exact pass -> Fuzzy pass on remaining parents
# # ============================================================
# def run_dedupe_pipeline_two_passes(
#     df: pd.DataFrame,
#     *,
#     stages_exact: List[Dict[str, Any]],
#     stages_fuzzy: List[Dict[str, Any]],
#     early_stop_if_new_labels_lt: int = 100,
#     print_summary: bool = True,
#     return_all_metrics: bool = True,
#     **common_kwargs,
# ) -> Tuple[pd.DataFrame, List[Dict[str, Any]]] | pd.DataFrame:
#     """
#     Pass A: run stages_exact (typically no fuzzy, prefilter_mode=title_dup).
#     Pass B: run stages_fuzzy (fuzzy enabled, prefilter_mode=author_dup), on remaining parents only.

#     IMPORTANT:
#       - For Pass A, it is normal to use overwrite_mode="any" for stage1, then "parent_only" for stage2-3.
#       - For Pass B, use overwrite_mode="parent_only" so we only touch unresolved parents.
#     """
#     all_metrics: List[Dict[str, Any]] = []
#     df_out = df

#     if print_summary:
#         print("\n=== PASS A: EXACT ===")

#     df_out, mA = run_dedupe_stages(
#         df_out,
#         stages=stages_exact,
#         early_stop_if_new_labels_lt=early_stop_if_new_labels_lt,
#         print_summary=print_summary,
#         return_all_metrics=True,
#         **common_kwargs,
#     )
#     all_metrics.extend(mA)

#     if print_summary:
#         print("\n=== PASS B: FUZZY (remaining parents) ===")

#     df_out, mB = run_dedupe_stages(
#         df_out,
#         stages=stages_fuzzy,
#         early_stop_if_new_labels_lt=early_stop_if_new_labels_lt,
#         print_summary=print_summary,
#         return_all_metrics=True,
#         **common_kwargs,
#     )
#     all_metrics.extend(mB)

#     return (df_out, all_metrics) if return_all_metrics else df_out


# # ============================================================
# # 7) Default stage configs (recommended)
# # ============================================================

# # PASS A (EXACT) — fast + high precision
# STAGES_EXACT = [
#     dict(
#         name="A1_tokenbag_exact",
#         authors_fp_mode="tokenbag",
#         prefilter_mode="title_dup",
#         min_title_tokens=3,
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=True,
#         min_authors_required=1,
#         min_shared_authors=1,
#         overwrite_mode="parent_only",
#         authors_fingerprint_col="authors_fp_tokenbag",
#     ),
#     dict(
#         name="A2_last_initial_exact",
#         authors_fp_mode="last_initial",
#         prefilter_mode="title_dup",
#         min_title_tokens=3,
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=True,
#         min_authors_required=2,
#         min_shared_authors=2,
#         overwrite_mode="parent_only",
#         authors_fingerprint_col="authors_fp_last_initial",
#     ),
#     dict(
#         name="A3_last_exact_strict",
#         authors_fp_mode="last",
#         prefilter_mode="title_dup",
#         min_title_tokens=3,
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=False,  # last-only already high recall; keep strict
#         overwrite_mode="parent_only",
#         authors_fingerprint_col="authors_fp_last",
#     ),
# ]

# # PASS B (FUZZY) — only remaining parents; block by authors_fp repetition
# # Note: relaxed_shared_authors is usually OFF here to keep false positives down.
# STAGES_FUZZY = [
#     dict(
#         name="B1_tokenbag_fuzzy",
#         authors_fp_mode="tokenbag",
#         prefilter_mode="author_dup",
#         title_fuzzy_fallback=True,
#         min_title_tokens=8,
#         min_title_containment=0.80,  # start conservative; lower = more recall, more risk
#         fuzzy_compare_strategy="parent_only",
#         relaxed_shared_authors=False,
#         # min_authors_required=1,
#         # min_shared_authors=1,
#         overwrite_mode="parent_only",
#         authors_fingerprint_col="authors_fp_tokenbag",
#     ),
#     dict(
#         name="B2_last_initial_fuzzy",
#         authors_fp_mode="last_initial",
#         prefilter_mode="author_dup",
#         title_fuzzy_fallback=True,
#         min_title_tokens=8,
#         min_title_containment=0.90,
#         fuzzy_compare_strategy="parent_only",
#         relaxed_shared_authors=False,
#         # min_authors_required=1,
#         # min_shared_authors=1,
#         overwrite_mode="parent_only",
#         authors_fingerprint_col="authors_fp_last_initial",
#     ),
# ]

# # ============================================================
# # 8) Example usage
# # ============================================================
# # df_out, metrics = run_dedupe_pipeline_two_passes(
# #     df,
# #     stages_exact=STAGES_EXACT,
# #     stages_fuzzy=STAGES_FUZZY,
# #     early_stop_if_new_labels_lt=500,
# #     print_summary=True,
# #     return_all_metrics=True,
# #     servers=None,
# #     across_servers=True,
# #     use_year=False,
# #     choose_parent="oldest",
# #     prefilter=True,
# #     date_candidates=('date_first_seen',),
# #     hierarchy_col="records_hierarchy",
# #     parent_id_col="parent_record_id",
# #     group_id_col="dup_group_id",
# #     add_authors_fingerprint_col=True,
# #     add_title_clean_col=True,
# #     title_clean_col="title_clean_v2",
# # )
# #
# # print(metrics[-1])
# # print(df_out["records_hierarchy"].value_counts(dropna=False).head(60))


In [43]:
# data_out, metrics = run_dedupe_pipeline_two_passes(
#     data_clean_hierarchy,
#     stages_exact=STAGES_EXACT,
#     stages_fuzzy=STAGES_FUZZY,
#     early_stop_if_new_labels_lt=1,
#     print_summary=True,
#     return_all_metrics=True,
#     servers=None,
#     across_servers=True,
#     use_year=False,
#     choose_parent="oldest",
#     prefilter=True,
#     date_candidates=('date_first_seen',),
#     hierarchy_col="records_hierarchy",
#     parent_id_col="parent_record_id",
#     group_id_col="dup_group_id",
#     add_authors_fingerprint_col=True,
#     add_title_clean_col=True,
#     title_clean_col="title_clean_v2",
# )

# print(metrics[-1])
# print(data_out["records_hierarchy"].value_counts(dropna=False).head(60))


## new

In [44]:
# import pandas as pd
# import numpy as np
# import re
# import time
# import unicodedata
# from typing import Iterable, Optional, Dict, Any, Tuple, List


# # ============================================================
# # 0) Regex + constants
# # ============================================================

# _WS = re.compile(r"\s+")
# _PUNCT_ALL = re.compile(r"[^\w\s]", re.UNICODE)

# NA_LIKE = {"", "none", "null", "nan", "n/a", "[]", "{}", "na"}

# GENERIC_TITLES = frozenset({
#     "front matter", "back matter", "summary", "summaries",
#     "reviews in brief", "publications received",
#     "agricultural letter",
#     "administrasi kurikulum", "administrasi peserta didik",
#     "experiment ended", "editorial", "introduction", "preface",
#     "contents", "table of contents", "index", "book review",
#     "letter", "news", "announcement", "abstract", "poster",
#     "supplement", "no title", "resumes", "retracted",
#     "bibliographie", "bremsstrahlung",
# })

# MIN_TITLE_TOKENS = 3
# MIN_TITLE_CHARS = 25


# # ============================================================
# # 1) Helpers
# # ============================================================

# def _strip_accents_text(x: str) -> str:
#     return "".join(
#         c for c in unicodedata.normalize("NFKD", str(x))
#         if not unicodedata.combining(c)
#     )


# def _clean_title_one(x: str) -> str:
#     if pd.isna(x):
#         return ""
#     x = str(x).strip().lower()
#     if x in NA_LIKE:
#         return ""
#     x = _strip_accents_text(x)
#     x = _PUNCT_ALL.sub(" ", x)
#     x = _WS.sub(" ", x).strip()
#     return x


# def is_risky_title(title_clean: str) -> bool:
#     if not title_clean:
#         return True

#     if title_clean in GENERIC_TITLES:
#         return True

#     tokens = title_clean.split()

#     if len(tokens) < MIN_TITLE_TOKENS:
#         return True

#     if len(title_clean) < MIN_TITLE_CHARS:
#         return True

#     return False


# def _record_id_key(s: pd.Series) -> pd.Series:
#     digits = s.astype("string").str.extract(r"(\d+)")[0]
#     return pd.to_numeric(digits, errors="coerce")


# def _pick_first_existing(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
#     for c in candidates:
#         if c in df.columns:
#             return c
#     return None


# def _title_tokens_from_clean(title_clean: str) -> List[str]:
#     if not title_clean:
#         return []
#     return [t for t in str(title_clean).split() if len(t) >= 2]


# def _containment_score(a_tokens: List[str], b_tokens: List[str]) -> float:
#     if not a_tokens or not b_tokens:
#         return 0.0
#     A, B = set(a_tokens), set(b_tokens)
#     denom = min(len(A), len(B))
#     if denom == 0:
#         return 0.0
#     return len(A & B) / denom


# def _author_tokens_from_fp(fp: str) -> List[str]:
#     if not fp:
#         return []
#     return [t for t in str(fp).split(";") if t]


# def _overlap_count(a_tokens: List[str], b_tokens: List[str]) -> int:
#     if not a_tokens or not b_tokens:
#         return 0
#     return len(set(a_tokens) & set(b_tokens))


# # ============================================================
# # 2) Author normalization
# # ============================================================

# def _normalize_one_author_tokenbag(author: str) -> str:
#     if not author:
#         return ""

#     a = _strip_accents_text(author).lower().strip()
#     if not a or a in NA_LIKE:
#         return ""

#     a = _PUNCT_ALL.sub(" ", a)
#     a = _WS.sub(" ", a).strip()

#     toks = [t for t in a.split() if t]
#     if not toks:
#         return ""

#     return "_".join(sorted(toks))


# def _normalize_one_author_last_initial(author: str) -> str:
#     if not author:
#         return ""

#     a = _strip_accents_text(author).lower().strip()
#     if not a or a in NA_LIKE:
#         return ""

#     if "," in a:
#         left, right = a.split(",", 1)

#         left = _PUNCT_ALL.sub(" ", left)
#         right = _PUNCT_ALL.sub(" ", right)

#         left = _WS.sub(" ", left).strip()
#         right = _WS.sub(" ", right).strip()

#         last_toks = [t for t in left.split() if t]
#         first_toks = [t for t in right.split() if t]

#         if not last_toks or not first_toks:
#             return ""

#         last = last_toks[0]
#         ini = first_toks[0][0]

#         return f"{last}|{ini}"

#     a = _PUNCT_ALL.sub(" ", a)
#     a = _WS.sub(" ", a).strip()

#     toks = [t for t in a.split() if t]

#     if len(toks) < 2:
#         return ""

#     ini = toks[0][0]
#     last = toks[-1]

#     return f"{last}|{ini}"


# def _normalize_one_author_last(author: str) -> str:
#     if not author:
#         return ""

#     a = _strip_accents_text(author).lower().strip()
#     if not a or a in NA_LIKE:
#         return ""

#     if "," in a:
#         left = a.split(",", 1)[0]
#         left = _PUNCT_ALL.sub(" ", left)
#         left = _WS.sub(" ", left).strip()
#         toks = [t for t in left.split() if t]
#         return toks[0] if toks else ""

#     a = _PUNCT_ALL.sub(" ", a)
#     a = _WS.sub(" ", a).strip()

#     toks = [t for t in a.split() if t]
#     return toks[-1] if toks else ""


# def build_authors_fingerprint_series(authors_flat: pd.Series, mode: str) -> pd.Series:
#     if mode not in {"tokenbag", "last_initial", "last"}:
#         raise ValueError("mode must be tokenbag | last_initial | last")

#     if mode == "tokenbag":
#         norm_fn = _normalize_one_author_tokenbag
#     elif mode == "last_initial":
#         norm_fn = _normalize_one_author_last_initial
#     else:
#         norm_fn = _normalize_one_author_last

#     def row_to_fp(x) -> str:
#         if pd.isna(x):
#             return ""

#         x = str(x).strip()
#         if not x or x.lower() in NA_LIKE:
#             return ""

#         authors = [a.strip() for a in x.split(";") if a.strip()]
#         norm = [norm_fn(a) for a in authors]
#         norm = sorted(set(z for z in norm if z))

#         return ";".join(norm)

#     return authors_flat.apply(row_to_fp)


# # ============================================================
# # 3) Precompute all expensive keys once
# # ============================================================

# def prepare_dedupe_keys(
#     df: pd.DataFrame,
#     *,
#     title_col: str = "title",
#     authors_col: str = "authors_flat",
#     title_clean_col: str = "title_clean_v2",
#     tokenbag_col: str = "authors_fp_tokenbag",
#     last_initial_col: str = "authors_fp_last_initial",
#     last_col: str = "authors_fp_last",
#     force_recompute: bool = False,
#     print_progress: bool = True,
# ) -> pd.DataFrame:
#     """
#     Precompute expensive title and author keys once.
#     This is the main speed improvement.
#     """

#     t0 = time.perf_counter()

#     if print_progress:
#         print("Preparing dedupe keys...")

#     if force_recompute or title_clean_col not in df.columns:
#         if print_progress:
#             print("  - Cleaning titles...")

#         df[title_clean_col] = df[title_col].apply(_clean_title_one)
#         risky = df[title_clean_col].apply(is_risky_title)
#         df.loc[risky, title_clean_col] = ""

#     if force_recompute or tokenbag_col not in df.columns:
#         if print_progress:
#             print("  - Building tokenbag author fingerprints...")

#         df[tokenbag_col] = build_authors_fingerprint_series(df[authors_col], "tokenbag")

#     if force_recompute or last_initial_col not in df.columns:
#         if print_progress:
#             print("  - Building last_initial author fingerprints...")

#         df[last_initial_col] = build_authors_fingerprint_series(df[authors_col], "last_initial")

#     if force_recompute or last_col not in df.columns:
#         if print_progress:
#             print("  - Building last-name author fingerprints...")

#         df[last_col] = build_authors_fingerprint_series(df[authors_col], "last")

#     if print_progress:
#         print(f"Keys prepared in {time.perf_counter() - t0:.2f}s")

#     return df


# # ============================================================
# # 4) One dedupe stage using cached keys
# # ============================================================

# def dedupe_title_authors_stage_fast(
#     df: pd.DataFrame,
#     *,
#     stage_name: str,
#     authors_fp_col: str,

#     title_col: str = "title",
#     authors_col: str = "authors_flat",
    
#     title_fuzzy_fallback: bool = False,
#     min_title_tokens: int = 6,
#     min_title_containment: float = 0.80,
#     fuzzy_compare_strategy: str = "parent_only",

#     relaxed_shared_authors: bool = True,
#     min_authors_required: int = 2,
#     min_shared_authors: int = 2,

#     prefilter_mode: str = "title_dup",
#     prefilter: bool = True,

#     servers=None,
#     across_servers: bool = True,
#     use_year: bool = False,
#     choose_parent: str = "oldest",
#     overwrite_mode: str = "parent_only",

#     server_col: str = "server_name",
#     record_id_col: str = "record_id",
#     year_col: str = "publication_year_first_seen",
#     date_candidates: Tuple[str, ...] = ("date_first_seen",),

#     title_clean_col: str = "title_clean_v2",
#     hierarchy_col: str = "records_hierarchy",
#     parent_id_col: str = "parent_record_id",
#     group_id_col: str = "dup_group_id",

#     print_progress: bool = True,
#     return_metrics: bool = False,
# ):
#     t0 = time.perf_counter()

#     metrics = {
#         "stage_name": stage_name,
#         "n_rows_df": int(len(df)),
#         "n_candidates_initial": 0,
#         "prefilter_rows": 0,
#         "prefilter_groups": 0,
#         "work_rows_after_keys": 0,
#         "stageA_groups": 0,
#         "stageA_children_labeled": 0,
#         "fuzzy_groups": 0,
#         "fuzzy_pairs_checked": 0,
#         "fuzzy_children_labeled": 0,
#         "stageB_title_groups": 0,
#         "stageB_clusters": 0,
#         "stageB_children_labeled": 0,
#         "time_s": 0.0,
#     }

#     for c in [hierarchy_col, parent_id_col, group_id_col]:
#         if c not in df.columns:
#             df[c] = pd.NA

#     if title_clean_col not in df.columns:
#         raise ValueError(f"Missing {title_clean_col}. Run prepare_dedupe_keys() first.")

#     if authors_fp_col not in df.columns:
#         raise ValueError(f"Missing {authors_fp_col}. Run prepare_dedupe_keys() first.")

#     h = df[hierarchy_col]

#     if overwrite_mode == "any":
#         eligible = pd.Series(True, index=df.index)
#     elif overwrite_mode == "parent_only":
#         eligible = h.astype("string").str.lower().str.strip().eq("parent")
#     elif overwrite_mode == "unlabeled_only":
#         eligible = h.isna()
#     else:
#         raise ValueError("overwrite_mode must be any | parent_only | unlabeled_only")

#     if servers is None:
#         server_mask = pd.Series(True, index=df.index)
#     elif isinstance(servers, str):
#         server_mask = df[server_col].eq(servers)
#     else:
#         server_mask = df[server_col].isin(list(servers))

#     m = eligible & server_mask
#     metrics["n_candidates_initial"] = int(m.sum())

#     if not m.any():
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     # -------------------------------
#     # Prefilter
#     # -------------------------------
#     if prefilter_mode == "title_dup":
#         key = df.loc[m, title_clean_col].astype("string").fillna("")
#     elif prefilter_mode == "author_dup":
#         key = df.loc[m, authors_fp_col].astype("string").fillna("")
#     elif prefilter_mode == "none":
#         keep_idx = df.index[m]
#         key = None
#     else:
#         raise ValueError("prefilter_mode must be title_dup | author_dup | none")

#     if prefilter_mode != "none":
#         key = key[key != ""]
#         vc = key.value_counts()
#         dup_values = vc[vc >= 2].index
#         keep_idx = key[key.isin(dup_values)].index
#         metrics["prefilter_groups"] = int(len(dup_values))

#     metrics["prefilter_rows"] = int(len(keep_idx))

#     if len(keep_idx) == 0:
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     date_col = _pick_first_existing(df, date_candidates)

#     cols = [server_col, record_id_col, title_clean_col, authors_fp_col]
#     if use_year:
#         cols.append(year_col)
#     if date_col:
#         cols.append(date_col)

#     work = df.loc[keep_idx, cols].copy()
#     work["_t"] = work[title_clean_col].astype("string").fillna("")
#     work["_a_fp"] = work[authors_fp_col].astype("string").fillna("")

#     if use_year:
#         y = pd.to_numeric(work[year_col], errors="coerce")
#         y = y.where((y >= 1000) & (y <= 3000)).round().astype("Int64")
#         work["_y"] = y
#         work = work[(work["_t"] != "") & (work["_a_fp"] != "") & work["_y"].notna()].copy()
#     else:
#         work["_y"] = pd.NA
#         work = work[(work["_t"] != "") & (work["_a_fp"] != "")].copy()

#     metrics["work_rows_after_keys"] = int(len(work))

#     if work.empty:
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     # -------------------------------
#     # Stage A: exact title + exact authors_fp
#     # -------------------------------
#     if use_year:
#         strict_base = work["_t"] + "||" + work["_a_fp"] + "||" + work["_y"].astype("string")
#     else:
#         strict_base = work["_t"] + "||" + work["_a_fp"]

#     if across_servers:
#         work["_grp_strict"] = strict_base
#     else:
#         work["_grp_strict"] = work[server_col].astype("string") + "||" + strict_base

#     strict = work

#     if prefilter:
#         vcg = strict["_grp_strict"].value_counts()
#         dup_keys = vcg[vcg >= 2].index
#         strict = strict[strict["_grp_strict"].isin(dup_keys)].copy()

#     metrics["stageA_groups"] = int(strict["_grp_strict"].nunique()) if not strict.empty else 0

#     if not strict.empty:
#         if date_col and date_col in strict.columns:
#             strict["_dt"] = pd.to_datetime(strict[date_col], errors="coerce")
#         else:
#             strict["_dt"] = pd.NaT

#         strict["_rid"] = _record_id_key(strict[record_id_col])

#         if choose_parent == "oldest":
#             strict = strict.sort_values(
#                 by=["_grp_strict", "_dt", "_rid"],
#                 ascending=[True, True, True],
#                 na_position="last",
#             )
#         elif choose_parent == "most_recent":
#             strict = strict.sort_values(
#                 by=["_grp_strict", "_dt", "_rid"],
#                 ascending=[True, False, False],
#                 na_position="last",
#             )
#         else:
#             raise ValueError("choose_parent must be oldest | most_recent")

#         parents = strict.groupby("_grp_strict", sort=False).head(1)
#         parent_rid_map = parents.set_index("_grp_strict")[record_id_col]
#         parent_srv_map = parents.set_index("_grp_strict")[server_col]

#         strict["_parent_rid"] = strict["_grp_strict"].map(parent_rid_map)
#         strict["_parent_srv"] = strict["_grp_strict"].map(parent_srv_map)

#         is_parent = strict[record_id_col].eq(strict["_parent_rid"])
#         parent_idx = strict.index[is_parent]
#         child_idx = strict.index[~is_parent]

#         metrics["stageA_children_labeled"] = int(len(child_idx))

#         df.loc[parent_idx, hierarchy_col] = "parent"
#         df.loc[parent_idx, parent_id_col] = pd.NA

#         df.loc[child_idx, hierarchy_col] = (
#             "parent - duplicate (" + strict.loc[child_idx, "_parent_srv"].astype("string") + ")"
#         )
#         df.loc[child_idx, parent_id_col] = strict.loc[child_idx, "_parent_rid"].values

#         df.loc[strict.index, group_id_col] = (
#             pd.util.hash_pandas_object(strict["_grp_strict"], index=False)
#             .astype("uint64")
#             .astype(str)
#             .values
#         )

#     # -------------------------------
#     # Fuzzy fallback
#     # -------------------------------
#     if title_fuzzy_fallback:
#         h2 = df[hierarchy_col]

#         if overwrite_mode == "parent_only":
#             eligible2 = h2.astype("string").str.lower().str.strip().eq("parent")
#         elif overwrite_mode == "unlabeled_only":
#             eligible2 = h2.isna()
#         else:
#             eligible2 = pd.Series(True, index=df.index)

#         remain_idx = work.index.intersection(df.index[eligible2])
#         wF = work.loc[remain_idx].copy()

#         if not wF.empty:
#             if use_year:
#                 wF["_grp_auth"] = wF["_a_fp"] + "||" + wF["_y"].astype("string")
#             else:
#                 wF["_grp_auth"] = wF["_a_fp"]

#             vc_auth = wF["_grp_auth"].value_counts()
#             keep_auth = vc_auth[vc_auth >= 2].index
#             wF = wF[wF["_grp_auth"].isin(keep_auth)].copy()

#             metrics["fuzzy_groups"] = int(wF["_grp_auth"].nunique()) if not wF.empty else 0

#             if not wF.empty:
#                 if date_col and date_col in wF.columns:
#                     wF["_dt"] = pd.to_datetime(wF[date_col], errors="coerce")
#                 else:
#                     wF["_dt"] = pd.NaT

#                 wF["_rid"] = _record_id_key(wF[record_id_col])
#                 tokens_map = {idx: _title_tokens_from_clean(wF.at[idx, "_t"]) for idx in wF.index}

#                 for grp, g in wF.groupby("_grp_auth", sort=False):
#                     idxs = [idx for idx in g.index if len(tokens_map.get(idx, [])) >= min_title_tokens]

#                     if len(idxs) < 2:
#                         continue

#                     gg = g.loc[idxs].copy()

#                     if choose_parent == "oldest":
#                         gg = gg.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
#                     else:
#                         gg = gg.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

#                     parent_idx = gg.index[0]
#                     parent_tokens = tokens_map[parent_idx]
#                     parent_rid = gg.loc[parent_idx, record_id_col]
#                     parent_srv = gg.loc[parent_idx, server_col]

#                     df.loc[parent_idx, hierarchy_col] = "parent"
#                     df.loc[parent_idx, parent_id_col] = pd.NA

#                     for idx in gg.index[1:]:
#                         metrics["fuzzy_pairs_checked"] += 1
#                         sc = _containment_score(parent_tokens, tokens_map[idx])

#                         if sc >= min_title_containment:
#                             df.loc[idx, hierarchy_col] = f"parent - duplicate ({parent_srv})"
#                             df.loc[idx, parent_id_col] = parent_rid
#                             df.loc[idx, group_id_col] = f"fuzzy::{stage_name}::{grp}"
#                             metrics["fuzzy_children_labeled"] += 1

#     # -------------------------------
#     # Stage B: relaxed same-title shared-author overlap
#     # -------------------------------
#     if relaxed_shared_authors:
#         h3 = df[hierarchy_col]

#         if overwrite_mode == "parent_only":
#             eligible3 = h3.astype("string").str.lower().str.strip().eq("parent")
#         elif overwrite_mode == "unlabeled_only":
#             eligible3 = h3.isna()
#         else:
#             eligible3 = pd.Series(True, index=df.index)

#         remain_idx = work.index.intersection(df.index[eligible3])
#         w2 = work.loc[remain_idx].copy()

#         if not w2.empty:
#             if use_year:
#                 relaxed_base = w2["_t"] + "||" + w2["_y"].astype("string")
#             else:
#                 relaxed_base = w2["_t"]

#             if across_servers:
#                 w2["_grp_title"] = relaxed_base
#             else:
#                 w2["_grp_title"] = w2[server_col].astype("string") + "||" + relaxed_base

#             vc2 = w2["_grp_title"].value_counts()
#             keep_groups = vc2[vc2 >= 2].index
#             w2 = w2[w2["_grp_title"].isin(keep_groups)].copy()

#             metrics["stageB_title_groups"] = int(w2["_grp_title"].nunique()) if not w2.empty else 0

#             if not w2.empty:
#                 w2["_a_tokens"] = w2["_a_fp"].apply(_author_tokens_from_fp)
#                 w2["_a_n"] = w2["_a_tokens"].apply(len)

#                 if date_col and date_col in w2.columns:
#                     w2["_dt"] = pd.to_datetime(w2[date_col], errors="coerce")
#                 else:
#                     w2["_dt"] = pd.NaT

#                 w2["_rid"] = _record_id_key(w2[record_id_col])

#                 group_counter = 0
#                 children_total = 0

#                 for grp, g in w2.groupby("_grp_title", sort=False):
#                     g = g[g["_a_n"] >= min_authors_required].copy()

#                     if len(g) < 2:
#                         continue

#                     idxs = g.index.tolist()
#                     used = set()
#                     clusters = []

#                     for i in idxs:
#                         if i in used:
#                             continue

#                         used.add(i)
#                         cluster = [i]

#                         for j in idxs:
#                             if j in used:
#                                 continue

#                             if _overlap_count(g.at[i, "_a_tokens"], g.at[j, "_a_tokens"]) >= min_shared_authors:
#                                 used.add(j)
#                                 cluster.append(j)

#                         if len(cluster) >= 2:
#                             clusters.append(cluster)

#                     for cluster in clusters:
#                         group_counter += 1
#                         cldf = g.loc[cluster].copy()

#                         if choose_parent == "oldest":
#                             cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[True, True], na_position="last")
#                         else:
#                             cldf = cldf.sort_values(by=["_dt", "_rid"], ascending=[False, False], na_position="last")

#                         parent_idx = cldf.index[0]
#                         parent_rid = cldf.loc[parent_idx, record_id_col]
#                         parent_srv = cldf.loc[parent_idx, server_col]

#                         child_idxs = [x for x in cldf.index if x != parent_idx]

#                         df.loc[parent_idx, hierarchy_col] = "parent"
#                         df.loc[parent_idx, parent_id_col] = pd.NA
#                         df.loc[parent_idx, group_id_col] = f"relaxed::{stage_name}::{group_counter}"

#                         df.loc[child_idxs, hierarchy_col] = f"parent - duplicate ({parent_srv})"
#                         df.loc[child_idxs, parent_id_col] = parent_rid
#                         df.loc[child_idxs, group_id_col] = f"relaxed::{stage_name}::{group_counter}"

#                         children_total += len(child_idxs)

#                 metrics["stageB_clusters"] = int(group_counter)
#                 metrics["stageB_children_labeled"] = int(children_total)

#     metrics["time_s"] = time.perf_counter() - t0

#     return (df, metrics) if return_metrics else df


# # ============================================================
# # 5) Runner
# # ============================================================

# def _count_children_labels(series: pd.Series) -> int:
#     s = series.astype("string").fillna("")
#     return int(s.str.startswith("parent - duplicate").sum())


# def run_dedupe_stages_fast(
#     df: pd.DataFrame,
#     *,
#     stages: List[Dict[str, Any]],
#     early_stop_if_new_labels_lt: int = 100,
#     print_summary: bool = True,
#     return_all_metrics: bool = True,
#     **common_kwargs,
# ):
#     df_out = df
#     metrics_all = []

#     hierarchy_col = common_kwargs.get("hierarchy_col", "records_hierarchy")
#     prev_children = _count_children_labels(df_out[hierarchy_col]) if hierarchy_col in df_out.columns else 0

#     for stage in stages:
#         name = stage.get("name", stage.get("stage_name", "stage"))

#         t0 = time.perf_counter()

#         # df_out, m = dedupe_title_authors_stage_fast(
#         #     df_out,
#         #     return_metrics=True,
#         #     stage_name=name,
#         #     **common_kwargs,
#         #     **{k: v for k, v in stage.items() if k not in {"name", "stage_name"}},
#         # )

#         stage_kwargs = dict(common_kwargs)
        
#         # These are only needed by prepare_dedupe_keys(), not by each stage
#         stage_kwargs.pop("title_col", None)
#         stage_kwargs.pop("authors_col", None)
        
#         df_out, m = dedupe_title_authors_stage_fast(
#             df_out,
#             return_metrics=True,
#             stage_name=name,
#             **stage_kwargs,
#             **{k: v for k, v in stage.items() if k not in {"name", "stage_name"}},
#         )

#         now_children = _count_children_labels(df_out[hierarchy_col])
#         delta = now_children - prev_children
#         prev_children = now_children

#         m["new_children_added"] = int(delta)
#         m["stage_runtime_s"] = time.perf_counter() - t0
#         metrics_all.append(m)

#         if print_summary:
#             print(
#                 f"[{name}] new_children={delta:,} | "
#                 f"cand={m['n_candidates_initial']:,} | "
#                 f"prefilter_rows={m['prefilter_rows']:,} | "
#                 f"A={m['stageA_children_labeled']:,} | "
#                 f"fuzzy={m['fuzzy_children_labeled']:,} | "
#                 f"B={m['stageB_children_labeled']:,} | "
#                 f"time={m['stage_runtime_s']:.2f}s"
#             )

#         if delta < early_stop_if_new_labels_lt:
#             if print_summary:
#                 print(f"Early stop after {name}: delta {delta:,} < {early_stop_if_new_labels_lt:,}")
#             break

#     return (df_out, metrics_all) if return_all_metrics else df_out


# def run_dedupe_pipeline_two_passes_fast(
#     df: pd.DataFrame,
#     *,
#     stages_exact: List[Dict[str, Any]],
#     stages_fuzzy: List[Dict[str, Any]],
#     early_stop_if_new_labels_lt: int = 100,
#     print_summary: bool = True,
#     return_all_metrics: bool = True,
#     prepare_keys: bool = True,
#     force_recompute_keys: bool = False,
#     **common_kwargs,
# ):
#     df_out = df
#     all_metrics = []

#     if prepare_keys:
#         df_out = prepare_dedupe_keys(
#             df_out,
#             title_col=common_kwargs.get("title_col", "title"),
#             authors_col=common_kwargs.get("authors_col", "authors_flat"),
#             title_clean_col=common_kwargs.get("title_clean_col", "title_clean_v2"),
#             force_recompute=force_recompute_keys,
#             print_progress=print_summary,
#         )

#     hierarchy_col = common_kwargs.get("hierarchy_col", "records_hierarchy")

#     if hierarchy_col not in df_out.columns:
#         df_out[hierarchy_col] = pd.NA

#     if print_summary:
#         print("\n=== PASS A: EXACT ===")

#     df_out, mA = run_dedupe_stages_fast(
#         df_out,
#         stages=stages_exact,
#         early_stop_if_new_labels_lt=early_stop_if_new_labels_lt,
#         print_summary=print_summary,
#         return_all_metrics=True,
#         **common_kwargs,
#     )

#     all_metrics.extend(mA)

#     if print_summary:
#         print("\n=== PASS B: FUZZY ===")

#     df_out, mB = run_dedupe_stages_fast(
#         df_out,
#         stages=stages_fuzzy,
#         early_stop_if_new_labels_lt=early_stop_if_new_labels_lt,
#         print_summary=print_summary,
#         return_all_metrics=True,
#         **common_kwargs,
#     )

#     all_metrics.extend(mB)

#     return (df_out, all_metrics) if return_all_metrics else df_out


# # ============================================================
# # 6) Recommended faster stages
# # ============================================================

# STAGES_EXACT_FAST = [
#     dict(
#         name="A1_tokenbag_exact",
#         authors_fp_col="authors_fp_tokenbag",
#         prefilter_mode="title_dup",
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=True,
#         min_authors_required=1,
#         min_shared_authors=1,
#         overwrite_mode="any",   # important for first stage
#     ),
#     dict(
#         name="A2_last_initial_exact",
#         authors_fp_col="authors_fp_last_initial",
#         prefilter_mode="title_dup",
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=True,
#         min_authors_required=2,
#         min_shared_authors=2,
#         overwrite_mode="parent_only",
#     ),
#     dict(
#         name="A3_last_exact_strict",
#         authors_fp_col="authors_fp_last",
#         prefilter_mode="title_dup",
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=False,
#         overwrite_mode="parent_only",
#     ),
# ]

# STAGES_FUZZY_FAST = [
#     dict(
#         name="B1_tokenbag_fuzzy",
#         authors_fp_col="authors_fp_tokenbag",
#         prefilter_mode="author_dup",
#         title_fuzzy_fallback=True,
#         min_title_tokens=8,
#         min_title_containment=0.80,
#         relaxed_shared_authors=False,
#         overwrite_mode="parent_only",
#     ),
#     dict(
#         name="B2_last_initial_fuzzy",
#         authors_fp_col="authors_fp_last_initial",
#         prefilter_mode="author_dup",
#         title_fuzzy_fallback=True,
#         min_title_tokens=8,
#         min_title_containment=0.90,
#         relaxed_shared_authors=False,
#         overwrite_mode="parent_only",
#     ),
# ]


# # ============================================================
# # 7) Usage
# # ============================================================

# # df_out, metrics = run_dedupe_pipeline_two_passes_fast(
# #     df,
# #     stages_exact=STAGES_EXACT_FAST,
# #     stages_fuzzy=STAGES_FUZZY_FAST,
# #     early_stop_if_new_labels_lt=500,
# #     print_summary=True,
# #     return_all_metrics=True,

# #     servers=None,
# #     across_servers=True,
# #     use_year=False,
# #     choose_parent="oldest",
# #     prefilter=True,

# #     server_col="server_name",
# #     record_id_col="record_id",
# #     title_col="title",
# #     authors_col="authors_flat",
# #     year_col="publication_year_first_seen",
# #     date_candidates=("date_first_seen",),

# #     hierarchy_col="records_hierarchy",
# #     parent_id_col="parent_record_id",
# #     group_id_col="dup_group_id",

# #     title_clean_col="title_clean_v2",
# # )

# # metrics_df = pd.DataFrame(metrics)
# # print(metrics_df)

# # print(df_out["records_hierarchy"].value_counts(dropna=False).head(30))

In [45]:
# df_out, metrics = run_dedupe_pipeline_two_passes_fast(
#     data_clean_hierarchy,
#     stages_exact=STAGES_EXACT_FAST,
#     stages_fuzzy=STAGES_FUZZY_FAST,
#     early_stop_if_new_labels_lt=500,
#     print_summary=True,
#     return_all_metrics=True,

#     servers=None,
#     across_servers=True,
#     use_year=False,
#     choose_parent="oldest",
#     prefilter=True,

#     server_col="server_name",
#     record_id_col="record_id",
#     title_col="title",
#     authors_col="authors_flat",
#     year_col="publication_year_first_seen",
#     date_candidates=("date_first_seen",),

#     hierarchy_col="records_hierarchy",
#     parent_id_col="parent_record_id",
#     group_id_col="dup_group_id",

#     title_clean_col="title_clean_v2",
# )

# metrics_df = pd.DataFrame(metrics)
# print(metrics_df)

# print(df_out["records_hierarchy"].value_counts(dropna=False).head(30))

In [46]:
# import pandas as pd
# import numpy as np
# import re
# import time
# import unicodedata
# from typing import Iterable, Optional, Dict, Any, Tuple, List


# # ============================================================
# # 0) Regex + constants
# # ============================================================

# _WS = re.compile(r"\s+")
# _PUNCT_ALL = re.compile(r"[^\w\s]", re.UNICODE)

# NA_LIKE = {"", "none", "null", "nan", "n/a", "[]", "{}", "na"}

# GENERIC_TITLES = frozenset({
#     "front matter", "back matter", "summary", "summaries",
#     "reviews in brief", "publications received",
#     "agricultural letter",
#     "administrasi kurikulum", "administrasi peserta didik",
#     "experiment ended", "editorial", "introduction", "preface",
#     "contents", "table of contents", "index", "book review",
#     "letter", "news", "announcement", "abstract", "poster",
#     "supplement", "no title", "resumes", "retracted",
#     "bibliographie", "bremsstrahlung",
# })

# MIN_TITLE_TOKENS = 3
# MIN_TITLE_CHARS = 25


# # ============================================================
# # 1) Helpers
# # ============================================================

# def _strip_accents_text(x: str) -> str:
#     return "".join(
#         c for c in unicodedata.normalize("NFKD", str(x))
#         if not unicodedata.combining(c)
#     )


# def _record_id_key(s: pd.Series) -> pd.Series:
#     digits = s.astype("string").str.extract(r"(\d+)")[0]
#     return pd.to_numeric(digits, errors="coerce")


# def _pick_first_existing(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
#     for c in candidates:
#         if c in df.columns:
#             return c
#     return None


# def _title_tokens_from_clean(title_clean: str) -> List[str]:
#     if not title_clean:
#         return []
#     return [t for t in str(title_clean).split() if len(t) >= 2]


# def _containment_score(a_tokens: List[str], b_tokens: List[str]) -> float:
#     if not a_tokens or not b_tokens:
#         return 0.0

#     A, B = set(a_tokens), set(b_tokens)
#     denom = min(len(A), len(B))

#     if denom == 0:
#         return 0.0

#     return len(A & B) / denom


# def _author_tokens_from_fp(fp: str) -> List[str]:
#     if not fp:
#         return []
#     return [t for t in str(fp).split(";") if t]


# def _overlap_count(a_tokens: List[str], b_tokens: List[str]) -> int:
#     if not a_tokens or not b_tokens:
#         return 0
#     return len(set(a_tokens) & set(b_tokens))


# # ============================================================
# # 2) Fast title cleaning
# # ============================================================

# def clean_title_series_fast(s: pd.Series) -> pd.Series:
#     """
#     Faster title normalization:
#     - lowercase
#     - remove accents
#     - remove punctuation
#     - collapse spaces
#     - remove risky/generic/too-short titles
#     """

#     titles = (
#         s.astype("string")
#         .fillna("")
#         .str.strip()
#         .str.lower()
#     )

#     titles = titles.where(~titles.isin(NA_LIKE), "")

#     # Accent stripping still needs apply, but only once
#     titles = titles.apply(_strip_accents_text)

#     titles = (
#         titles
#         .str.replace(_PUNCT_ALL, " ", regex=True)
#         .str.replace(_WS, " ", regex=True)
#         .str.strip()
#     )

#     # Vectorized risky-title filtering
#     token_counts = titles.str.count(r"\S+")
#     char_counts = titles.str.len()

#     risky = (
#         titles.eq("")
#         | titles.isin(GENERIC_TITLES)
#         | (token_counts < MIN_TITLE_TOKENS)
#         | (char_counts < MIN_TITLE_CHARS)
#     )

#     titles = titles.mask(risky, "")

#     return titles


# # ============================================================
# # 3) Author normalization
# # ============================================================

# def _normalize_one_author_tokenbag(author: str) -> str:
#     if not author:
#         return ""

#     a = _strip_accents_text(author).lower().strip()

#     if not a or a in NA_LIKE:
#         return ""

#     a = _PUNCT_ALL.sub(" ", a)
#     a = _WS.sub(" ", a).strip()

#     toks = [t for t in a.split() if t]

#     if not toks:
#         return ""

#     return "_".join(sorted(toks))


# def _normalize_one_author_last_initial(author: str) -> str:
#     if not author:
#         return ""

#     a = _strip_accents_text(author).lower().strip()

#     if not a or a in NA_LIKE:
#         return ""

#     if "," in a:
#         left, right = a.split(",", 1)

#         left = _PUNCT_ALL.sub(" ", left)
#         right = _PUNCT_ALL.sub(" ", right)

#         left = _WS.sub(" ", left).strip()
#         right = _WS.sub(" ", right).strip()

#         last_toks = [t for t in left.split() if t]
#         first_toks = [t for t in right.split() if t]

#         if not last_toks or not first_toks:
#             return ""

#         last = last_toks[0]
#         ini = first_toks[0][0]

#         return f"{last}|{ini}"

#     a = _PUNCT_ALL.sub(" ", a)
#     a = _WS.sub(" ", a).strip()

#     toks = [t for t in a.split() if t]

#     if len(toks) < 2:
#         return ""

#     ini = toks[0][0]
#     last = toks[-1]

#     return f"{last}|{ini}"


# def _normalize_one_author_last(author: str) -> str:
#     if not author:
#         return ""

#     a = _strip_accents_text(author).lower().strip()

#     if not a or a in NA_LIKE:
#         return ""

#     if "," in a:
#         left = a.split(",", 1)[0]
#         left = _PUNCT_ALL.sub(" ", left)
#         left = _WS.sub(" ", left).strip()
#         toks = [t for t in left.split() if t]
#         return toks[0] if toks else ""

#     a = _PUNCT_ALL.sub(" ", a)
#     a = _WS.sub(" ", a).strip()

#     toks = [t for t in a.split() if t]

#     return toks[-1] if toks else ""


# def build_authors_fingerprint_series(authors_flat: pd.Series, mode: str) -> pd.Series:
#     """
#     Build author fingerprint per row.
#     """

#     if mode not in {"tokenbag", "last_initial", "last"}:
#         raise ValueError("mode must be tokenbag | last_initial | last")

#     if mode == "tokenbag":
#         norm_fn = _normalize_one_author_tokenbag
#     elif mode == "last_initial":
#         norm_fn = _normalize_one_author_last_initial
#     else:
#         norm_fn = _normalize_one_author_last

#     def row_to_fp(x) -> str:
#         if pd.isna(x):
#             return ""

#         x = str(x).strip()

#         if not x or x.lower() in NA_LIKE:
#             return ""

#         authors = [a.strip() for a in x.split(";") if a.strip()]
#         norm = [norm_fn(a) for a in authors]
#         norm = sorted(set(z for z in norm if z))

#         return ";".join(norm)

#     return authors_flat.apply(row_to_fp)


# # ============================================================
# # 4) Precompute expensive keys once
# # ============================================================

# def prepare_dedupe_keys_fast(
#     df: pd.DataFrame,
#     *,
#     title_col: str = "title",
#     authors_col: str = "authors_flat",
#     title_clean_col: str = "title_clean_v2",
#     tokenbag_col: str = "authors_fp_tokenbag",
#     last_initial_col: str = "authors_fp_last_initial",
#     last_col: str = "authors_fp_last",
#     force_recompute: bool = False,
#     print_progress: bool = True,
# ) -> pd.DataFrame:

#     t0 = time.perf_counter()

#     if print_progress:
#         print("Preparing dedupe keys...")

#     if force_recompute or title_clean_col not in df.columns:
#         if print_progress:
#             print("  - Cleaning titles...")

#         df[title_clean_col] = clean_title_series_fast(df[title_col])

#     if force_recompute or tokenbag_col not in df.columns:
#         if print_progress:
#             print("  - Building tokenbag author fingerprints...")

#         df[tokenbag_col] = build_authors_fingerprint_series(df[authors_col], "tokenbag")

#     if force_recompute or last_initial_col not in df.columns:
#         if print_progress:
#             print("  - Building last_initial author fingerprints...")

#         df[last_initial_col] = build_authors_fingerprint_series(df[authors_col], "last_initial")

#     if force_recompute or last_col not in df.columns:
#         if print_progress:
#             print("  - Building last-name author fingerprints...")

#         df[last_col] = build_authors_fingerprint_series(df[authors_col], "last")

#     if print_progress:
#         print(f"Keys prepared in {time.perf_counter() - t0:.2f}s")

#     return df


# # ============================================================
# # 5) One dedupe stage using cached keys
# # ============================================================

# def dedupe_title_authors_stage_fast(
#     df: pd.DataFrame,
#     *,
#     stage_name: str,
#     authors_fp_col: str,

#     title_col: str = "title",
#     authors_col: str = "authors_flat",

#     title_fuzzy_fallback: bool = False,
#     min_title_tokens: int = 6,
#     min_title_containment: float = 0.80,

#     relaxed_shared_authors: bool = True,
#     min_authors_required: int = 2,
#     min_shared_authors: int = 2,
#     max_relaxed_title_group_size: Optional[int] = None,

#     prefilter_mode: str = "title_dup",
#     prefilter: bool = True,

#     servers=None,
#     across_servers: bool = True,
#     use_year: bool = False,
#     choose_parent: str = "oldest",
#     overwrite_mode: str = "parent_only",

#     server_col: str = "server_name",
#     record_id_col: str = "record_id",
#     year_col: str = "publication_year_first_seen",
#     date_candidates: Tuple[str, ...] = ("date_first_seen",),

#     title_clean_col: str = "title_clean_v2",
#     hierarchy_col: str = "records_hierarchy",
#     parent_id_col: str = "parent_record_id",
#     group_id_col: str = "dup_group_id",

#     return_metrics: bool = False,
# ):

#     t0 = time.perf_counter()

#     metrics = {
#         "stage_name": stage_name,
#         "n_rows_df": int(len(df)),
#         "n_candidates_initial": 0,
#         "prefilter_rows": 0,
#         "prefilter_groups": 0,
#         "work_rows_after_keys": 0,
#         "stageA_groups": 0,
#         "stageA_children_labeled": 0,
#         "fuzzy_groups": 0,
#         "fuzzy_pairs_checked": 0,
#         "fuzzy_children_labeled": 0,
#         "stageB_title_groups": 0,
#         "stageB_clusters": 0,
#         "stageB_children_labeled": 0,
#         "stageB_groups_skipped_too_large": 0,
#         "time_s": 0.0,
#     }

#     for c in [hierarchy_col, parent_id_col, group_id_col]:
#         if c not in df.columns:
#             df[c] = pd.NA

#     if title_clean_col not in df.columns:
#         raise ValueError(f"Missing {title_clean_col}. Run prepare_dedupe_keys_fast() first.")

#     if authors_fp_col not in df.columns:
#         raise ValueError(f"Missing {authors_fp_col}. Run prepare_dedupe_keys_fast() first.")

#     h = df[hierarchy_col]

#     if overwrite_mode == "any":
#         eligible = pd.Series(True, index=df.index)
#     elif overwrite_mode == "parent_only":
#         eligible = h.astype("string").str.lower().str.strip().eq("parent")
#     elif overwrite_mode == "unlabeled_only":
#         eligible = h.isna()
#     else:
#         raise ValueError("overwrite_mode must be any | parent_only | unlabeled_only")

#     if servers is None:
#         server_mask = pd.Series(True, index=df.index)
#     elif isinstance(servers, str):
#         server_mask = df[server_col].eq(servers)
#     else:
#         server_mask = df[server_col].isin(list(servers))

#     m = eligible & server_mask
#     metrics["n_candidates_initial"] = int(m.sum())

#     if not m.any():
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     # -------------------------------
#     # Prefilter
#     # -------------------------------

#     if prefilter_mode == "title_dup":
#         key = df.loc[m, title_clean_col].astype("string").fillna("")
#     elif prefilter_mode == "author_dup":
#         key = df.loc[m, authors_fp_col].astype("string").fillna("")
#     elif prefilter_mode == "none":
#         keep_idx = df.index[m]
#         key = None
#     else:
#         raise ValueError("prefilter_mode must be title_dup | author_dup | none")

#     if prefilter_mode != "none":
#         key = key[key != ""]
#         vc = key.value_counts()
#         dup_values = vc[vc >= 2].index
#         keep_idx = key[key.isin(dup_values)].index
#         metrics["prefilter_groups"] = int(len(dup_values))

#     metrics["prefilter_rows"] = int(len(keep_idx))

#     if len(keep_idx) == 0:
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     date_col = _pick_first_existing(df, date_candidates)

#     cols = [server_col, record_id_col, title_clean_col, authors_fp_col]

#     if use_year:
#         cols.append(year_col)

#     if date_col:
#         cols.append(date_col)

#     work = df.loc[keep_idx, cols].copy()

#     work["_t"] = work[title_clean_col].astype("string").fillna("")
#     work["_a_fp"] = work[authors_fp_col].astype("string").fillna("")

#     if use_year:
#         y = pd.to_numeric(work[year_col], errors="coerce")
#         y = y.where((y >= 1000) & (y <= 3000)).round().astype("Int64")
#         work["_y"] = y
#         work = work.loc[
#             (work["_t"] != "") &
#             (work["_a_fp"] != "") &
#             (work["_y"].notna())
#         ]
#     else:
#         work["_y"] = pd.NA
#         work = work.loc[
#             (work["_t"] != "") &
#             (work["_a_fp"] != "")
#         ]

#     metrics["work_rows_after_keys"] = int(len(work))

#     if work.empty:
#         metrics["time_s"] = time.perf_counter() - t0
#         return (df, metrics) if return_metrics else df

#     # -------------------------------
#     # Stage A: exact title + exact authors
#     # -------------------------------

#     if use_year:
#         strict_base = work["_t"] + "||" + work["_a_fp"] + "||" + work["_y"].astype("string")
#     else:
#         strict_base = work["_t"] + "||" + work["_a_fp"]

#     if across_servers:
#         work["_grp_strict"] = strict_base
#     else:
#         work["_grp_strict"] = work[server_col].astype("string") + "||" + strict_base

#     strict = work

#     if prefilter:
#         vcg = strict["_grp_strict"].value_counts()
#         dup_keys = vcg[vcg >= 2].index
#         strict = strict.loc[strict["_grp_strict"].isin(dup_keys)]

#     metrics["stageA_groups"] = int(strict["_grp_strict"].nunique()) if not strict.empty else 0

#     if not strict.empty:
#         if date_col and date_col in strict.columns:
#             strict = strict.copy()
#             strict["_dt"] = pd.to_datetime(strict[date_col], errors="coerce")
#         else:
#             strict = strict.copy()
#             strict["_dt"] = pd.NaT

#         strict["_rid"] = _record_id_key(strict[record_id_col])

#         if choose_parent == "oldest":
#             strict = strict.sort_values(
#                 by=["_grp_strict", "_dt", "_rid"],
#                 ascending=[True, True, True],
#                 na_position="last",
#             )
#         elif choose_parent == "most_recent":
#             strict = strict.sort_values(
#                 by=["_grp_strict", "_dt", "_rid"],
#                 ascending=[True, False, False],
#                 na_position="last",
#             )
#         else:
#             raise ValueError("choose_parent must be oldest | most_recent")

#         parents = strict.groupby("_grp_strict", sort=False).head(1)

#         parent_rid_map = parents.set_index("_grp_strict")[record_id_col]
#         parent_srv_map = parents.set_index("_grp_strict")[server_col]

#         strict["_parent_rid"] = strict["_grp_strict"].map(parent_rid_map)
#         strict["_parent_srv"] = strict["_grp_strict"].map(parent_srv_map)

#         is_parent = strict[record_id_col].eq(strict["_parent_rid"])

#         parent_idx = strict.index[is_parent]
#         child_idx = strict.index[~is_parent]

#         metrics["stageA_children_labeled"] = int(len(child_idx))

#         df.loc[parent_idx, hierarchy_col] = "parent"
#         df.loc[parent_idx, parent_id_col] = pd.NA

#         df.loc[child_idx, hierarchy_col] = (
#             "parent - duplicate (" +
#             strict.loc[child_idx, "_parent_srv"].astype("string") +
#             ")"
#         )

#         df.loc[child_idx, parent_id_col] = strict.loc[child_idx, "_parent_rid"].values

#         df.loc[strict.index, group_id_col] = (
#             pd.util.hash_pandas_object(strict["_grp_strict"], index=False)
#             .astype("uint64")
#             .astype(str)
#             .values
#         )

#     # -------------------------------
#     # Fuzzy fallback
#     # -------------------------------

#     if title_fuzzy_fallback:
#         h2 = df[hierarchy_col]

#         if overwrite_mode == "parent_only":
#             eligible2 = h2.astype("string").str.lower().str.strip().eq("parent")
#         elif overwrite_mode == "unlabeled_only":
#             eligible2 = h2.isna()
#         else:
#             eligible2 = pd.Series(True, index=df.index)

#         remain_idx = work.index.intersection(df.index[eligible2])
#         wF = work.loc[remain_idx]

#         if not wF.empty:
#             wF = wF.copy()

#             if use_year:
#                 wF["_grp_auth"] = wF["_a_fp"] + "||" + wF["_y"].astype("string")
#             else:
#                 wF["_grp_auth"] = wF["_a_fp"]

#             vc_auth = wF["_grp_auth"].value_counts()
#             keep_auth = vc_auth[vc_auth >= 2].index
#             wF = wF.loc[wF["_grp_auth"].isin(keep_auth)]

#             metrics["fuzzy_groups"] = int(wF["_grp_auth"].nunique()) if not wF.empty else 0

#             if not wF.empty:
#                 wF = wF.copy()

#                 if date_col and date_col in wF.columns:
#                     wF["_dt"] = pd.to_datetime(wF[date_col], errors="coerce")
#                 else:
#                     wF["_dt"] = pd.NaT

#                 wF["_rid"] = _record_id_key(wF[record_id_col])

#                 tokens_map = {
#                     idx: _title_tokens_from_clean(wF.at[idx, "_t"])
#                     for idx in wF.index
#                 }

#                 for grp, g in wF.groupby("_grp_auth", sort=False):
#                     idxs = [
#                         idx for idx in g.index
#                         if len(tokens_map.get(idx, [])) >= min_title_tokens
#                     ]

#                     if len(idxs) < 2:
#                         continue

#                     gg = g.loc[idxs].copy()

#                     if choose_parent == "oldest":
#                         gg = gg.sort_values(
#                             by=["_dt", "_rid"],
#                             ascending=[True, True],
#                             na_position="last",
#                         )
#                     else:
#                         gg = gg.sort_values(
#                             by=["_dt", "_rid"],
#                             ascending=[False, False],
#                             na_position="last",
#                         )

#                     parent_idx = gg.index[0]
#                     parent_tokens = tokens_map[parent_idx]
#                     parent_rid = gg.loc[parent_idx, record_id_col]
#                     parent_srv = gg.loc[parent_idx, server_col]

#                     df.loc[parent_idx, hierarchy_col] = "parent"
#                     df.loc[parent_idx, parent_id_col] = pd.NA

#                     for idx in gg.index[1:]:
#                         metrics["fuzzy_pairs_checked"] += 1

#                         sc = _containment_score(parent_tokens, tokens_map[idx])

#                         if sc >= min_title_containment:
#                             df.loc[idx, hierarchy_col] = f"parent - duplicate ({parent_srv})"
#                             df.loc[idx, parent_id_col] = parent_rid
#                             df.loc[idx, group_id_col] = f"fuzzy::{stage_name}::{grp}"
#                             metrics["fuzzy_children_labeled"] += 1

#     # -------------------------------
#     # Stage B: relaxed same-title shared-author overlap
#     # -------------------------------

#     if relaxed_shared_authors:
#         h3 = df[hierarchy_col]

#         if overwrite_mode == "parent_only":
#             eligible3 = h3.astype("string").str.lower().str.strip().eq("parent")
#         elif overwrite_mode == "unlabeled_only":
#             eligible3 = h3.isna()
#         else:
#             eligible3 = pd.Series(True, index=df.index)

#         remain_idx = work.index.intersection(df.index[eligible3])
#         w2 = work.loc[remain_idx]

#         if not w2.empty:
#             w2 = w2.copy()

#             if use_year:
#                 relaxed_base = w2["_t"] + "||" + w2["_y"].astype("string")
#             else:
#                 relaxed_base = w2["_t"]

#             if across_servers:
#                 w2["_grp_title"] = relaxed_base
#             else:
#                 w2["_grp_title"] = w2[server_col].astype("string") + "||" + relaxed_base

#             vc2 = w2["_grp_title"].value_counts()
#             keep_groups = vc2[vc2 >= 2].index
#             w2 = w2.loc[w2["_grp_title"].isin(keep_groups)]

#             metrics["stageB_title_groups"] = int(w2["_grp_title"].nunique()) if not w2.empty else 0

#             if not w2.empty:
#                 w2 = w2.copy()

#                 w2["_a_tokens"] = w2["_a_fp"].apply(_author_tokens_from_fp)
#                 w2["_a_n"] = w2["_a_tokens"].apply(len)

#                 if date_col and date_col in w2.columns:
#                     w2["_dt"] = pd.to_datetime(w2[date_col], errors="coerce")
#                 else:
#                     w2["_dt"] = pd.NaT

#                 w2["_rid"] = _record_id_key(w2[record_id_col])

#                 group_counter = 0
#                 children_total = 0

#                 for grp, g in w2.groupby("_grp_title", sort=False):
#                     if (
#                         max_relaxed_title_group_size is not None
#                         and len(g) > max_relaxed_title_group_size
#                     ):
#                         metrics["stageB_groups_skipped_too_large"] += 1
#                         continue
    
#                     g = g.loc[g["_a_n"] >= min_authors_required]

#                     if len(g) < 2:
#                         continue

#                     idxs = g.index.tolist()
#                     used = set()
#                     clusters = []

#                     for i in idxs:
#                         if i in used:
#                             continue

#                         used.add(i)
#                         cluster = [i]

#                         for j in idxs:
#                             if j in used:
#                                 continue

#                             if _overlap_count(
#                                 g.at[i, "_a_tokens"],
#                                 g.at[j, "_a_tokens"]
#                             ) >= min_shared_authors:
#                                 used.add(j)
#                                 cluster.append(j)

#                         if len(cluster) >= 2:
#                             clusters.append(cluster)

#                     for cluster in clusters:
#                         group_counter += 1
#                         cldf = g.loc[cluster].copy()

#                         if choose_parent == "oldest":
#                             cldf = cldf.sort_values(
#                                 by=["_dt", "_rid"],
#                                 ascending=[True, True],
#                                 na_position="last",
#                             )
#                         else:
#                             cldf = cldf.sort_values(
#                                 by=["_dt", "_rid"],
#                                 ascending=[False, False],
#                                 na_position="last",
#                             )

#                         parent_idx = cldf.index[0]
#                         parent_rid = cldf.loc[parent_idx, record_id_col]
#                         parent_srv = cldf.loc[parent_idx, server_col]

#                         child_idxs = [x for x in cldf.index if x != parent_idx]

#                         df.loc[parent_idx, hierarchy_col] = "parent"
#                         df.loc[parent_idx, parent_id_col] = pd.NA
#                         df.loc[parent_idx, group_id_col] = f"relaxed::{stage_name}::{group_counter}"

#                         df.loc[child_idxs, hierarchy_col] = f"parent - duplicate ({parent_srv})"
#                         df.loc[child_idxs, parent_id_col] = parent_rid
#                         df.loc[child_idxs, group_id_col] = f"relaxed::{stage_name}::{group_counter}"

#                         children_total += len(child_idxs)

#                 metrics["stageB_clusters"] = int(group_counter)
#                 metrics["stageB_children_labeled"] = int(children_total)

#     metrics["time_s"] = time.perf_counter() - t0

#     return (df, metrics) if return_metrics else df


# # ============================================================
# # 6) Runner
# # ============================================================

# def _count_children_labels(series: pd.Series) -> int:
#     s = series.astype("string").fillna("")
#     return int(s.str.startswith("parent - duplicate").sum())


# def run_dedupe_stages_fast(
#     df: pd.DataFrame,
#     *,
#     stages: List[Dict[str, Any]],
#     early_stop_if_new_labels_lt: int = 100,
#     print_summary: bool = True,
#     return_all_metrics: bool = True,
#     **common_kwargs,
# ):

#     df_out = df
#     metrics_all = []

#     hierarchy_col = common_kwargs.get("hierarchy_col", "records_hierarchy")

#     if hierarchy_col not in df_out.columns:
#         df_out[hierarchy_col] = pd.NA

#     prev_children = _count_children_labels(df_out[hierarchy_col])

#     for stage in stages:
#         name = stage.get("name", stage.get("stage_name", "stage"))

#         if print_summary:
#             print(f"\nStarting stage: {name}")

#         t0 = time.perf_counter()

#         stage_kwargs = dict(common_kwargs)

#         # Only needed by key preparation
#         stage_kwargs.pop("title_col", None)
#         stage_kwargs.pop("authors_col", None)

#         df_out, m = dedupe_title_authors_stage_fast(
#             df_out,
#             return_metrics=True,
#             stage_name=name,
#             **stage_kwargs,
#             **{k: v for k, v in stage.items() if k not in {"name", "stage_name"}},
#         )

#         now_children = _count_children_labels(df_out[hierarchy_col])
#         delta = now_children - prev_children
#         prev_children = now_children

#         m["new_children_added"] = int(delta)
#         m["stage_runtime_s"] = time.perf_counter() - t0
#         metrics_all.append(m)

#         if print_summary:
#             print(
#                 f"[{name}] new_children={delta:,} | "
#                 f"cand={m['n_candidates_initial']:,} | "
#                 f"prefilter_rows={m['prefilter_rows']:,} | "
#                 f"A={m['stageA_children_labeled']:,} | "
#                 f"fuzzy={m['fuzzy_children_labeled']:,} | "
#                 f"B={m['stageB_children_labeled']:,} | "
#                 f"skipped_large_B={m['stageB_groups_skipped_too_large']:,} | "
#                 f"time={m['stage_runtime_s']:.2f}s"
#             )

#         if delta < early_stop_if_new_labels_lt:
#             if print_summary:
#                 print(
#                     f"Early stop after {name}: "
#                     f"delta {delta:,} < {early_stop_if_new_labels_lt:,}"
#                 )
#             break

#     return (df_out, metrics_all) if return_all_metrics else df_out


# # ============================================================
# # 7) Full two-pass pipeline
# # ============================================================

# def run_dedupe_pipeline_two_passes_fast(
#     df: pd.DataFrame,
#     *,
#     stages_exact: List[Dict[str, Any]],
#     stages_fuzzy: List[Dict[str, Any]],
#     early_stop_if_new_labels_lt: int = 100,
#     print_summary: bool = True,
#     return_all_metrics: bool = True,
#     prepare_keys: bool = True,
#     force_recompute_keys: bool = False,
#     **common_kwargs,
# ):

#     df_out = df
#     all_metrics = []

#     if prepare_keys:
#         df_out = prepare_dedupe_keys_fast(
#             df_out,
#             title_col=common_kwargs.get("title_col", "title"),
#             authors_col=common_kwargs.get("authors_col", "authors_flat"),
#             title_clean_col=common_kwargs.get("title_clean_col", "title_clean_v2"),
#             force_recompute=force_recompute_keys,
#             print_progress=print_summary,
#         )

#     hierarchy_col = common_kwargs.get("hierarchy_col", "records_hierarchy")

#     if hierarchy_col not in df_out.columns:
#         df_out[hierarchy_col] = pd.NA

#     if print_summary:
#         print("\n=== PASS A: EXACT ===")

#     df_out, mA = run_dedupe_stages_fast(
#         df_out,
#         stages=stages_exact,
#         early_stop_if_new_labels_lt=early_stop_if_new_labels_lt,
#         print_summary=print_summary,
#         return_all_metrics=True,
#         **common_kwargs,
#     )

#     all_metrics.extend(mA)

#     if print_summary:
#         print("\n=== PASS B: FUZZY ===")

#     df_out, mB = run_dedupe_stages_fast(
#         df_out,
#         stages=stages_fuzzy,
#         early_stop_if_new_labels_lt=early_stop_if_new_labels_lt,
#         print_summary=print_summary,
#         return_all_metrics=True,
#         **common_kwargs,
#     )

#     all_metrics.extend(mB)

#     return (df_out, all_metrics) if return_all_metrics else df_out


# # ============================================================
# # 8) Recommended stages
# # ============================================================

# STAGES_EXACT_FAST = [
#     dict(
#         name="A1_tokenbag_exact",
#         authors_fp_col="authors_fp_tokenbag",
#         prefilter_mode="title_dup",
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=True,
#         min_authors_required=1,
#         min_shared_authors=1,
#         overwrite_mode="any",
#         max_relaxed_title_group_size=None, #100,
#     ),
#     dict(
#         name="A2_last_initial_exact",
#         authors_fp_col="authors_fp_last_initial",
#         prefilter_mode="title_dup",
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=True,
#         min_authors_required=2,
#         min_shared_authors=2,
#         overwrite_mode="parent_only",
#         max_relaxed_title_group_size=None, #100,
#     ),
#     dict(
#         name="A3_last_exact_strict",
#         authors_fp_col="authors_fp_last",
#         prefilter_mode="title_dup",
#         title_fuzzy_fallback=False,
#         relaxed_shared_authors=False,
#         overwrite_mode="parent_only",
#     ),
# ]

# STAGES_FUZZY_FAST = [
#     dict(
#         name="B1_tokenbag_fuzzy",
#         authors_fp_col="authors_fp_tokenbag",
#         prefilter_mode="author_dup",
#         title_fuzzy_fallback=True,
#         min_title_tokens=8,
#         min_title_containment=0.80,
#         relaxed_shared_authors=False,
#         overwrite_mode="parent_only",
#     ),
#     dict(
#         name="B2_last_initial_fuzzy",
#         authors_fp_col="authors_fp_last_initial",
#         prefilter_mode="author_dup",
#         title_fuzzy_fallback=True,
#         min_title_tokens=8,
#         min_title_containment=0.90,
#         relaxed_shared_authors=False,
#         overwrite_mode="parent_only",
#     ),
# ]


# # ============================================================
# # 9) Usage
# # ============================================================

# df_out, metrics = run_dedupe_pipeline_two_passes_fast(
#     data_clean_hierarchy,
#     stages_exact=STAGES_EXACT_FAST,
#     stages_fuzzy=STAGES_FUZZY_FAST,
#     early_stop_if_new_labels_lt=10,
#     print_summary=True,
#     return_all_metrics=True,

#     servers=None,
#     across_servers=True,
#     use_year=False,
#     choose_parent="oldest",
#     prefilter=True,

#     server_col="server_name",
#     record_id_col="record_id",
#     title_col="title",
#     authors_col="authors_flat",
#     year_col="publication_year_first_seen",
#     date_candidates=("date_first_seen",),

#     hierarchy_col="records_hierarchy",
#     parent_id_col="parent_record_id",
#     group_id_col="dup_group_id",

#     title_clean_col="title_clean_v2",
# )

# metrics_df = pd.DataFrame(metrics)

# print(metrics_df)

# print(df_out["records_hierarchy"].value_counts(dropna=False).head(30))

## Save outputs

In [47]:
from pathlib import Path
import json
import pandas as pd

OUT_DIR = Path("outputs_new")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save metrics
with open(OUT_DIR / "dedupe_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame(metrics).to_csv(OUT_DIR / "dedupe_metrics.csv", index=False)

# Save dataset
data_out.to_parquet(OUT_DIR / "dedupe_data_out.parquet", index=False)

print("✅ All files saved to outputs/")


✅ All files saved to outputs/


In [48]:
from pathlib import Path

OUT_DIR = Path("outputs_new")
OUT_DIR.mkdir(exist_ok=True)

new_cols = [
    "records_hierarchy_backup",
    "records_hierarchy",
    "parent_record_id",
    "dup_group_id",
    "authors_fp_tokenbag",
    "authors_fp_last_initial",
    "authors_fp_last",
    "title_clean_v2",
]
# original_cols = set(df.columns)

# detect new columns
# new_cols = [c for c in data_out.columns if c not in original_cols]

cols_to_save = ["record_id"] + new_cols

data_out[cols_to_save].to_parquet(
    OUT_DIR / "dedupe_data_out_new_cols.parquet",
    index=False
)
data_out[cols_to_save].to_csv(
    OUT_DIR / "dedupe_data_out_new_cols.csv",
    index=False
)
print("✅ Saved:", cols_to_save)



✅ Saved: ['record_id', 'records_hierarchy_backup', 'records_hierarchy', 'parent_record_id', 'dup_group_id', 'authors_fp_tokenbag', 'authors_fp_last_initial', 'authors_fp_last', 'title_clean_v2']


## exploration

In [49]:
metrics

[{'stage_name': 'A1_tokenbag_exact',
  'n_rows_df': 7911901,
  'n_candidates_initial': 7911901,
  'prefilter_mode': 'title_dup',
  'prefilter_rows': 893355,
  'prefilter_groups': 384787,
  'work_rows_after_keys': 845065,
  'stageA_groups': 297159,
  'stageA_children_labeled': 367853,
  'fuzzy_enabled': False,
  'fuzzy_groups': 0,
  'fuzzy_pairs_checked': 0,
  'fuzzy_children_labeled': 0,
  'stageB_enabled': True,
  'stageB_title_groups': 91962,
  'stageB_clusters': 67881,
  'stageB_children_labeled': 69905,
  'time_s': 432.0437816370004,
  'risky_titles_suppressed': 0,
  'stage_runtime_s': 434.90138524400027,
  'new_children_added': 437758},
 {'stage_name': 'A2_last_initial_exact',
  'n_rows_df': 7911901,
  'n_candidates_initial': 7474143,
  'prefilter_mode': 'title_dup',
  'prefilter_rows': 106603,
  'prefilter_groups': 35793,
  'work_rows_after_keys': 56195,
  'stageA_groups': 10634,
  'stageA_children_labeled': 10681,
  'fuzzy_enabled': False,
  'fuzzy_groups': 0,
  'fuzzy_pairs_che

In [50]:
data_out

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,date_first_seen,publication_year,publication_year_first_seen,raw_dates,records_hierarchy_backup,parent_record_id,dup_group_id,authors_fp_tokenbag,title_clean_v2,authors_fp_last_initial,authors_fp_last
0,crossref::10.21467/preprints.48,AIJR Preprints,crossref,10.21467/preprints.48,https://doi.org/10.21467/preprints.48,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,"Bird’s Eye View on the Diagnosis, Treatment, &...","Panchalingala, Sai Bhargavi",None,None,None,None,,,,,false,None,None,None,None,parent,2020-05-03,2020.0,2020,None,parent,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,crossref::10.21467/preprints.43,AIJR Preprints,crossref,10.21467/preprints.43,https://doi.org/10.21467/preprints.43,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Doxycycline and Minocycline Drugs as a Treatme...,"Mostafa, Mohamed",None,None,None,None,,,,,false,None,None,None,None,parent,2020-04-25,2020.0,2020,None,parent,<NA>,<NA>,mohamed_mostafa,doxycycline and minocycline drugs as a treatme...,mostafa|m,<NA>
2,crossref::10.21467/preprints.39,AIJR Preprints,crossref,10.21467/preprints.39,https://doi.org/10.21467/preprints.39,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,A Genetic Perspective of 2019-nCoV in Relation...,"Dasgupta, Rimjhim",None,None,None,None,,,,,false,None,None,None,None,parent,2020-04-16,2020.0,2020,None,parent,<NA>,<NA>,dasgupta_rimjhim,a genetic perspective of 2019 ncov in relation...,dasgupta|r,<NA>
3,crossref::10.21467/preprints.38,AIJR Preprints,crossref,10.21467/preprints.38,https://doi.org/10.21467/preprints.38,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Marine Algae as a Natural Source for Antiviral...,"Musale, Amar S; G., Raja Krishna Kumar; Sapre,...",None,None,None,None,,,,,false,None,None,None,None,parent,2020-04-15,2020.0,2020,None,parent,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,crossref::10.21467/preprints.36,AIJR Preprints,crossref,10.21467/preprints.36,https://doi.org/10.21467/preprints.36,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Possible Prevention of COVID 19 by Using Linol...,"Subhash, Venkata; G, Raja Krishna Kumar; Sapre...",None,None,None,None,,,,,false,None,None,None,None,parent,2020-04-15,2020.0,2020,None,parent,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7961685,openalex::W999325625,viXra,openalex,None,None,https://vixra.org/pdf/1409.0090v1.pdf,preprint,None,Three Objections to Modern Physics,Lubomir Vlcek,None,None,None,None,None,None,None,None,None,None,None,None,None,parent,2014-09-01,2014.0,2014,None,parent,<NA>,<NA>,lubomir_vlcek,three objections to modern physics,vlcek|l,<NA>
7961686,openalex::W999460032,viXra,openalex,None,None,https://vixra.org/abs/1112.0094,preprint,None,Particle Mass Ratios,DT Froedge,None,None,None,None,None,None,None,None,None,None,None,None,None,parent,2011-12-01,2011.0,2011,None,parent,<NA>,<NA>,dt_froedge,particle mass ratios,froedge|d,<NA>
7961687,openalex::W99967155,viXra,openalex,None,None,https://vixra.org/pdf/1406.0019v1.pdf,preprint,None,Quantum FFF Theory Proposals for Some Unsolved...,Leo Vuyk,None,None,None,None,None,None,None,None,None,None,None,None,None,parent,2014-06-01,2014.0,2014,None,parent,<NA>,<NA>,leo_vuyk,quantum fff theory proposals for some unsolved...,vuyk|l,<NA>
7961688,openalex::W999790414,viXra,openalex,None,None,https://vixra.org/pdf/1306.0105v3.pdf,preprint,None,Investigation of the Formalism of Particle Dyn...,Chi-Yi Chen,None,None,None,None,None,None,

In [51]:
print(data_out["records_hierarchy_backup"].value_counts(dropna=False).head(60))

records_hierarchy_backup
parent                              7771465
version                              102316
publish_version                       12888
mirror (AgEcon Search)                 6607
mirror (arXiv)                         6419
part_of                                5921
NaN                                    2464
child                                  2061
mirror (ResearchGate)                   827
correction                              354
comment                                 242
mirror (Zenodo)                         191
mirror (bioRxiv)                         29
review                                   27
mirror (SSRN)                            23
mirror (Open Science Framework)          21
mirror (Humanities Commons CORE)         16
others                                   12
parent_duplicate                          4
mirror (AfricArXiv)                       2
mirror (Research Square)                  2
mirror (EarthArXiv)                       2
mirror 

In [52]:
print(data_out["parent_record_id"].value_counts(dropna=False).head(60))

parent_record_id
<NA>                                           7350175
datacite::10.5281/zenodo.15609432                  569
datacite::10.5281/zenodo.15832876                  285
datacite::10.5281/zenodo.8111116                   180
datacite::10.5281/zenodo.15631517                  166
datacite::10.17613/m69p2w550                       163
datacite::10.5281/zenodo.15690627                  158
datacite::10.5281/zenodo.17172763                  127
datacite::10.5281/zenodo.16684574                  118
datacite::10.5281/zenodo.17373285                  113
datacite::10.13140/rg.2.2.20767.61601/1            106
crossref::10.31237/osf.io/yex9a                     99
crossref::10.31237/osf.io/f2zda                     95
datacite::10.5281/zenodo.14599183                   95
datacite::10.5281/zenodo.14599184                   80
datacite::10.5281/zenodo.8140963                    78
datacite::10.5281/zenodo.16411731                   76
datacite::10.5281/zenodo.15882045               

In [53]:
pattern = "fuzzy::las"

mask = data_out['dup_group_id'].str.contains(pattern, regex=False, na=False)
result = data_out[mask]
print(len(result))
result

9648


,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,date_first_seen,publication_year,publication_year_first_seen,raw_dates,records_hierarchy_backup,parent_record_id,dup_group_id,authors_fp_tokenbag,title_clean_v2,authors_fp_last_initial,authors_fp_last
242,crossref::10.12688/healthopenres.13821.2,AMRC Open Research,crossref,10.12688/healthopenres.13821.2,https://doi.org/10.12688/healthopenres.13821.2,https://healthopenresearch.org/articles/7-10/v2,journal-article,None,Mortality and Predictors of Mortality among Br...,"Tola, Diriba; Solbana, Lencho; Mosisa, Wakgari...",None,None,"{""has-version"": [{""asserted-by"": ""subject"", ""i...",New version,10.12688/healthopenres.13821.1,,,,false,None,None,"[{""DOI"": ""10.12688/healthopenres.13821.1"", ""la...",None,parent - duplicate (AMRC Open Research),2025-08-26,2025.0,2025,None,version,crossref::10.12688/healthopenres.13821.1,fuzzy::last_initial::geneti|d;ilala|b;mosisa|w...,<NA>,mortality and predictors of mortality among br...,geneti|d;ilala|b;mosisa|w;solbana|l;tesfaye|a;...,<NA>
1273,crossref::10.33774/apsa-2024-xzkc5,APSA Preprints,crossref,10.33774/apsa-2024-xzkc5,https://doi.org/10.33774/apsa-2024-xzkc5,https://preprints.apsanet.org/engage/apsa/arti...,posted-content,preprint,The Impact of Job Growth and Inflation on Pres...,"Doti, James; Campbell, Tom",Chapman University,None,None,None,,,,,false,None,None,None,None,parent - duplicate (SSRN),2024-05-29,2024.0,2024,None,parent,crossref::10.2139/ssrn.4657592,fuzzy::last_initial::campbell|t;doti|j,campbell_tom;doti_james,the impact of job growth and inflation on pres...,campbell|t;doti|j,<NA>
1758,crossref::10.3897/arphapreprints.e86933,ARPHA Preprints,crossref,10.3897/arphapreprints.e86933,https://doi.org/10.3897/arphapreprints.e86933,https://preprints.arphahub.com/article/86933/,posted-content,preprint,Evidence of plant-soil feedback in South Texas...,"Bowman, Elizabeth; Plowes, Robert; Gilbert, La...",None,None,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.3897/neobiota.81.86672,,,true,None,None,None,None,parent - duplicate (Research Square),2022-06-01,2022.0,2022,None,parent,crossref::10.21203/rs.3.rs-668160/v1,fuzzy::last_initial::bowman|e;gilbert|l;plowes|r,<NA>,evidence of plant soil feedback in south texas...,bowman|e;gilbert|l;plowes|r,<NA>
1813,crossref::10.3897/arphapreprints.e61912,ARPHA Preprints,crossref,10.3897/arphapreprints.e61912,https://doi.org/10.3897/arphapreprints.e61912,https://preprints.arphahub.com/article/61912/,posted-content,preprint,EcoBank: A flexible database platform for shar...,"Kim, Hyun Woo; Yoon, Sungsoo; Kim, Mokyoung; S...",None,None,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.3897/bdj.9.e61866,,,true,None,None,None,None,parent - duplicate (Authorea Inc.),2020-12-10,2020.0,2020,None,parent,crossref::10.22541/au.160490531.17626170/v1,fuzzy::last_initial::kim|h;kim|k;kim|m;shin|m;...,<NA>,ecobank a flexible database platform for shari...,kim|h;kim|k;kim|m;shin|m;yoon|h;yoon|s,<NA>
2207,crossref::10.3897/arphapreprints.e68669,ARPHA Preprints,crossref,10.3897/arphapreprints.e68669,https://doi.org/10.3897/arphapreprints.e68669,https://preprints.arphahub.com/article/68669/,posted-content,preprint,"Phrynarachne birudis&amp;nbsp;sp. nov., a new ...","Im, Jae Seong; Kim, Seung Tae; Lee, Sue Yeon",None,None,None,None,,,,,false,None,None,None,None,parent - duplicate (ARPHA Preprints),2021-05-20,2021.0,2021,None,parent,crossref::10.3897/arphapreprints.e67978,fuzzy::last_initial::im|j;kim|s;lee|s,<NA>,phrynarachne birudis amp nbsp sp nov a new cra...,im|j;kim|s;lee|s,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [54]:
# result['relations_json'][8401396]

In [55]:
data_out[data_out["record_id"]=='crossref::10.26434/chemrxiv-2021-cj17s']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,date_first_seen,publication_year,publication_year_first_seen,raw_dates,records_hierarchy_backup,parent_record_id,dup_group_id,authors_fp_tokenbag,title_clean_v2,authors_fp_last_initial,authors_fp_last
285204,crossref::10.26434/chemrxiv-2021-cj17s,ChemRxiv,crossref,10.26434/chemrxiv-2021-cj17s,https://doi.org/10.26434/chemrxiv-2021-cj17s,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,First-Principles Plane-Wave-Based Exploration ...,"Ertural, Christina; Stoffel, Ralf; Müller, Pet...",RWTH Aachen University,None,None,None,,,,,false,None,None,None,None,parent,2021-07-26,2021.0,2021,None,parent,<NA>,<NA>,<NA>,first principles plane wave based exploration ...,dronskowski|r;ertural|c;muller|p;stoffel|r;vogt|c,<NA>


In [56]:
data_out[data_out["parent_record_id"]=='crossref::10.26434/chemrxiv-2022-fd190']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,date_first_seen,publication_year,publication_year_first_seen,raw_dates,records_hierarchy_backup,parent_record_id,dup_group_id,authors_fp_tokenbag,title_clean_v2,authors_fp_last_initial,authors_fp_last
286219,crossref::10.26434/chemrxiv-2022-fd190-v2,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v2,https://doi.org/10.26434/chemrxiv-2022-fd190-v2,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2022-04-25,2022.0,2022,None,parent,crossref::10.26434/chemrxiv-2022-fd190,relaxed::A1_tokenbag_exact::8698,alexander_bagger;alonso_hernandez_rosas;christ...,selectivity and intrinsic activity of function...,<NA>,<NA>
286274,crossref::10.26434/chemrxiv-2022-fd190-v3,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v3,https://doi.org/10.26434/chemrxiv-2022-fd190-v3,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190;10.26434/chemrxiv...,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2022-05-02,2022.0,2022,None,parent,crossref::10.26434/chemrxiv-2022-fd190,8105542989974901890,alexander_bagger;christensen_oliver;daasbjerg_...,selectivity and intrinsic activity of function...,<NA>,<NA>
286281,crossref::10.26434/chemrxiv-2022-fd190-v4,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v4,https://doi.org/10.26434/chemrxiv-2022-fd190-v4,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190;10.26434/chemrxiv...,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2022-05-04,2022.0,2022,None,parent,crossref::10.26434/chemrxiv-2022-fd190,8105542989974901890,alexander_bagger;christensen_oliver;daasbjerg_...,selectivity and intrinsic activity of function...,<NA>,<NA>
287198,crossref::10.26434/chemrxiv-2022-fd190-v5,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v5,https://doi.org/10.26434/chemrxiv-2022-fd190-v5,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190;10.26434/chemrxiv...,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2022-08-24,2022.0,2022,None,parent,crossref::10.26434/chemrxiv-2022-fd190,8105542989974901890,alexander_bagger;christensen_oliver;daasbjerg_...,selectivity and intrinsic activity of function...,<NA>,<NA>
313184,crossref::10.26434/chemrxiv-2022-fd190-v6,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v6,https://doi.org/10.26434/chemrxiv-2022-fd190-v6,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Can the CO2 Reduction Reaction be Improved on ...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,10.26434/chemrxiv-2022-f

In [57]:
pattern = "chemrxiv.11846943"


mask = data_out['doi'].str.contains(pattern, regex=False, na=False)
result = data_out[mask]
print(len(result))
result

9


,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,date_first_seen,publication_year,publication_year_first_seen,raw_dates,records_hierarchy_backup,parent_record_id,dup_group_id,authors_fp_tokenbag,title_clean_v2,authors_fp_last_initial,authors_fp_last
280392,crossref::10.26434/chemrxiv.11846943.v1,ChemRxiv,crossref,10.26434/chemrxiv.11846943.v1,https://doi.org/10.26434/chemrxiv.11846943.v1,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Analysis of Whole Genome Sequences and Homolog...,"Shanker, Arun; Bhanu, Divya; Alluri, Anajani",Central Research Institute for Dryland Agricul...,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv.11846943.v2;10.26434/chemrxi...,,,,false,None,None,None,None,parent - duplicate (Open Science Framework),2020-02-13,2020.0,2020,None,parent,crossref::10.31219/osf.io/2zuea,16256016634036771957,alluri_anajani;arun_shanker;bhanu_divya,analysis of whole genome sequences and homolog...,<NA>,<NA>
280428,crossref::10.26434/chemrxiv.11846943.v2,ChemRxiv,crossref,10.26434/chemrxiv.11846943.v2,https://doi.org/10.26434/chemrxiv.11846943.v2,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Analysis of Whole Genome Sequences and Homolog...,"Shanker, Arun; Bhanu, Divya; Alluri, Anajani",Central Research Institute for Dryland Agricul...,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv.11846943.v1;10.26434/chemrxi...,,,,false,None,None,None,None,parent - duplicate (Open Science Framework),2020-02-14,2020.0,2020,None,parent,crossref::10.31219/osf.io/2zuea,fuzzy::tokenbag::alluri_anajani;arun_shanker;b...,alluri_anajani;arun_shanker;bhanu_divya,analysis of whole genome sequences and homolog...,<NA>,<NA>
280462,crossref::10.26434/chemrxiv.11846943.v3,ChemRxiv,crossref,10.26434/chemrxiv.11846943.v3,https://doi.org/10.26434/chemrxiv.11846943.v3,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Analysis of Whole Genome Sequences and Homolog...,"Shanker, Arun; Alluri, Anjani; Bhanu, Divya",Central Research Institute for Dryland Agricul...,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv.11846943.v1;10.26434/chemrxi...,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2020-02-17,2020.0,2020,None,parent,crossref::10.26434/chemrxiv.11846943.v2,relaxed::A1_tokenbag_exact::8475,alluri_anjani;arun_shanker;bhanu_divya,analysis of whole genome sequences and homolog...,<NA>,<NA>
280491,crossref::10.26434/chemrxiv.11846943.v4,ChemRxiv,crossref,10.26434/chemrxiv.11846943.v4,https://doi.org/10.26434/chemrxiv.11846943.v4,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Whole Genome Sequences Analysis and Homology M...,"Shanker, Arun; Alluri, Anjani; Bhanu, Divya",Central Research Institute for Dryland Agricul...,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv.11846943.v1;10.26434/chemrxi...,,,,false,None,None,None,None,parent - duplicate (Open Science Framework),2020-02-21,2020.0,2020,None,parent,crossref::10.31219/osf.io/2zuea,fuzzy::last_initial::alluri|a;bhanu|d;shanker|a,alluri_anjani;arun_shanker;bhanu_divya,whole genome sequences analysis and homology m...,alluri|a;bhanu|d;shanker|a,<NA>
280526,crossref::10.26434/chemrxiv.11846943.v5,ChemRxiv,crossref,10.26434/chemrxiv.11846943.v5,https://doi.org/10.26434/chemrxiv.11846943.v5,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Whole Genome Sequence Analysis and Homology Mo...,"Shanker, Arun; Alluri, Anjani; Bhanu, Divya",Central Research Institute for Dryland Agricul...,None,"{""is-version-of"": [{""asserted

In [58]:
result['records_hierarchy'].value_counts()

records_hierarchy
parent - duplicate (ChemRxiv)                  6
parent - duplicate (Open Science Framework)    3
Name: count, dtype: int64

In [59]:
result['title'].value_counts()

title
Whole Genome Sequence Analysis and Homology Modelling of a 3C Like Peptidase and a Non-Structural Protein 3 of the SARS-CoV-2 Shows Protein Ligand Interaction with an Aza-Peptide and a Noncovalent Lead Inhibitor with Possible Antiviral Properties                       5
Analysis of Whole Genome Sequences and Homology Modelling of a 3-C Like Peptidase and a Non-Structural Protein of the Novel Coronavirus COVID-19 Shows Protein Ligand Interaction with an Aza-Peptide and a Noncovalent Lead Inhibitor with Possible Antiviral Properties    2
Analysis of Whole Genome Sequences and Homology Modelling of a 3C Like Peptidase and a Non-Structural Protein of the Novel Coronavirus COVID-19 Shows Protein Ligand Interaction with an Aza-Peptide and a Noncovalent Lead Inhibitor with Possible Antiviral Properties     1
Whole Genome Sequences Analysis and Homology Modelling of a 3C Like Peptidase and a Non-Structural Protein 3 of the SARS-CoV-2 Shows Protein Ligand Interaction with an Aza-Peptide a

In [60]:
result['authors_flat'].value_counts()

authors_flat
Shanker, Arun; Alluri, Anjani; Bhanu, Divya                      3
Shanker, Arun; Bhanu, Divya; Alluri, Anjani                      3
Shanker, Arun; Bhanu, Divya; Alluri, Anajani                     2
Shanker, Arun; Bhanu, Divya; Alluri, Anjani; Gupta, Samriddhi    1
Name: count, dtype: int64

In [61]:
pattern = "10.26434/chemrxiv-2022-fd190"


mask = data_out['doi'].str.contains(pattern, regex=False, na=False)
result = data_out[mask]
print(len(result))
result

6


,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,date_first_seen,publication_year,publication_year_first_seen,raw_dates,records_hierarchy_backup,parent_record_id,dup_group_id,authors_fp_tokenbag,title_clean_v2,authors_fp_last_initial,authors_fp_last
285948,crossref::10.26434/chemrxiv-2022-fd190,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190,https://doi.org/10.26434/chemrxiv-2022-fd190,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,None,None,,,,,false,None,None,None,None,parent,2022-03-18,2022.0,2022,None,parent,<NA>,relaxed::A1_tokenbag_exact::8698,alexander_bagger;christensen_oliver;daasbjerg_...,selectivity and intrinsic activity of function...,<NA>,<NA>
286219,crossref::10.26434/chemrxiv-2022-fd190-v2,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v2,https://doi.org/10.26434/chemrxiv-2022-fd190-v2,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2022-04-25,2022.0,2022,None,parent,crossref::10.26434/chemrxiv-2022-fd190,relaxed::A1_tokenbag_exact::8698,alexander_bagger;alonso_hernandez_rosas;christ...,selectivity and intrinsic activity of function...,<NA>,<NA>
286274,crossref::10.26434/chemrxiv-2022-fd190-v3,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v3,https://doi.org/10.26434/chemrxiv-2022-fd190-v3,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190;10.26434/chemrxiv...,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2022-05-02,2022.0,2022,None,parent,crossref::10.26434/chemrxiv-2022-fd190,8105542989974901890,alexander_bagger;christensen_oliver;daasbjerg_...,selectivity and intrinsic activity of function...,<NA>,<NA>
286281,crossref::10.26434/chemrxiv-2022-fd190-v4,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v4,https://doi.org/10.26434/chemrxiv-2022-fd190-v4,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190;10.26434/chemrxiv...,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2022-05-04,2022.0,2022,None,parent,crossref::10.26434/chemrxiv-2022-fd190,8105542989974901890,alexander_bagger;christensen_oliver;daasbjerg_...,selectivity and intrinsic activity of function...,<NA>,<NA>
287198,crossref::10.26434/chemrxiv-2022-fd190-v5,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v5,https://doi.org/10.26434/chemrxiv-2022-fd190-v5,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190;10.26434/chemrxiv...,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2022-08-24,2022.0,2022,None,parent,crossref::10.26434/chemrxiv-2022-fd

In [62]:
pattern = "10.26434/chemrxiv-2022-fd190"


mask = data['doi'].str.contains(pattern, regex=False, na=False)
result = data[mask]
print(len(result))
result

6


,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json
291173,crossref::10.26434/chemrxiv-2022-fd190,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190,https://doi.org/10.26434/chemrxiv-2022-fd190,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,None,None,,,,,false,None,None,None,None
291444,crossref::10.26434/chemrxiv-2022-fd190-v2,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v2,https://doi.org/10.26434/chemrxiv-2022-fd190-v2,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190,,,,false,None,None,None,None
291499,crossref::10.26434/chemrxiv-2022-fd190-v3,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v3,https://doi.org/10.26434/chemrxiv-2022-fd190-v3,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190;10.26434/chemrxiv...,,,,false,None,None,None,None
291506,crossref::10.26434/chemrxiv-2022-fd190-v4,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v4,https://doi.org/10.26434/chemrxiv-2022-fd190-v4,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190;10.26434/chemrxiv...,,,,false,None,None,None,None
292423,crossref::10.26434/chemrxiv-2022-fd190-v5,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v5,https://doi.org/10.26434/chemrxiv-2022-fd190-v5,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Selectivity and Intrinsic Activity of Function...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv-2022-fd190;10.26434/chemrxiv...,,,,false,None,None,None,None
318409,crossref::10.26434/chemrxiv-2022-fd190-v6,ChemRxiv,crossref,10.26434/chemrxiv-2022-fd190-v6,https://doi.org/10.26434/chemrxiv-2022-fd190-v6,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Can the CO2 Reduction Reaction be Improved on ...,"Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong...",University of Copenhagen; Aarhus University,None,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,10.26434/chemrxiv-2022-fd190;10.26434/chemrxiv...,10.1021/acscatal.2c04200,,,true,None,None,None,None


In [63]:
result['title'].value_counts()

title
Selectivity and Intrinsic Activity of Functionalized Cu Surfaces: Can the CO2 Reduction Reaction be Improved on Cu?    5
Can the CO2 Reduction Reaction be Improved on Cu: Selectivity and Intrinsic Activity of Functionalized Cu Surfaces     1
Name: count, dtype: int64

In [64]:
result['authors_flat'].value_counts()

authors_flat
Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong; Bagger, Alexander; Vang Lauritsen, Jeppe; Uttrup Pedersen, Steen; Daasbjerg, Kim; Rossmeisl, Jan                             5
Christensen, Oliver; Zhao, Siqi; Sun, Zhaozong; Bagger, Alexander; Rosas-Hernández, Alonso; Vang Lauritsen, Jeppe; Uttrup Pedersen, Steen; Daasbjerg, Kim; Rossmeisl, Jan    1
Name: count, dtype: int64

In [65]:
sample_titles = data_out.sample(5)[['title', 'authors_flat']].title
data_out[data_out.title.isin(sample_titles)][['title','authors_flat','records_hierarchy','date_first_seen']]

,title,authors_flat,records_hierarchy,date_first_seen
2077039,Testing the 'home market eect' in a multi-coun...,Kristian Behrens; Andrea Lamorgese,parent,2004-01-01
4613384,The Scale of New Physics from the Higgs Coupli...,"Abu-Ajamieh, Fayez",parent,2021-12-27
5355301,On the generators of the canonical module of a...,"Miyazaki, Mitsuhiro",parent,2016-06-14
6043011,Is a global virtual currency with universal ac...,"Jegatheesan, Sowmyan; Ahmed, Sabbir; Chamney, ...",parent,2015-02-07
6512695,BaBar Web job submission with Globus authentic...,"Barlow, R. J.; Forti, A.; McNab, A.; Salih, S....",parent,2003-06-14


In [66]:
pattern = "10.26434/chemrxiv.13102877"


mask = data_out['doi'].str.contains(pattern, regex=False, na=False)
result = data_out[mask]
print(len(result))
result

32


,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,institutions_flat,countries_flat,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,date_first_seen,publication_year,publication_year_first_seen,raw_dates,records_hierarchy_backup,parent_record_id,dup_group_id,authors_fp_tokenbag,title_clean_v2,authors_fp_last_initial,authors_fp_last
283340,crossref::10.26434/chemrxiv.13102877.v1,ChemRxiv,crossref,10.26434/chemrxiv.13102877.v1,https://doi.org/10.26434/chemrxiv.13102877.v1,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Aptamers for Detection and Diagnostics (ADD): ...,"Datta, Shoumen",MIT,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv.13102877.v2;10.26434/chemrxi...,,,,false,None,None,None,None,parent,2020-10-23,2020.0,2020,None,parent,<NA>,16854196709595633026,datta_shoumen,aptamers for detection and diagnostics add pro...,datta|s,<NA>
283370,crossref::10.26434/chemrxiv.13102877.v2,ChemRxiv,crossref,10.26434/chemrxiv.13102877.v2,https://doi.org/10.26434/chemrxiv.13102877.v2,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Aptamers for Detection and Diagnostics (ADD): ...,"Datta, Shoumen",MIT,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv.13102877.v1;10.26434/chemrxi...,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2020-10-27,2020.0,2020,None,parent,crossref::10.26434/chemrxiv.13102877.v1,16854196709595633026,datta_shoumen,aptamers for detection and diagnostics add pro...,<NA>,<NA>
283397,crossref::10.26434/chemrxiv.13102877.v3,ChemRxiv,crossref,10.26434/chemrxiv.13102877.v3,https://doi.org/10.26434/chemrxiv.13102877.v3,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Aptamers for Detection and Diagnostics (ADD) i...,"Datta, Shoumen",MIT,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv.13102877.v1;10.26434/chemrxi...,,,,false,None,None,None,None,parent,2020-11-06,2020.0,2020,None,parent,<NA>,15378799223749550050,datta_shoumen,aptamers for detection and diagnostics add is ...,datta|s,<NA>
283428,crossref::10.26434/chemrxiv.13102877.v4,ChemRxiv,crossref,10.26434/chemrxiv.13102877.v4,https://doi.org/10.26434/chemrxiv.13102877.v4,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Aptamers for Detection and Diagnostics (ADD) i...,"Datta, Shoumen",MIT,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv.13102877.v1;10.26434/chemrxi...,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2020-11-09,2020.0,2020,None,parent,crossref::10.26434/chemrxiv.13102877.v3,15378799223749550050,datta_shoumen,aptamers for detection and diagnostics add is ...,<NA>,<NA>
283468,crossref::10.26434/chemrxiv.13102877.v5,ChemRxiv,crossref,10.26434/chemrxiv.13102877.v5,https://doi.org/10.26434/chemrxiv.13102877.v5,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Aptamers for Detection and Diagnostics (ADD) i...,"Datta, Shoumen",MIT,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv.13102877.v1;10.26434/chemrxi...,,,,false,None,None,None,None,parent - duplicate (ChemRxiv),2020-11-11,2020.0,2020,None,parent,crossref::10.26434/chemrxiv.13102877.v3,15378799223749550050,datta_shoumen,aptamers for detection and diagnostics add is ...,<NA>,<NA>
283498,crossref::10.26434/chemrxiv.13102877.v6,ChemRxiv,crossref,10.26434/chemrxiv.13102877.v6,https://doi.org/10.26434/chemrxiv.13102877.v6,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Aptamers for Detection and Diagnostics (ADD) i...,"Datta, Shoumen",MIT,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.26434/chemrxiv.13102877.v1;10.2